# 🧬 **pdb2reaction**: End-to-End Reaction-Path Elucidation from PDB Structures Using Machine-Learning Interatomic Potentials

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/t-0hmura/pdb2reaction/blob/main/examples/pdb2reaction_colab.ipynb)

Build enzymatic reaction paths from PDB/mmCIF or small-molecule XYZ/GJF structures in four steps: **1 Input → 2 Viewer → 3 Options → 4 Results**.

> Choose a **GPU** runtime, run **Setup** once, then run **Launch GUI**.

[GitHub](https://github.com/t-0hmura/pdb2reaction) · [ChemRxiv](https://chemrxiv.org/doi/full/10.26434/chemrxiv.15003538/v1)


In [ ]:
#@title  ⚙️ Setup — install pdb2reaction + a backend  (first run ~5 min) { display-mode: "form" }
#@markdown Only the **selected backend** is installed (MACE and UMA cannot coexist because their `e3nn` requirements conflict). To switch, restart the runtime and run Setup again.
#@markdown **MACE / ORB = no login; UMA = gated Hugging Face model.** ORB/MACE run fp64 and UMA runs fp32 by default. The release ref defaults to the notebook's matching version.
backend = "mace"  #@param ["mace", "uma", "orb"]
pdb2reaction_ref = "v0.4.12"  #@param {type:"string"}
#@markdown Tick **install_dft** to add PySCF + GPU4PySCF for the standalone `dft` workflow or the optional DFT stage in `all` — a few extra minutes.
install_dft = False  #@param {type:"boolean"}
import os, sys, subprocess, time, importlib, importlib.util
from pathlib import Path
# pysisyphus reads this optional user config during import.  An empty file keeps
# a fresh Colab runtime quiet while preserving the package defaults.
Path.home().joinpath('.pysisyphusrc').touch(exist_ok=True)
_setup_started = time.monotonic()
def _phase(number, label):
    print('[%d/5] %s …' % (number, label), flush=True)
def _phase_done(label):
    print('      ✓ %s (%.1f min elapsed)' % (label, (time.monotonic() - _setup_started) / 60), flush=True)
if globals().get('BACKEND') not in (None, backend):
    raise RuntimeError('Backend switch requested. Restart the Colab runtime first, then rerun Setup.')
try: _gpu = subprocess.run(['nvidia-smi','-L'], capture_output=True, text=True).stdout.strip()
except FileNotFoundError: _gpu = ''   # nvidia-smi absent on CPU runtimes -> don't crash Setup
print(_gpu or '⚠️ No GPU detected — for real runs pick a GPU runtime (Runtime ▸ Change runtime type ▸ GPU). The GUI still loads.')
def pip(*a): subprocess.run([sys.executable,'-m','pip','install','-q',*a], check=True)
# Install the pinned release from PyPI — the same path a normal user takes.
# The wheel carries the package only, so the bundled example structures are
# fetched from the matching git tag on demand (see the GUI's Load example).
REPO_DIR = 'pdb2reaction-src'          # only present in a source/debug checkout
_phase(1, 'pdb2reaction %s from PyPI' % pdb2reaction_ref)
pip('pdb2reaction' + ('[dft]' if install_dft else '') + '==' + pdb2reaction_ref.lstrip('v'))
if install_dft:
    from importlib.metadata import PackageNotFoundError, version as _dist_version
    _dft_packages = {'pyscf': 'pyscf', 'gpu4pyscf': 'gpu4pyscf-cuda12x'}
    _dft_missing = [module for module in _dft_packages
                    if importlib.util.find_spec(module) is None]
    _dft_versions = {}
    for _module, _distribution in _dft_packages.items():
        try: _dft_versions[_module] = _dist_version(_distribution)
        except PackageNotFoundError: _dft_missing.append(_module)
    if _dft_missing:
        raise RuntimeError('install_dft was selected, but these modules are missing: %s' %
                           ', '.join(sorted(set(_dft_missing))))
    try:
        _dft_imports = ('pyscf', 'basis_set_exchange', 'gpu4pyscf.dft')
        for _module in _dft_imports: importlib.import_module(_module)
        _cupy = importlib.import_module('cupy')
        _dft_gpu_count = int(_cupy.cuda.runtime.getDeviceCount())
    except Exception as _dft_exc:
        raise RuntimeError('DFT packages installed but failed their import/GPU check: %s' %
                           _dft_exc) from _dft_exc
    if _dft_gpu_count < 1:
        raise RuntimeError('DFT support needs a GPU runtime; no CUDA device is visible.')
    print('DFT support installed: PySCF %s · GPU4PySCF %s · CUDA devices %d' %
          (_dft_versions['pyscf'], _dft_versions['gpu4pyscf'], _dft_gpu_count))
_phase_done('tool package installed')
_phase(2, '%s MLIP backend' % backend.upper())
if backend == 'mace':
    subprocess.run([sys.executable,'-m','pip','uninstall','-y','-q','fairchem-core'], check=False)
    pip('mace-torch>=0.3.8'); print('MACE installed (no login needed).')
elif backend == 'orb':
    # ORB coexists with the base install (the e3nn clash is MACE vs fairchem-core only).
    pip('orb-models'); print('ORB installed (no login needed; runs fp64 by default).')
else:
    # UMA is gated: accept the FAIR Chemistry License at https://huggingface.co/facebook/UMA
    from huggingface_hub import login, notebook_login
    if os.environ.get('HF_TOKEN'): login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    else: notebook_login()
    print('UMA installed (uses fairchem-core; needs HF license + login).')
_phase_done('backend installed')
_phase(3, 'notebook widgets')
pip('ipywidgets','matplotlib')
_phase_done('GUI dependencies installed; Mol* loads in the browser')
# Plotly static-image export (scan maps, energy diagrams) needs a kaleido/choreographer
# combo with a working Chrome bridge. Unpinned, pip resolves kaleido 1.3 / choreographer 1.3,
# which raise 'Kaleido requires Google Chrome' on Colab and lose every PNG in the Results tab.
_phase(4, 'plot export and Chromium bridge')
pip('plotly==6.6.0','kaleido==1.2.0','choreographer==1.2.1')
_chrome = subprocess.run(['plotly_get_chrome','-y'], capture_output=True, text=True)
if _chrome.returncode != 0:
    _chrome = subprocess.run(['choreo_get_chrome'], capture_output=True, text=True)
print('PNG export:', 'chromium ready' if _chrome.returncode == 0 else
      'chromium unavailable - PNG export will be skipped (interactive HTML plots still work)')
_phase_done('plot export checked')
_phase(5, 'installed-version verification')
INSTALL_DFT = install_dft
BACKEND = backend; TOOL = 'pdb2reaction'
from importlib.metadata import version
installed_version = version('pdb2reaction')
if pdb2reaction_ref.startswith('v') and installed_version != pdb2reaction_ref[1:]:
    raise RuntimeError('Installed version %s does not match requested ref %s.' % (installed_version, pdb2reaction_ref))
print('pdb2reaction', installed_version, '| ref', pdb2reaction_ref, '| from PyPI')
_phase_done('version verified')
print('\n✅ Setup done in %.1f min. backend = %s — now run "Launch GUI".' %
      ((time.monotonic() - _setup_started) / 60, BACKEND))


In [ ]:
#@title 🖥️ Launch GUI  (run once, after Setup) { display-mode: "form" }
import os, glob, json, shlex, math, shutil, subprocess, time, zipfile, hashlib, importlib.util, signal, html, tempfile, base64, csv, io, weakref
from pathlib import Path
import ipywidgets as W
from IPython.display import display, clear_output, Image, HTML
import click
try: TOOL
except NameError: TOOL = 'pdb2reaction'
try: BACKEND
except NameError: BACKEND = 'mace'
try: REPO_DIR
except NameError: REPO_DIR = 'pdb2reaction-src'
IS_CLUSTER = True
_RUNTIME_DIR = Path(tempfile.mkdtemp(prefix='%s-colab-' % TOOL.replace('_', '-')))

def _runtime_path(*parts):
    """Allocate notebook-owned files away from uploads and run outputs."""
    path = _RUNTIME_DIR.joinpath(*parts)
    path.parent.mkdir(parents=True, exist_ok=True)
    return str(path)

def _unique_path(path):
    """Return path, or a numbered sibling, without replacing an existing file."""
    candidate = Path(path)
    if not candidate.exists(): return str(candidate)
    for number in range(2, 10000):
        numbered = candidate.with_name('%s_%d%s' % (candidate.stem, number, candidate.suffix))
        if not numbered.exists(): return str(numbered)
    raise RuntimeError('Could not allocate a unique path for %s.' % candidate.name)
# Colab needs the custom widget manager for ipywidgets uploads to render.
try:
    from google.colab import output as _cwm; _cwm.enable_custom_widget_manager()
except Exception:
    _cwm = None            # custom-widget-manager is Colab-only; skip elsewhere

ACCENT = '#114b8a'
display(HTML("""<style>
.rxapp {
  --rx-ink:#172033; --rx-muted:#607087; --rx-line:#dfe6ef; --rx-soft:#f5f8fc;
  --rx-blue:#1268b3; --rx-blue-soft:#eaf4ff; --rx-green:#157347;
  --rx-amber:#b85d20; --rx-navy:#101827;
  width:100%; max-width:100%; box-sizing:border-box; padding:0 8px 8px;
}
.rxheader { display:flex; align-items:center; justify-content:space-between; gap:12px;
  flex-wrap:wrap; background:var(--rx-navy); border-radius:13px; padding:9px 14px;
  margin-bottom:5px; box-shadow:0 5px 16px rgba(15,23,42,.16); }
.rxheader-product { font-size:21px; line-height:1.2; font-weight:720; letter-spacing:.15px; color:#f8fafc; }
.rxheader-meta { display:flex; align-items:center; gap:7px; flex-wrap:wrap;
  font-size:12px; color:#aebbd0; }
.rxheader-chip { display:inline-block; background:#1d2a40; color:#a8d2ff;
  padding:3px 9px; border-radius:999px; font-weight:650; letter-spacing:.2px; }
.rxtabs { width:100%; margin:5px 0 8px !important; column-gap:6px !important; row-gap:5px !important; }
.rxtabs > .widget-button { flex:1 1 135px !important; min-width:118px !important; min-height:42px !important; }
.rxtabs .jupyter-button.mod-primary { background:var(--rx-blue) !important;
  border-color:var(--rx-blue) !important; box-shadow:0 0 0 2px rgba(18,104,179,.13) !important; }
.rxfold { margin:4px 0 2px; }
.rxapp .rxcard { padding:6px 8px !important; margin:0 !important; }
.rxworkspace { row-gap:8px !important; }
.rxapp .widget-vbox > .widget-html:empty { display:none; }
.rxapp .widget-html-content > div { margin:0; }
.rxinspector, .rxviewer { min-width:0; }
.rxfold > button { background:#f8fafc !important; border:1px solid #e2e8f0 !important;
  color:#334155 !important; font-weight:600 !important; text-align:left !important; }
.rxapp, .rxapp .widget-label, .rxapp .widget-html-content, .rxapp .widget-readout {
  font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Helvetica,Arial,sans-serif !important;
  color:#1f2937; }
.rxapp .widget-html-content code { background:#f1f5f9; padding:1px 5px; border-radius:5px;
  font-family:ui-monospace,SFMono-Regular,Menlo,monospace; font-size:12px; color:#334155;
  overflow-wrap:anywhere; word-break:break-word; white-space:normal; }
.rxapp .widget-button { border-radius:9px !important; font-weight:500 !important;
  box-shadow:none !important; min-height:38px; }
/* Button labels are complete; hide Font Awesome glyphs that become tofu boxes
   when a Colab/static frontend does not provide the icon font. */
.rxapp .widget-button .fa, .rxapp .widget-upload .fa { display:none !important; }
.rxapp button.jupyter-button, .rxapp .widget-upload > button { min-height:38px !important; }
.rxapp .widget-dropdown select, .rxapp .widget-text input, .rxapp .widget-textarea textarea,
.rxapp input[type="number"] { border-radius:9px !important; border:1px solid #e2e8f0 !important; }
.rxapp .widget-button:focus-visible, .rxapp select:focus-visible,
.rxapp input:focus-visible, .rxapp textarea:focus-visible {
  outline:3px solid rgba(37,99,235,.28) !important; outline-offset:2px !important; }
.rxapp .widget-tab > .rxapp-tabs, .rxapp .p-TabBar-tab { font-weight:500; }
.rxcard { border:1px solid var(--rx-line); border-radius:11px; padding:7px 9px; background:#ffffff;
  box-shadow:0 1px 2px rgba(16,24,40,0.045); box-sizing:border-box; flex:0 0 auto; min-width:0; }
.rxapp .rxworkspace { align-items:stretch !important; column-gap:8px; row-gap:7px !important;
  margin:0 0 7px !important; flex-wrap:nowrap !important; }
/* the lower panel row reuses the workspace column tracks so the frame edges line up */
.rxworkspace > .rxviewer > .rxcard { margin:0; }
.rxviewer { flex:2 1 520px !important; min-width:300px; position:sticky; top:8px; align-self:flex-start; }
.rxinspector { flex:1 1 300px !important; min-width:260px; }
.rxinspector > .rxcard { width:100%; }
.rxviewer .jupyter-widgets-output-area, .rxviewer iframe { width:100% !important; max-width:100% !important; }
.rxviewer .rxmolstar-frame, .rxresults .rxmolstar-frame {
  display:block; width:100% !important; max-width:100% !important; height:auto !important;
  aspect-ratio:4 / 3; box-sizing:border-box !important; overflow:hidden !important;
}
.rxresults { width:100% !important; max-width:100% !important; min-width:0 !important; }
.rxresults .jupyter-widgets-output-area, .rxresults iframe,
.rxresults canvas { width:100% !important; max-width:100% !important; box-sizing:border-box !important; }
.rxapp img, .rxapp svg, .rxapp iframe { max-width:100%; }
.rxcmd textarea { font-family:ui-monospace,SFMono-Regular,Menlo,monospace !important;
  font-size:13px !important; background:#0f172a !important; color:#7ee787 !important;
  border-radius:11px !important; line-height:1.5 !important; border:1px solid #1e293b !important;
  padding:9px 11px !important; }
.rxdrop { position:relative; min-height:112px;
  border:2px dashed #cbd5e1; border-radius:14px; padding:12px; background:#f8fafc;
  transition:border-color .15s ease,background .15s ease;
  align-items:center !important; justify-content:center !important; text-align:center; box-sizing:border-box; }
.rxdrop:hover, .rxdrop:focus-within, .rxdrop.rxdrag {
  border-color:#2563eb; background:#eff6ff; }
.rxdrop.rxbusy { opacity:.72; }
.rxdrop .rxdrop-prompt { width:100%; color:#52647b; line-height:1.35; pointer-events:none; }
.rxdrop .rxdrop-prompt b { color:#334155; }
.rxdrop .widget-upload { width:220px !important; max-width:100% !important;
  align-self:center !important; margin:3px auto 0 !important; }
.rxdrop .widget-upload > button { width:220px !important; max-width:100% !important; }
.rxapp .rxfile { width:100%; flex-wrap:nowrap !important; align-items:center !important;
  border:1px solid #e2e8f0; border-radius:10px; padding:5px 7px; background:#fff;
  box-sizing:border-box; }
.rxapp .rxfile > .widget-html { flex:1 1 auto !important; min-width:0 !important; }
.rxapp .rxfile > .widget-hbox { flex:0 0 auto !important; flex-wrap:nowrap !important; }
.rxapp .rxfile button { min-width:34px !important; padding:0 8px !important; font-size:0 !important; }
.rxapp .rxfile button::after { font-size:16px; line-height:1; }
.rxapp .rxmove-earlier::after, .rxapp .rxmove-earlier button::after { content:'↑'; }
.rxapp .rxmove-later::after, .rxapp .rxmove-later button::after { content:'↓'; }
.rxapp .rxremove-file::after, .rxapp .rxremove-file button::after { content:'×'; }
.rxhelp-row, .rxflagrow, .rxinfo, .rxinfo .widget-html-content {
  position:relative; overflow:visible !important;
}
.rxapp .rxhelp-row, .rxapp .rxflagrow {
  flex-wrap:wrap !important; align-items:center !important; width:100% !important;
}
.rxapp .rxinfo { flex:0 0 26px; width:26px; min-width:26px; }
.rxapp .rxinfo:has(.rxinfo-details[open]) {
  flex:1 1 100%; width:100%; min-width:100%;
}
.rxinfo-details > summary {
  width:24px; min-width:24px; height:24px; box-sizing:border-box;
  display:flex; align-items:center; justify-content:center;
  cursor:pointer; list-style:none; color:#52647b; background:transparent;
  border:0; border-radius:4px; font-size:17px; font-weight:600; line-height:1;
}
.rxinfo-details > summary::-webkit-details-marker { display:none; }
.rxinfo-details > summary:hover, .rxinfo-details > summary:focus-visible,
.rxinfo-details[open] > summary {
  color:#1d4ed8; background:#eff6ff; outline:none;
}
.rxinfo-details > summary:focus-visible {
  box-shadow:0 0 0 3px rgba(37,99,235,.24);
}
.rxinfo-details .rxhelp-panel {
  position:relative; border:1px solid #bfdbfe; border-radius:9px;
  background:#eff6ff; color:#1e3a5f; padding:8px 10px;
  margin:5px 0 5px; line-height:1.4;
}
.rxinfo-details .rxhelp-panel::before {
  content:''; position:absolute; top:-6px; left:7px; width:10px; height:10px;
  background:#eff6ff; border-left:1px solid #bfdbfe;
  border-top:1px solid #bfdbfe; transform:rotate(45deg);
}
.rxsr-only { position:absolute !important; width:1px !important; height:1px !important;
  padding:0 !important; margin:-1px !important; overflow:hidden !important;
  clip:rect(0,0,0,0) !important; white-space:nowrap !important; border:0 !important; }
.rxchip button { border-radius:999px !important; font-size:12px !important; padding:1px 10px !important; }
.rxrun { margin-left:auto !important; }
.rxrun button { font-weight:600 !important; }
.rxapp .widget-hbox { flex-wrap:wrap !important; row-gap:6px; }
.rxapp .widget-hbox.rxfile, .rxapp .rxfile > .widget-hbox {
  flex-wrap:nowrap !important; }
.rxapp .rxworkflow-controls { display:grid !important;
  grid-template-columns:minmax(280px,320px) minmax(150px,1fr) 240px; gap:6px; align-items:center; }
.rxapp .rxworkflow-contract { display:grid !important;
  grid-template-columns:minmax(220px,1fr) minmax(420px,2fr); gap:8px; }
.rxpath { border:1px solid #cfe0f2; border-radius:13px; background:#f8fbff;
  padding:9px !important; margin-top:5px; box-shadow:0 2px 8px rgba(30,64,110,.055); }
.rxpath-head { width:100%; align-items:center !important; justify-content:space-between !important;
  column-gap:10px !important; }
.rxpath-title { font-size:15px; font-weight:700; color:var(--rx-ink); }
.rxpath-subtitle { color:var(--rx-muted); font-size:12px; margin-left:7px; }
.rxpath-state .widget-html-content { display:inline-block; background:#fff; color:#334155;
  border:1px solid #cad8e8; border-radius:999px; padding:4px 10px; white-space:nowrap; }
.rxpath-controls { width:100%; align-items:center !important; flex-wrap:nowrap !important;
  column-gap:6px !important; padding:2px 0 3px; }
.rxpath-controls > .widget-slider { flex:1 1 360px !important; min-width:190px !important; }
.rxpath-controls .widget-play { flex:0 0 auto !important; }
.rxpath-grid { width:100%; display:grid !important;
  grid-template-columns:repeat(2,minmax(0,1fr)); gap:8px; align-items:start; }
.rxpath-panel { min-width:0; overflow:hidden; border:1px solid var(--rx-line);
  border-radius:11px; background:#fff; }
.rxpath-panel-title { padding:6px 9px; border-bottom:1px solid #edf1f6;
  color:#44546a; font-size:12px; font-weight:700; letter-spacing:.15px; }
.rxpath-panel > .widget-output { padding:0 4px 4px; }
.rxartifact { margin-top:6px; }
.rxartifact > button { background:#fff !important; }
.rxapp hr { border:none; border-top:1px solid #eef0f4; }
@media (max-width: 820px) {
  .rxapp .rxworkspace { flex-wrap:wrap !important; }
  .rxviewer, .rxinspector { flex:1 1 100% !important; min-width:0 !important; }
  .rxviewer { position:static; }
  .rxapp .rxworkflow-controls, .rxapp .rxworkflow-contract {
    display:flex !important; flex-flow:row wrap !important; }
  .rxpath-grid { grid-template-columns:1fr; }
  .rxpath-controls { flex-wrap:wrap !important; }
}
@media (max-width: 600px) {
  .rxapp .rxfilename .widget-html-content > span { white-space:normal !important; overflow-wrap:anywhere; }
  .rxapp .widget-dropdown, .rxapp .widget-text,
  .rxapp .widget-select-multiple { max-width:100% !important; }
  .rxapp .lm-TabBar-tab, .rxapp .p-TabBar-tab { min-width:0 !important; }
  .rxapp .lm-TabBar-tabLabel, .rxapp .p-TabBar-tabLabel {
    overflow:hidden; text-overflow:ellipsis; }
}
@media (prefers-reduced-motion: reduce) {
  .rxapp * { transition:none !important; animation:none !important; }
}
</style>"""))

CLI = 'pdb2reaction'
from pdb2reaction.cli import cli as PRODUCT_CLI
from pdb2reaction.domain.residue_data import AMINO_ACIDS as _AA_DATA, ION as _ION_CHARGES, WATER_RES as _WATER_DATA
_INITIAL_MODEL = {'mace': 'MACE-OMOL-0', 'uma': 'uma-s-1p2',
                  'orb': 'orb_v3_conservative_omol'}.get(BACKEND, 'MACE-OMOL-0')
DFT_READY = (importlib.util.find_spec('pyscf') is not None and
             importlib.util.find_spec('gpu4pyscf') is not None)
DMF_READY = (importlib.util.find_spec('dmf') is not None and
             importlib.util.find_spec('cyipopt') is not None)
S = {'tool': TOOL, 'backend': BACKEND, 'model': _INITIAL_MODEL,
     'mode': None, 'inputs': [], 'subcmd': 'all',
     'advanced_overrides': {},
     'parm': None, 'model_pdb': None, 'center': [], 'center_ids': [], 'lcharge': {},
     '_pre_extract': None,
     'scan_atoms': [None, None], 'scan_target': 1.6, 'scan_preset': '', 'scan_stages': [], 'scan_axes': [],
     'freeze_buf': [None, None], 'freeze_pairs': [], 'freeze_atoms': [], 'charge': 0,
     'charge_explicit': False,
     'tsopt': False, 'thermo': False, 'out_dir': 'result',
     '_last_out_dir': None, '_last_subcmd': None, '_last_argv': [], '_last_files': [], '_last_manifest': {}, '_last_log': '',
     '_pdb_path': None, '_pdb_text': '', '_view_format': 'pdb',
     '_hetero': [], '_atoms': {}, '_atom_meta': [],
     '_last_pick': None, '_pick_history': [],
     '_last_pick_message': '', '_last_pick_tone': 'ok', '_view_input_index': 0, '_view_mapping_ok': True,
     '_primary_atom_signatures': [], '_primary_atom_meta': [],
     '_uploaded_paths': set(), '_installed_backend': BACKEND,
     'show_water': False, 'surface': False, 'spin': False,
     'measure_atoms': [], 'rep': 'cartoon', 'color': 'element',
     'viewer_width': 720, 'viewer_height': 540}

_AA = set(_AA_DATA)
_WATER = set(_WATER_DATA) | {'T3P','TIP3P','TIP4P','TIP5P','SPC','SPCE','OPC','OPC3'}

def _pdb_coordinate_rows(text):
    rows = []
    for line in text.splitlines():
        if line[0:6].strip() not in ('ATOM', 'HETATM'): continue
        try:
            rows.append((float(line[30:38]), float(line[38:46]), float(line[46:54])))
        except ValueError:
            raise ValueError('Viewer bridge contains an invalid PDB coordinate row.')
    return rows

# Link/cap hydrogens are appended by `extract` as HL atoms in residue LKH.
# They are neither a ligand (no -l charge) nor a sensible extraction center.
_CAP = {'LKH'}


_EXAMPLE_URL = 'https://raw.githubusercontent.com/t-0hmura/pdb2reaction/%s/examples/%s'


def _example_file(relpath):
    """Return a local path to a bundled example structure.

    A source checkout (debug builds) has them on disk; a PyPI install does not,
    because the wheel ships the package only. In that case fetch the file from
    the git tag matching the installed release, so example and code agree.
    """
    local = os.path.join(REPO_DIR, 'examples', relpath)
    if os.path.exists(local):
        return local
    dest = _runtime_path('examples', relpath)
    if not os.path.exists(dest):
        import urllib.request
        urllib.request.urlretrieve(_EXAMPLE_URL % (pdb2reaction_ref, relpath), dest)
    return dest

def parse_residues(text, metadata=None):
    allr, het = set(), set()
    if metadata:
        for atom in metadata:
            r = str(atom.get('resname') or '').strip().upper()
            if not r or r in _CAP: continue
            allr.add(r)
            if r not in _WATER and r not in _AA: het.add(r)
        return sorted(allr), sorted(het)
    for ln in text.splitlines():
        if ln[0:6].strip() not in ('ATOM', 'HETATM'): continue
        r = ln[17:20].strip()
        if r in _CAP: continue
        allr.add(r)
        if r not in _WATER and r not in _AA: het.add(r)
    return sorted(allr), sorted(het)

def parse_atoms(text, metadata=None):
    if metadata:
        coords = _pdb_coordinate_rows(text)
        if len(coords) != len(metadata):
            raise ValueError('Viewer PDB atom count does not match retained structure metadata.')
        out = {}
        for index, (meta, xyz) in enumerate(zip(metadata, coords)):
            meta['xyz'] = xyz
            meta['index'] = index
            key = (str(meta.get('chain') or ''), str(meta.get('resname') or ''),
                   str(meta.get('resseq')), str(meta.get('name') or ''))
            out[key] = xyz
        return out
    d = {}
    for ln in text.splitlines():
        if ln[0:6].strip() in ('ATOM', 'HETATM'):
            try:
                d[(ln[21:22].strip(), ln[17:20].strip(), ln[22:26].strip(), ln[12:16].strip())] = \
                    (float(ln[30:38]), float(ln[38:46]), float(ln[46:54]))
            except ValueError:
                continue
    return d

def _load_view_structure(path):
    """Return a safe viewer PDB plus original/auth atom metadata."""
    from pdb2reaction.core.utils import prepare_input_structure, load_pdb_atom_metadata
    with prepare_input_structure(Path(path)) as prepared:
        source = Path(prepared.geom_path)
        text = source.read_text(encoding='utf-8', errors='replace')
        metadata = load_pdb_atom_metadata(source)
        if not metadata:
            raise ValueError('Structure contains no coordinate atoms.')
        viewer_path = Path(_runtime_path('viewer_input.pdb'))
        viewer_path.write_text(text, encoding='utf-8')
    parse_atoms(text, metadata)       # attach stable 0-based index + coordinates
    return text, metadata, str(viewer_path)

def _load_small_view_structure(path):
    """Return the first XYZ geometry plus stable metadata for small molecules."""
    from pdb2reaction.core.utils import prepare_input_structure
    with prepare_input_structure(Path(path)) as prepared:
        source = Path(prepared.geom_path)
        raw = source.read_text(encoding='utf-8', errors='replace')
    lines = raw.splitlines()
    try: atom_count = int(lines[0].strip())
    except (IndexError, ValueError):
        raise ValueError('Small-molecule viewer needs XYZ/GJF coordinates.')
    if atom_count < 1 or len(lines) < atom_count + 2:
        raise ValueError('XYZ input is incomplete.')
    frame = lines[:atom_count + 2]
    metadata = []
    for index, line in enumerate(frame[2:]):
        fields = line.split()
        if len(fields) < 4:
            raise ValueError('XYZ atom row %d is incomplete.' % (index + 1))
        element = fields[0].strip()
        try: xyz = tuple(float(value) for value in fields[1:4])
        except ValueError:
            raise ValueError('XYZ atom row %d has invalid coordinates.' % (index + 1))
        metadata.append({'serial': index + 1, 'chain': '', 'resname': 'MOL',
                         'resseq': 1, 'icode': '', 'name': '%s%d' % (element.upper(), index + 1),
                         'element': element, 'index': index, 'xyz': xyz})
    text = '\n'.join(frame) + '\n'
    viewer_path = Path(_runtime_path('viewer_input.xyz'))
    viewer_path.write_text(text, encoding='utf-8')
    return text, metadata, str(viewer_path)

def _aspec(x):
    chain = str(x.get('chain') or '').strip()
    if chain:
        return '%s:%s:%s:%s' % (chain, x['resn'], x['resi'], x['atom'])
    return '%s %s %s' % (x['resn'], x['resi'], x['atom'])

def _atom_identity(atom):
    if not atom: return None
    try:
        if atom.get('index') is not None: return ('index', int(atom['index']))
    except (TypeError, ValueError):
        pass
    return ('name', str(atom.get('chain') or ''), str(atom.get('resn') or '').upper(),
            str(atom.get('resi') or ''), str(atom.get('atom') or '').upper())

def _same_atom(first, second):
    return bool(first and second and _atom_identity(first) == _atom_identity(second))

def _stored_atom_records():
    seen = set()
    def emit(value):
        if isinstance(value, dict) and id(value) not in seen:
            seen.add(id(value)); return value
        return None
    for atom in S.get('scan_atoms', []):
        record = emit(atom)
        if record is not None: yield record
    for stage in S.get('scan_stages', []):
        for bond in stage:
            for atom in (bond.get('a'), bond.get('b')):
                record = emit(atom)
                if record is not None: yield record
    for axis in S.get('scan_axes', []):
        for atom in (axis.get('a'), axis.get('b')):
            record = emit(atom)
            if record is not None: yield record
    for atom in S.get('freeze_buf', []):
        record = emit(atom)
        if record is not None: yield record
    for pair in S.get('freeze_pairs', []):
        for atom in (pair.get('a'), pair.get('b')):
            record = emit(atom)
            if record is not None: yield record
    for atom in S.get('measure_atoms', []):
        record = emit(atom)
        if record is not None: yield record
    for atom in S.get('_pick_history', []):
        record = emit(atom)
        if record is not None: yield record
    record = emit(S.get('_last_pick'))
    if record is not None: yield record

def _remap_stored_atom_coordinates(metadata):
    for atom in _stored_atom_records():
        try: index = int(atom.get('index'))
        except (TypeError, ValueError): continue
        if not (0 <= index < len(metadata)): continue
        current = metadata[index]
        atom.update(chain=str(current.get('chain') or ''),
                    resn=str(current.get('resname') or ''),
                    resi=str(current.get('resseq')) + str(current.get('icode') or ''),
                    atom=str(current.get('name') or ''), xyz=current.get('xyz'))

def _assert_distinct_pairs():
    pairs = []
    a, b = S.get('scan_atoms', [None, None])
    if a and b: pairs.append(('scan bond', a, b))
    for stage in S.get('scan_stages', []):
        pairs.extend(('staged scan bond', bond.get('a'), bond.get('b')) for bond in stage)
    pairs.extend(('scan axis', axis.get('a'), axis.get('b')) for axis in S.get('scan_axes', []))
    pairs.extend(('distance restraint', pair.get('a'), pair.get('b')) for pair in S.get('freeze_pairs', []))
    for label, first, second in pairs:
        if _same_atom(first, second):
            raise ValueError('%s needs two different atoms.' % label.capitalize())

def scan_literals():
    """One literal per stage (multiple = staged); tuples within a stage = concerted."""
    if S['scan_stages']:
        return ['[' + ','.join('(\"%s\",\"%s\",%g)' % (_aspec(bd['a']), _aspec(bd['b']), bd['t'])
                               for bd in stage) + ']' for stage in S['scan_stages']]
    if S['scan_preset'] and not all(S['scan_atoms']): return [S['scan_preset']]
    a, b = S['scan_atoms']
    if a and b: return ['[(\"%s\",\"%s\",%g)]' % (_aspec(a), _aspec(b), S['scan_target'])]
    return []

def scan2d_literal():
    """scan2d/scan3d need ONE -s with per-axis quadruples: [(a,b,low,high),...]."""
    ax = S.get('scan_axes', [])
    if not ax: return ''
    return '[' + ','.join('(\"%s\",\"%s\",%g,%g)' % (_aspec(a['a']), _aspec(a['b']), a['lo'], a['hi'])
                          for a in ax) + ']'

def freeze_pair_lit():
    if not S['freeze_pairs']: return ''
    parts = []
    for p in S['freeze_pairs']:
        if p.get('t') is not None:
            parts.append('(\"%s\",\"%s\",%g)' % (_aspec(p['a']), _aspec(p['b']), p['t']))
        else:
            parts.append('(\"%s\",\"%s\")' % (_aspec(p['a']), _aspec(p['b'])))
    return '[' + ','.join(parts) + ']'

def scan_distance():
    a, b = S['scan_atoms']
    if a and b and a.get('xyz') and b.get('xyz'):
        return math.dist(a['xyz'], b['xyz'])
    return None

def _xyz(d):
    if not d: return None
    if d.get('xyz') is not None: return d['xyz']
    return S['_atoms'].get((str(d.get('chain') or ''), d['resn'], str(d['resi']), d['atom']))

CLUSTER_SUBS = ['all', 'opt', 'sp', 'tsopt', 'freq', 'irc', 'dft', 'scan', 'scan2d', 'scan3d',
            'path-opt', 'path-search', 'extract', 'fix-altloc', 'add-elem-info',
            'energy-diagram', 'bond-summary', 'trj2fig']
MLMM_SUBS = ['all', 'opt', 'sp', 'tsopt', 'freq', 'irc', 'dft', 'scan', 'scan2d', 'scan3d',
             'path-opt', 'path-search', 'extract', 'define-layer', 'mm-parm', 'oniom-export',
             'oniom-import', 'fix-altloc', 'add-elem-info', 'energy-diagram', 'bond-summary', 'trj2fig']
SUBS = CLUSTER_SUBS if IS_CLUSTER else MLMM_SUBS
COMPUTE = {'all', 'opt', 'sp', 'tsopt', 'freq', 'irc', 'dft', 'scan', 'scan2d', 'scan3d',
           'path-opt', 'path-search'}
MLIP_COMPUTE = COMPUTE - {'dft'}
# Subcommands a non-advanced user can run entirely from the GUI (no CLI typing):
BASIC_SUBS = ['all', 'opt', 'sp', 'tsopt', 'freq', 'irc', 'dft', 'scan', 'scan2d', 'scan3d',
              'path-opt', 'path-search']
SUB_LABELS = {
    'all': 'Full mechanism · all', 'opt': 'Geometry optimization · opt',
    'sp': 'Single-point energy · sp', 'tsopt': 'TS optimization · tsopt',
    'freq': 'Frequencies / thermochemistry · freq', 'irc': 'IRC endpoints · irc',
    'dft': 'DFT single point · dft', 'scan': '1D bond scan · scan',
    'scan2d': '2D scan grid · scan2d', 'scan3d': '3D scan grid · scan3d',
    'path-opt': 'Optimize one path · path-opt', 'path-search': 'Search a multi-step path · path-search',
    'extract': 'Extract a model · extract', 'define-layer': 'Define ONIOM layers · define-layer',
    'mm-parm': 'Build MM parameters · mm-parm', 'oniom-export': 'Export ONIOM input · oniom-export',
    'oniom-import': 'Import ONIOM result · oniom-import',
    'fix-altloc': 'Resolve alternate locations · fix-altloc',
    'add-elem-info': 'Repair element columns · add-elem-info',
    'energy-diagram': 'Draw an energy diagram · energy-diagram',
    'bond-summary': 'Compare bonds · bond-summary', 'trj2fig': 'Plot a trajectory · trj2fig',
}
def _sub_options(names): return [(SUB_LABELS.get(name, name), name) for name in names]
def _option_values(options): return [item[1] if isinstance(item, tuple) else item for item in options]
if not DFT_READY:
    SUBS = [name for name in SUBS if name != 'dft']
    BASIC_SUBS = [name for name in BASIC_SUBS if name != 'dft']
# ── SPEC: what every subcommand needs, produces and accepts ────────────────
# One declarative table drives every branch: the input requirement hint, which
# Viewer panels apply, which Options are shown, and what the Results page
# looks for. Verified against the CLI itself (click introspection) and docs/.
#   n_in   : (min, max) input files; max None = unbounded
#   panels : Viewer-page panels that apply ('center' | 'scan' | 'freeze')
#   out    : the deliverables worth pointing the user at
SPEC = {
    'all':      dict(n_in=(1, None), panels=('center', 'scan'),
                     req='R + P structures (MEP) · or 1 file + scan-lists · or 1 TS + TS-only mode',
                     out=('summary.log', 'mep.pdb', 'energy_diagram_MEP.png', 'segments/seg_NN/')),
    'extract':  dict(n_in=(1, None), panels=('center',),
                     req='a complex PDB/mmCIF + center residues (-c, required)',
                     out=('the extracted cluster model (-o)',)),
    'opt':      dict(n_in=(1, 1), panels=('freeze',), req='one structure to optimize',
                     out=('final_geometry.{xyz,pdb,cif}', 'optimization_trj.xyz')),
    'sp':       dict(n_in=(1, 1), panels=(), req='one structure (single-point E/F)',
                     out=('stdout energy', 'forces.npy', 'hessian.npy (--hess)')),
    'tsopt':    dict(n_in=(1, 1), panels=('freeze',), req='one TS-candidate structure',
                     out=('final_geometry.{xyz,pdb,cif}', 'vib/imag_*cm-1 (expect exactly one)')),
    'freq':     dict(n_in=(1, 1), panels=('freeze',), req='one optimized structure',
                     out=('frequencies_cm-1.txt', 'mode_*', 'thermoanalysis.yaml (--dump/--thermo)')),
    'irc':      dict(n_in=(1, 1), panels=('freeze',), req='one TS structure',
                     out=('*finished_irc_trj.xyz', 'forward / backward branches')),
    'dft':      dict(n_in=(1, 1), panels=(), req='one structure (DFT single-point)',
                     out=('result.yaml', 'result.json (--out-json)')),
    'scan':     dict(n_in=(1, 1), panels=('scan', 'freeze'),
                     req='1 file + scan-lists (pick atoms A & B in Viewer)',
                     out=('stage_XX/result.*', 'scan_trj.xyz')),
    'scan2d':   dict(n_in=(1, 1), panels=('scan', 'freeze'),
                     req='one structure + 2 scan axes (pick bonds + low/high in Viewer)',
                     out=('scan grid outputs',)),
    'scan3d':   dict(n_in=(1, 1), panels=('scan', 'freeze'),
                     req='one structure + 3 scan axes (pick bonds + low/high in Viewer)',
                     out=('scan grid outputs',)),
    'path-opt': dict(n_in=(2, 2), panels=('freeze',), req='exactly two structures (segment endpoints; -i takes both)',
                     out=('final_geometries_trj.xyz', 'hei.xyz (highest-energy image)')),
    'path-search': dict(n_in=(2, None), panels=('freeze',), req='two or more structures (reactant … product)',
                        out=('per-segment path outputs',)),
}
SPEC.update({
    'fix-altloc': dict(n_in=(1, 1), panels=(), req='one PDB (or directory of PDB files) with alternate locations',
                       out=('cleaned structure (-o)',)),
    'add-elem-info': dict(n_in=(1, 1), panels=(), req='one PDB with missing/incorrect element columns',
                          out=('element-repaired PDB (-o)',)),
    'energy-diagram': dict(n_in=(0, None), panels=(), req='label/energy values supplied with repeated -i',
                           out=('energy diagram image (-o)',)),
    'bond-summary': dict(n_in=(2, None), panels=(), req='two or more structures to compare',
                         out=('bond table or JSON on stdout (--json)',)),
    'trj2fig': dict(n_in=(1, 1), panels=(), req='one XYZ trajectory with per-frame energies',
                    out=('energy profile image / HTML / CSV',)),
})
SUBREQ = {k: v['req'] for k, v in SPEC.items()}
# Keep the basic GUI chemically scoped: bare MACE "small/medium/large" aliases are
# materials checkpoints, not MACE-OMOL-0 variants. Advanced users can still type a
# different --backend-model in the editable command line.
MODELS = {'mace': ['MACE-OMOL-0'],
          'uma': ['uma-s-1p2', 'uma-s-1p1', 'uma-m-1p1'],
          'orb': ['orb_v3_conservative_omol']}
DEFAULT_MODEL = {'mace': 'MACE-OMOL-0', 'uma': 'uma-s-1p2', 'orb': 'orb_v3_conservative_omol'}
# Which subcommand accepts which surfaced flag (verified against the CLI, not
# guessed). Widget name -> accepting subcommands; a flag is hidden elsewhere.
FLAG_SUBS = {
    'adv_mep':     {'all', 'path-opt', 'path-search'},
    'adv_dmf':     {'all', 'path-opt', 'path-search'},
    'adv_refine':  {'all'},                                   # --refine-path: path-opt -> path-search
    'adv_flatten': {'all', 'opt', 'tsopt'},
    'adv_thresh':  {'all', 'opt', 'tsopt', 'scan', 'scan2d', 'scan3d', 'path-opt', 'path-search'},
    'adv_maxcyc':  {'all', 'opt', 'tsopt', 'irc', 'path-opt', 'path-search'},
    'adv_radius':  {'all', 'extract'},
    'adv_dft':     {'all'},
    'adv_prec':    MLIP_COMPUTE,
    'adv_det':     MLIP_COMPUTE,
    'adv_mult':    COMPUTE,
}
TOOL_CAPABILITIES = {                    # derived view kept for existing consumers
    'mep_mode': FLAG_SUBS['adv_mep'],
    'threshold': FLAG_SUBS['adv_thresh'],
}
OUT_JSON_SUBS = {'opt', 'sp', 'tsopt', 'freq', 'irc', 'dft', 'scan',
                 'scan2d', 'scan3d', 'path-opt', 'extract'}
AUTOFILL_UTILS = {'fix-altloc', 'add-elem-info', 'bond-summary', 'trj2fig'}
_PREP_SUBS = set(COMPUTE)

def _wv(name, default=None):
    """Safely read a widget's .value by global name (widget may not exist yet)."""
    w = globals().get(name)
    return getattr(w, 'value', default) if w is not None else default

def set_subcmd(v):
    S['subcmd'] = v
    dd = globals().get('dd_subcmd')
    if dd is not None and dd.value != v: dd.value = v

def _residue_id_selector(meta):
    """Return one selector in the CLI's chain/sequence-number dialect."""
    resi = str(meta.get('resseq')) + str(meta.get('icode') or '')
    chain = str(meta.get('chain') or '').strip()
    return ('%s:%s' % (chain, resi)) if chain else resi

def _rich_residue_selector(meta):
    chain = str(meta.get('chain') or '').strip()
    resname = str(meta.get('resname') or '').strip().upper()
    resi = str(meta.get('resseq')) + str(meta.get('icode') or '')
    return '%s:%s:%s' % (chain, resname, resi) if chain else resi

def _center_cli_selectors(center_names=None, exact_ids=None, metadata=None):
    """Render -c without mixing the CLI's name and residue-ID grammars.

    Exact picks are kept as rich CHAIN:RESNAME:RESSEQ labels in the GUI.  Once
    one is present, broad residue-name selections are expanded through the
    primary input metadata so the emitted value uses one coherent CLI grammar.
    """
    names = [str(value).strip().upper() for value in
             (S.get('center', []) if center_names is None else center_names) if str(value).strip()]
    exact = [str(value).strip() for value in
             (S.get('center_ids', []) if exact_ids is None else exact_ids) if str(value).strip()]
    if not exact:
        return list(dict.fromkeys(names))
    metadata = S.get('_primary_atom_meta', []) if metadata is None else metadata
    residues = []; seen_residues = set()
    for meta in metadata:
        key = (str(meta.get('chain') or ''), str(meta.get('resname') or '').strip().upper(),
               str(meta.get('resseq')), str(meta.get('icode') or ''))
        if key not in seen_residues:
            seen_residues.add(key); residues.append((key, meta))
    selected = set(); matched_names = set()
    for key, meta in residues:
        if key[1] in names: selected.add(key); matched_names.add(key[1])
    missing = [name for name in names if name not in matched_names]
    if missing:
        raise ValueError('Return to the primary input to resolve center name(s): %s.' % ', '.join(missing))
    for value in exact:
        parts = value.split(':'); matches = []
        for key, meta in residues:
            resi = key[2] + key[3]
            if ((len(parts) == 3 and key[0] == parts[0] and key[1] == parts[1].upper() and resi == parts[2]) or
                (len(parts) == 2 and key[0] == parts[0] and resi == parts[1]) or
                (len(parts) == 1 and resi == parts[0])):
                matches.append(key)
        if not matches:
            raise ValueError('Exact center %s is not present in the primary input.' % value)
        selected.update(matches)
    chosen = [(key, meta) for key, meta in residues if key in selected]
    use_rich = bool(chosen) and all(key[0] for key, _ in chosen)
    # Both CLI selector grammars treat an omitted insertion code as “any code”.
    # Reject a token if that would silently include a residue outside the GUI set.
    for key, _ in chosen:
        if key[3]: continue
        broadened = {other for other, _meta in residues
                     if other[2] == key[2] and
                     ((other[0] == key[0] and other[1] == key[1]) if use_rich else
                      (other[0] == key[0] if key[0] else True))}
        if not broadened.issubset(selected):
            raise ValueError('The CLI cannot express exact center %s without also selecting an insertion-code sibling.' %
                             _rich_residue_selector(dict(chain=key[0], resname=key[1], resseq=key[2], icode=key[3])))
    selectors = [(_rich_residue_selector(meta) if use_rich else _residue_id_selector(meta))
                 for key, meta in chosen]
    return list(dict.fromkeys(selectors))

def _input_count_error(sub, count):
    if sub == 'all':
        mode = _wv('all_mode', 'mep')
        required = 2 if mode == 'mep' else 1
        if count != required and not (mode == 'mep' and count > required):
            return ('all MEP needs 2 or more structures.' if mode == 'mep' else
                    'all %s needs exactly 1 input structure.' % ('Scan' if mode == 'scan' else 'TS-only'))
        return ''
    bounds = SPEC.get(sub, {}).get('n_in')
    if bounds is None: return ''
    lo, hi = bounds
    if count >= lo and (hi is None or count <= hi): return ''
    limit = str(lo) if lo == hi else ('%d or more' % lo if hi is None else '%d-%d' % (lo, hi))
    return '%s needs %s input file(s).' % (sub, limit)

_ADV_OWNED_SUBS = {
    'backend_model': set(MLIP_COMPUTE), 'deterministic': set(MLIP_COMPUTE),
    'backend': set(MLIP_COMPUTE), 'charge': set(COMPUTE), 'charge_override': {'all'},
    'center_spec': {'all'}, 'substrate_pdb': {'extract'},
    'dist_freeze_raw': {'opt'}, 'do_dft': {'all'}, 'do_thermo': {'all'}, 'do_tsopt': {'all'},
    'dft_func_basis': {'all'}, 'func_basis': {'dft'},
    'dmf_backend': set(FLAG_SUBS['adv_dmf']), 'flatten': set(FLAG_SUBS['adv_flatten']),
    'freeze_atoms_text': {s for s, spec in SPEC.items() if 'freeze' in spec.get('panels', ())},
    'ligand_charge': set(COMPUTE) | {'extract'},
    'max_cycles': set(FLAG_SUBS['adv_maxcyc']), 'mep_mode': set(FLAG_SUBS['adv_mep']),
    'precision': set(MLIP_COMPUTE), 'radius': {'all', 'extract'},
    'refine_path': set(FLAG_SUBS['adv_refine']), 'spin': set(COMPUTE),
    'scan_lists_raw': {'all', 'scan'}, 'scan_list_raw': {'scan2d', 'scan3d'},
    'thresh': set(FLAG_SUBS['adv_thresh']),
}
_ADV_GENERATED = {'dry_run'}
_ADV_BLOCKED = {'help_advanced', 'tr_projection', 'verbose'}
_ADV_AUTO_IO_SUBS = set(COMPUTE) | {'extract'} | set(AUTOFILL_UTILS)
_ADV_IO_FLAGS = {'-i', '--input', '-o', '--out', '--output', '--out-dir', '--out-prefix'}
_PATH_ADVANCED = {'align','climb','conv_tol','endopt','fix_ends','max_cycles','max_nodes',
                  'max_step_size','preopt','preopt_max_cycles','reference_mode_path',
                  'relax_max_cycles','thresh'}
_POST_ENDPOINT_ADVANCED = {'irc_step_size','irc_never_stop','opt_mode_post',
                           'reject_uphill','thresh_post'}
_HESSIAN_STAGE_ADVANCED = {'hessian_calc_mode'}
_TSOPT_STAGE_ADVANCED = set() if IS_CLUSTER else {'skip_final_freq'}

def _advanced_command(sub):
    try: return PRODUCT_CLI.get_command(click.Context(PRODUCT_CLI), sub)
    except Exception: return None

def _advanced_hidden_options(sub):
    command = _advanced_command(sub)
    return tuple(getattr(command, '_advanced_hidden_options', ()) or ()) if command else ()

def _advanced_options(sub):
    """Every Click option, including the normally visible and hidden pages."""
    command = _advanced_command(sub)
    return tuple(param for param in (getattr(command, 'params', ()) or ())
                 if isinstance(param, click.Option)) if command else ()

def _advanced_status(sub, param):
    name = param.name
    if name in _ADV_BLOCKED: return 'blocked'
    if name == 'out_json' and sub in OUT_JSON_SUBS: return 'generated'
    if name in _ADV_GENERATED: return 'generated'
    if sub in _ADV_AUTO_IO_SUBS and set(param.opts + param.secondary_opts) & _ADV_IO_FLAGS: return 'owned'
    if sub in _ADV_OWNED_SUBS.get(name, set()): return 'owned'
    return 'rendered'

def _advanced_semantic_applicable(sub, name):
    if sub != 'all': return True
    mode = _wv('all_mode', 'mep')
    ts_enabled = mode == 'tsonly' or bool(_wv('w_ts', S.get('tsopt')))
    thermo_enabled = bool(_wv('w_th', S.get('thermo')))
    dft_enabled = bool(_wv('adv_dft', False))
    # Cluster workflows run IRC/endpoint optimization only with --tsopt;
    # ML/MM workflows share those post stages across TS, thermo, and DFT depth.
    post_enabled = ts_enabled if IS_CLUSTER else (ts_enabled or thermo_enabled or dft_enabled)
    if name.startswith('scan_'): return mode == 'scan'
    if mode == 'tsonly' and name in _PATH_ADVANCED: return False
    if name in _POST_ENDPOINT_ADVANCED: return post_enabled
    if name in _HESSIAN_STAGE_ADVANCED: return ts_enabled or thermo_enabled
    if name.startswith('tsopt_') or name in _TSOPT_STAGE_ADVANCED: return ts_enabled
    if name.startswith('freq_'): return thermo_enabled
    if name.startswith('dft_'): return dft_enabled
    return True

def _named_flag_applies(widget_name, sub):
    if sub not in FLAG_SUBS.get(widget_name, set()): return False
    if sub != 'all': return True
    mode = _wv('all_mode', 'mep')
    if widget_name in {'adv_mep','adv_dmf','adv_refine','adv_thresh','adv_maxcyc'}:
        return mode != 'tsonly'
    if widget_name == 'adv_flatten':
        return mode == 'tsonly' or bool(_wv('w_ts', S.get('tsopt')))
    return True

def _advanced_flag(param):
    return next((opt for opt in param.opts if opt.startswith('--')), param.opts[0])

def _advanced_argv(sub):
    saved = S.get('advanced_overrides', {}).get(sub, {})
    if not saved: return []
    command = _advanced_command(sub)
    try:
        bool_values, bool_toggles, negative_aliases, bool_single = PRODUCT_CLI._resolve_bool_options(
            click.Context(PRODUCT_CLI), sub)
    except Exception:
        bool_values, bool_toggles, negative_aliases, bool_single = set(), set(), {}, set()
    argv = []
    for param in _advanced_options(sub):
        if (_advanced_status(sub, param) != 'rendered' or param.name not in saved or
                not _advanced_semantic_applicable(sub, param.name)):
            continue
        value = saved[param.name]; flag = _advanced_flag(param)
        is_bool = param.is_bool_flag or isinstance(param.type, click.types.BoolParamType)
        if is_bool:
            if value is True:
                argv += [flag] if flag in (set(bool_toggles) | set(bool_single)) or param.is_bool_flag else [flag, 'true']
            elif value is False:
                negative = (next((opt for opt in param.secondary_opts if opt.startswith('--')), None)
                            or negative_aliases.get(flag))
                argv += [negative] if negative else [flag, 'false']
        elif param.multiple:
            values = shlex.split(value) if isinstance(value, str) else list(value)
            for item in values: argv += [flag, str(item)]
        elif value not in (None, ''):
            argv += [flag, str(value)]
    return argv

def _advanced_coverage(sub):
    return {param.name: _advanced_status(sub, param) for param in _advanced_options(sub)}

def build_cmd():
    if center_widget is not None: S['center'] = list(center_widget.value)
    if charge_rows is not None:
        S['lcharge'] = {r: x['val'].value for r, x in charge_rows.items()
                        if x['use'].value and not x.get('auto')}
    sub = _wv('dd_subcmd', S['subcmd']) or 'all'
    S['subcmd'] = sub
    _assert_distinct_pairs()
    count_error = _input_count_error(sub, len(S.get('inputs', [])))
    if count_error:
        raise ValueError(count_error + (' Put structures in reaction order.' if sub in COMPUTE else ''))
    if sub in COMPUTE or sub == 'extract' or sub in AUTOFILL_UTILS:
        missing = [path for path in S['inputs'] if not os.path.isfile(path)]
        if missing:
            raise ValueError('Re-upload missing input file(s): %s.' % ', '.join(os.path.basename(p) for p in missing))
        if S.get('model_pdb') and not os.path.isfile(S['model_pdb']):
            raise ValueError('Re-upload the missing prepared model: %s.' % os.path.basename(S['model_pdb']))
    csv_plot = sub == 'scan3d' and bool(S.get('advanced_overrides', {}).get('scan3d', {}).get('csv_path'))
    if csv_plot:
        return [CLI, 'scan3d', '-o', S['out_dir'], '--out-json'] + _advanced_argv('scan3d')
    bk = S.get('backend', BACKEND)
    if sub in MLIP_COMPUTE and bk != S.get('_installed_backend'):
        raise ValueError('Backend %s is not installed in this runtime; rerun Setup with backend=%s.' % (bk, bk))
    if sub == 'path-search' and len(S['inputs']) < 2:
        raise ValueError('path-search needs two or more structures in reaction order.')
    if sub == 'path-opt' and len(S['inputs']) != 2:
        raise ValueError('path-opt needs exactly two endpoint structures.')
    if sub == 'all':
        all_kind = _wv('all_mode', 'mep')
        if all_kind == 'mep' and len(S['inputs']) < 2:
            raise ValueError('all MEP mode needs two or more structures.')
        if all_kind == 'scan' and (len(S['inputs']) != 1 or not scan_literals()):
            raise ValueError('all scan mode needs one structure and a picked scan bond.')
        if all_kind == 'tsonly' and len(S['inputs']) != 1:
            raise ValueError('all TS-only mode needs exactly one TS candidate.')
    if sub == 'scan' and not scan_literals():
        raise ValueError('scan needs a picked atom pair and target distance.')
    if sub in ('scan2d', 'scan3d'):
        expected = 2 if sub == 'scan2d' else 3
        if len(S.get('scan_axes', [])) != expected:
            raise ValueError('%s needs exactly %d scan axes.' % (sub, expected))
    if not IS_CLUSTER and sub in COMPUTE and not S.get('parm'):
        raise ValueError('ML/MM compute commands require a matching Amber parm7 upload in Colab.')
    if (sub == 'dft' or (sub == 'all' and _wv('adv_dft', False))) and not DFT_READY:
        raise ValueError('DFT support is not installed; rerun Setup with install_dft enabled.')
    center_all = _center_cli_selectors()
    if sub == 'extract' and not center_all:
        raise ValueError('extract needs a center residue (-c); choose one in Viewer.')
    lc = ','.join('%s:%g' % (k, v) for k, v in S['lcharge'].items())
    gjf_supplies_charge = bool(S['inputs']) and all(Path(path).suffix.lower() == '.gjf' for path in S['inputs'])
    needs_system_charge = (sub in COMPUTE and not lc and
                           (sub != 'all' or S.get('mode') not in ('pdb', 'mmcif') or not center_all) and
                           not gjf_supplies_charge)
    if needs_system_charge and not S.get('charge_explicit'):
        raise ValueError('Verify the system charge (-q) in Options before running this command.')
    extra = _advanced_argv(sub)
    if sub not in COMPUTE and sub != 'extract':
        cmd = [CLI, sub]
        inputs = list(S.get('inputs', []))
        if sub == 'fix-altloc':
            src = Path(inputs[0]); cmd += ['-i', str(src), '-o', str(src.with_name(src.stem + '_altloc_fixed.pdb'))]
        elif sub == 'add-elem-info':
            src = Path(inputs[0]); cmd += ['-i', str(src)]
            # p2r's --overwrite is an explicit in-place request. Keep the safe
            # generated output unless the user selected that advanced flag.
            if not bool(S.get('advanced_overrides', {}).get(sub, {}).get('overwrite')):
                cmd += ['-o', str(src.with_name(src.stem + '_elements.pdb'))]
        elif sub == 'bond-summary':
            for path in inputs: cmd += ['-i', path]
        elif sub == 'trj2fig':
            src = Path(inputs[0])
            if src.suffix.lower() != '.xyz': raise ValueError('trj2fig needs an uploaded XYZ trajectory.')
            cmd += ['-i', str(src), '-o', str(src.with_name(src.stem + '_energy.png'))]
        return cmd + extra
    cmd = [CLI, sub, '-i', *S['inputs']]
    if sub in MLIP_COMPUTE: cmd += ['-b', bk]
    mdl = S.get('model')
    if sub in MLIP_COMPUTE and mdl and mdl != DEFAULT_MODEL.get(bk):
        cmd += ['--backend-model', mdl]
    output_arg = S['out_dir']
    if sub == 'extract' and Path(output_arg).suffix.lower() not in ('.pdb', '.ent', '.cif', '.mmcif'):
        output_arg = os.path.join(output_arg, 'cluster.pdb')
    cmd += ['-o', output_arg]
    if sub in ('all', 'extract') and center_all: cmd += ['-c', ','.join(center_all)]
    if lc: cmd += ['-l', lc]
    if not IS_CLUSTER and sub in COMPUTE: cmd += ['--parm', S['parm']]
    if sub in OUT_JSON_SUBS: cmd += ['--out-json']
    if sub in ('scan2d', 'scan3d'):
        if scan2d_literal(): cmd += ['-s', scan2d_literal()]  # one -s, per-axis (i,j,low,high) quadruples
    elif (sub == 'scan') or (sub == 'all' and all_kind == 'scan'):
        for lit in scan_literals(): cmd += ['-s', lit]        # multiple -s = staged stages
    if (sub in COMPUTE and not lc and
            (sub != 'all' or S.get('mode') not in ('pdb', 'mmcif') or not center_all) and
            (not gjf_supplies_charge or S.get('charge_explicit'))):
        cmd += ['-q', str(S['charge'])]
    m = _wv('adv_mult', 1)
    if sub in COMPUTE and m not in (None, 1): cmd += ['-m', str(int(m))]
    precision = _wv('adv_prec', 'auto')
    if sub in MLIP_COMPUTE and precision in ('fp32', 'fp64'): cmd += ['--precision', precision]
    if sub in MLIP_COMPUTE and _wv('adv_det', False): cmd += ['--deterministic']
    r = _wv('adv_radius', 0.0)
    # Radius controls cluster extraction; small-molecule `all` skips it.
    radius_applies = (sub == 'extract' or
                      (sub == 'all' and S.get('mode') in ('pdb', 'mmcif')))
    if radius_applies and r and r > 0: cmd += ['-r', str(r)]
    th = _wv('adv_thresh', '(default)')
    if _named_flag_applies('adv_thresh', sub) and th and th != '(default)': cmd += ['--thresh', th]
    mep = _wv('adv_mep', '(default)')
    if _named_flag_applies('adv_mep', sub) and mep and mep != '(default)':
        if mep == 'dmf' and not DMF_READY:
            raise ValueError('DMF needs pydmf + cyipopt, which are not installed by base Setup.')
        cmd += ['--mep-mode', mep]
        dmf_backend = _wv('adv_dmf', '(default)')
        if mep == 'dmf' and dmf_backend != '(default)':
            cmd += ['--dmf-backend', dmf_backend]
    if _named_flag_applies('adv_flatten', sub) and _wv('adv_flatten', False): cmd += ['--flatten']
    if _named_flag_applies('adv_refine', sub) and _wv('adv_refine', False): cmd += ['--refine-path']
    mc = _wv('adv_maxcyc', 0)
    if _named_flag_applies('adv_maxcyc', sub) and mc and int(mc) > 0: cmd += ['--max-cycles', str(int(mc))]
    if sub == 'opt' and freeze_pair_lit(): cmd += ['--dist-freeze', freeze_pair_lit()]
    if 'freeze' in SPEC.get(sub, {}).get('panels', ()) and S['freeze_atoms']:
        cmd += ['--freeze-atoms', ','.join(str(i) for i in S['freeze_atoms'])]
    if sub == 'all':
        if S['tsopt']: cmd.append('--tsopt')
        if S['thermo']: cmd.append('--thermo')
        if _wv('adv_dft', False):
            cmd.append('--dft')
            fb = _wv('adv_dftfb', '')
            if fb: cmd += ['--dft-func-basis', fb]
    elif sub == 'dft':
        fb = _wv('adv_dftfb', '')
        if fb: cmd += ['--func-basis', fb]
    cmd += extra
    return cmd

# -------- PyMOL-style editable command line + readiness chip ------------------
ready_chip = W.HTML()
toast = W.HTML()
_ACTION_STATE = {'running': False, 'auto_ready': False}
def _manual_command_ready():
    line = cmd_box.value.strip() if 'cmd_box' in globals() else ''
    if not line or line.startswith('#'): return False
    try: argv = shlex.split(line)
    except ValueError: return False
    return bool(argv and os.path.basename(argv[0]) == CLI)

def _current_command_subcommand():
    line = cmd_box.value.strip() if 'cmd_box' in globals() else ''
    try: argv = shlex.split(line)
    except ValueError: return ''
    if len(argv) < 2 or os.path.basename(argv[0]) != CLI: return ''
    return argv[1]

def _sync_action_enabled():
    run = globals().get('b_run'); validate = globals().get('b_validate')
    if run is None or validate is None: return
    usable = (_ACTION_STATE['auto_ready'] if _auto.get('on', True) else _manual_command_ready())
    blocked = _ACTION_STATE['running'] or not usable
    run.disabled = blocked
    validate.disabled = blocked or _current_command_subcommand() not in COMPUTE

run_status = W.HTML(value='<div role="status" aria-live="polite" aria-atomic="true" style="min-height:24px"></div>')
_RUN_STATE = {'validated_fingerprint': None, 'validation_log': '', 'kind': ''}
_RUN_TONES = {'info': '#2563eb', 'ok': '#1f7a3d', 'warn': '#7c5c00', 'error': '#a00', 'muted': '#64748b'}

def _set_run_status(text='', tone='muted', kind=''):
    """Update one consistently announced status region."""
    _RUN_STATE['kind'] = kind
    badge = (('<span style="background:%s;color:#fff;padding:2px 9px;border-radius:11px;">%s</span>' %
              (_RUN_TONES.get(tone, _RUN_TONES['muted']), html.escape(str(text)))) if text else '')
    run_status.value = ('<div role="status" aria-live="polite" aria-atomic="true" '
                        'style="min-height:24px">%s</div>' % badge)

def _command_fingerprint(text):
    return hashlib.sha256(str(text or '').encode('utf-8')).hexdigest()

def _invalidate_last_run(reason='Inputs changed; run again to populate Results.'):
    """Detach every Results widget from an obsolete compute identity."""
    had_run = bool(S.get('_last_manifest') or S.get('_last_files') or S.get('_last_log'))
    S.update(_last_out_dir=None, _last_subcmd=None, _last_argv=[], _last_files=[],
             _last_manifest={}, _last_log='', _results_notice=(reason if had_run else ''))
    _RUN_STATE['validated_fingerprint'] = None
    _RUN_STATE['validation_log'] = ''
    reuse = globals().get('w_reuse')
    if reuse is not None: reuse.value = False
    if had_run:
        status = ('command changed · validate and run again'
                  if str(reason).lower().startswith('command') else
                  'inputs changed · validate and run again')
        _set_run_status(status, 'warn', 'changed')
    guard = globals().get('_result_pick_guard')
    if guard is not None: guard['active'] = True
    try:
        for name in ('artifact_choice', 'traj_choice'):
            widget = globals().get(name)
            if widget is not None:
                widget.options = []; widget.disabled = True
        slider = globals().get('frame_slider')
        if slider is not None:
            slider.max = 0; slider.value = 0; slider.disabled = True
        play = globals().get('frame_play')
        if play is not None:
            play.max = 0; play.value = 0; play.disabled = True
        for name in ('frame_prev', 'frame_next'):
            button = globals().get(name)
            if button is not None: button.disabled = True
    finally:
        if guard is not None: guard['active'] = False
    traj = globals().get('_TRAJ')
    if traj is not None: traj.update(frames=[], energies=[], path=None, semantics={})
    for name in ('res_out', 'artifact_out', 'traj_out', 'plot_out'):
        output = globals().get(name)
        if output is not None:
            with output: clear_output()
    for name in ('artifact_fold', 'trajectory_box'):
        box = globals().get(name)
        if box is not None: box.layout.display = 'none'
    empty = globals().get('results_empty')
    if empty is not None:
        empty.layout.display = ''
        empty.value = ('<div role="status" style="padding:10px;border:1px dashed #64748b;'
                       'border-radius:10px;">%s</div>' % html.escape(reason if had_run else 'No results yet.'))
    context = globals().get('result_context')
    if context is not None: context.value = ''
    for name in ('traj_label', 'frame_state', 'trajectory_intro'):
        widget = globals().get(name)
        if widget is not None: widget.value = ''
    button = globals().get('dl_btn')
    if button is not None: button.disabled = True
    button = globals().get('res_btn')
    if button is not None:
        button.description = 'Show results'; button.disabled = True
cmd_box = W.Textarea(value='', placeholder='the generated command appears here — edit only when needed',
                     layout=W.Layout(width='99%', height='48px'))
cmd_box.add_class('rxcmd')
_OK = '<span style="background:#1f7a3d;color:#fff;padding:2px 9px;border-radius:11px;font-size:12px;">● ready to run</span>'
_NO = '<span role="status" style="background:#64748b;color:#fff;padding:2px 9px;border-radius:11px;font-size:12px;">○ %s</span>'
_MANUAL = '<span role="status" style="background:#7c5c00;color:#fff;padding:2px 9px;border-radius:11px;font-size:12px;">◆ add utility arguments below</span>'
_MANUAL_EDIT = '<span role="status" style="background:#7c5c00;color:#fff;padding:2px 9px;border-radius:11px;font-size:12px;">◆ manual command · GUI changes are detached</span>'
_auto = {'on': True, 'guard': False}
def _has_current_run():
    return bool(S.get('_last_manifest') or S.get('_last_files') or S.get('_last_log'))
def _matches_last_run(text):
    if not S.get('_last_argv'): return not _has_current_run()
    try: return shlex.split(str(text or '')) == list(S['_last_argv'])
    except ValueError: return False
def _set_cmd(txt):
    _auto['guard'] = True; cmd_box.value = txt; _auto['guard'] = False
    if _has_current_run() and not _matches_last_run(txt):
        _invalidate_last_run('Command changed; run again to populate Results.')
def refresh(_=None):
    command_ready = False
    try:
        built = build_cmd()
        if _auto['on']: _set_cmd(' '.join(shlex.quote(c) for c in built))
        is_utility = len(built) > 1 and built[1] not in COMPUTE and built[1] != 'extract'
        manual_utility = is_utility and built[1] not in AUTOFILL_UTILS
        ready_chip.value = (_MANUAL if manual_utility and _auto['on'] else
                            (_OK if _auto['on'] else _MANUAL_EDIT))
        command_ready = not manual_utility
    except Exception as e:
        if _auto['on']: _set_cmd('# ' + str(e))
        ready_chip.value = _NO % html.escape(str(e))
    _ACTION_STATE['auto_ready'] = command_ready
    _sync_action_enabled()
    _rs = globals().get('_render_summary')
    if _rs is not None: _rs()
    _rc = globals().get('_render_chips')
    if _rc is not None: _rc()
    _ro = globals().get('_render_output_note')
    if _ro is not None: _ro()
    valid = _RUN_STATE.get('validated_fingerprint')
    valid_command = valid[0] if isinstance(valid, tuple) else valid
    if valid and valid_command != _command_fingerprint(cmd_box.value):
        _RUN_STATE['validated_fingerprint'] = None
        _RUN_STATE['validation_log'] = ''
        _set_run_status('command changed · validate again', 'warn', 'changed')
def _on_cmd_edit(_):
    if not _auto['guard']:
        _auto['on'] = False   # user took manual control of the line
        ready_chip.value = _MANUAL_EDIT
        _sync_action_enabled()
        if _has_current_run():
            _invalidate_last_run('Command changed; run again to populate Results.')
        elif _RUN_STATE.get('validated_fingerprint'):
            _RUN_STATE['validated_fingerprint'] = None
            _RUN_STATE['validation_log'] = ''
            _set_run_status('command changed · validate again', 'warn', 'changed')
cmd_box.observe(_on_cmd_edit, names='value')

# ============================================================== INPUT tab
input_msg = W.HTML()
def _goto(i): _tab_go(i)

def _clear_structure_bound_state():
    """Clear every atom/residue selection before the input identity can change."""
    global center_widget, charge_rows
    _invalidate_last_run('Input identity changed; validate and run the new system.')
    center_widget = None; charge_rows = None
    S.update(center=[], center_ids=[], lcharge={}, model_pdb=None, _pre_extract=None,
             scan_atoms=[None, None], scan_preset='', scan_stages=[], scan_axes=[],
             freeze_buf=[None, None], freeze_pairs=[], freeze_atoms=[], measure_atoms=[],
             charge=0, charge_explicit=False, _pdb_path=None, _pdb_text='', _view_format='pdb',
             _hetero=[], _atoms={}, _atom_meta=[], _last_out_dir=None,
             _last_subcmd=None, _last_argv=[], _last_files=[], _last_manifest={}, _last_log='',
             _last_pick=None, _pick_history=[],
             _last_pick_message='', _last_pick_tone='ok',
             _view_input_index=0, _view_mapping_ok=True,
             _primary_atom_signatures=[], _primary_atom_meta=[])
    charge_widget = globals().get('w_q')
    if charge_widget is not None: charge_widget.value = 0
    charge_ok = globals().get('w_charge_ok')
    if charge_ok is not None: charge_ok.value = False
    S['charge_explicit'] = False
    revert = globals().get('b_revert')
    if revert is not None: revert.layout.display = 'none'
    for name, message in (('center_panel', 'Load a primary PDB/mmCIF to choose center residues.'),
                          ('charge_panel', 'Ligand charges appear after the primary structure loads.')):
        panel = globals().get(name)
        if panel is not None: panel.children = [W.HTML('<small>%s</small>' % message)]

def load_pdb(paths, parm=None, center=None, lcharge=None, scan_preset='', mode='pdb',
             append=False, keep_subcmd=False):
    previous_subcmd = S.get('subcmd', 'all')
    target_subcmd = previous_subcmd if (append or keep_subcmd) else 'all'
    paths = [p for p in paths if p]
    if append:
        _invalidate_last_run('Input files changed; validate and run the updated system.')
        paths = list(S.get('inputs', [])) + [p for p in paths if p not in S.get('inputs', [])]
        S.update(inputs=paths, parm=(parm or S.get('parm')), mode=mode, subcmd=target_subcmd)
        if center is not None: S['center'] = list(center)
        if lcharge is not None: S['lcharge'] = dict(lcharge)
        if scan_preset: S['scan_preset'] = scan_preset
    else:
        _clear_structure_bound_state()
        S.update(inputs=paths, parm=parm, mode=mode, subcmd=target_subcmd,
                 center=list(center or []), center_ids=[], lcharge=dict(lcharge or {}),
                 scan_atoms=[None, None], scan_preset=scan_preset)
    set_subcmd(target_subcmd)
    am = globals().get('all_mode')
    mode_state = globals().get('_ALL_MODE_STATE', {})
    if (am is not None and not keep_subcmd and target_subcmd == 'all' and
            (not append or not mode_state.get('user'))):
        mode_state['sync'] = True
        try: am.value = 'mep' if len(paths) >= 2 else 'scan'
        finally: mode_state['sync'] = False
    _riq = globals().get('_render_input_queue')
    if _riq is not None: _riq()
    build_selection()
    _scc = globals().get('_sync_capability_controls')
    if _scc is not None: _scc()
    refresh()   # stay on Input: more files may still be added

_acc = ('.pdb,.ent,.cif,.mmcif,.xyz,.gjf,.csv' if IS_CLUSTER else
        '.pdb,.ent,.cif,.mmcif,.parm7,.xyz,.gjf,.com,.inp,.csv')
_drop_formats = ('.pdb / .cif / .mmcif / .xyz / .gjf / .csv' if IS_CLUSTER else
                 '.pdb / .cif / .mmcif + .parm7 · utility .xyz / .gjf / .com / .inp / .csv')
upl = W.FileUpload(accept=_acc, multiple=True, description='Upload files', icon='upload',
                   layout=W.Layout(width='220px'))
def _save_upload(name, content):
    """Write one browser file to a collision-free path without an overwrite race."""
    clean = os.path.basename(str(name).replace('\\', '/')) or 'upload.dat'
    if '\x00' in clean: raise ValueError('Invalid upload filename.')
    path = Path(clean)
    for number in range(1, 10000):
        target = path if number == 1 else path.with_name('%s_%d%s' % (path.stem, number, path.suffix))
        try:
            with open(target, 'xb') as fh: fh.write(memoryview(content))
            return str(target)
        except FileExistsError:
            continue
        except Exception:
            try: target.unlink()
            except OSError: pass
            raise
    raise RuntimeError('Could not allocate a unique name for %s.' % clean)

def _discard_rejected_uploads(paths):
    """Remove only newly saved browser files that no UI role accepted."""
    attached = {os.path.abspath(path) for path in list(S.get('inputs', [])) +
                [p for p in (S.get('parm'), S.get('model_pdb')) if p]}
    for path in paths:
        if os.path.abspath(path) not in attached and os.path.isfile(path): os.remove(path)

def _remember_uploaded_paths(paths):
    """Track only browser-created runtime copies; source/example paths stay external."""
    S.setdefault('_uploaded_paths', set()).update(
        os.path.abspath(str(path)) for path in paths if path)

def _delete_owned_uploads(paths):
    """Delete detached browser copies without touching user/source/example paths."""
    owned = S.setdefault('_uploaded_paths', set())
    for path in paths:
        if not path: continue
        absolute = os.path.abspath(str(path))
        if absolute not in owned: continue
        try:
            if os.path.isfile(absolute): os.remove(absolute)
        finally:
            owned.discard(absolute)

def _reset_file_upload(widget):
    value = widget.value
    if isinstance(value, dict):
        value.clear()
        if hasattr(widget, '_counter'): widget._counter = 0
    else:
        widget.value = ()

def _preflight_structures(paths):
    for path in paths:
        _load_view_structure(path)

def _rebind_missing_uploads(loaded):
    remaining = list(loaded); rebound = []
    references = [('input', i, path) for i, path in enumerate(S.get('inputs', [])) if not os.path.isfile(path)]
    if S.get('parm') and not os.path.isfile(S['parm']): references.append(('parm', None, S['parm']))
    if S.get('model_pdb') and not os.path.isfile(S['model_pdb']): references.append(('model', None, S['model_pdb']))
    def bind(ref, candidate):
        kind, index, old = ref
        if kind == 'input':
            paths = list(S.get('inputs', [])); paths[index] = candidate; S['inputs'] = paths
        elif kind == 'parm': S['parm'] = candidate
        else: S['model_pdb'] = candidate
        remaining.remove(candidate); references.remove(ref)
        rebound.append('%s → %s' % (os.path.basename(old), os.path.basename(candidate)))
    for candidate in list(remaining):
        exact = [ref for ref in references if os.path.basename(ref[2]) == os.path.basename(candidate)]
        if len(exact) == 1: bind(exact[0], candidate)
    for candidate in list(remaining):
        if candidate.lower().endswith('.parm7'):
            matches = [ref for ref in references if ref[0] == 'parm']
            if len(matches) == 1: bind(matches[0], candidate)
    input_refs = [ref for ref in references if ref[0] == 'input']
    structures = [p for p in remaining if p.lower().endswith(('.pdb', '.ent', '.cif', '.mmcif'))]
    if input_refs and len(input_refs) == len(structures):
        for ref, candidate in zip(input_refs, list(structures)): bind(ref, candidate)
    if rebound:
        _invalidate_last_run('Missing session files were re-attached; validate before running.')
    return remaining, rebound

def _ingest_saved_files(loaded, source='upload'):
    """Attach one saved browser/drop batch without replacing earlier batches."""
    loaded = [str(path) for path in loaded if path]
    structures = [p for p in loaded if p.lower().endswith(('.pdb', '.ent', '.cif', '.mmcif'))]
    parm_files = [p for p in loaded if p.lower().endswith('.parm7')]
    smalls = [p for p in loaded if p.lower().endswith(('.xyz', '.gjf'))]
    csv_files = [p for p in loaded if p.lower().endswith('.csv')]
    known = set(structures + parm_files + smalls + csv_files)
    unsupported = [p for p in loaded if p not in known]
    mixed_utility = bool(csv_files) and len(loaded) != 1
    if unsupported or len(parm_files) > 1 or (structures and smalls) or (parm_files and IS_CLUSTER) or mixed_utility:
        reason = ('parm7 belongs in the mlmm notebook, not pdb2reaction' if parm_files and IS_CLUSTER else
                  'unsupported file: %s' % ', '.join(os.path.basename(p) for p in unsupported)
                  if unsupported else 'attach one file type at a time')
        input_msg.value = '<div role="alert" style="color:#991b1b">%s; existing files were kept.</div>' % html.escape(reason)
        return False
    try:
        _preflight_structures(structures)
    except Exception as exc:
        input_msg.value = ('<div role="alert" style="color:#991b1b">Structure was not attached: %s</div>' %
                           html.escape(str(exc)))
        return False
    loaded, rebound = _rebind_missing_uploads(loaded)
    if rebound and not loaded:
        _render_input_queue()
        if S.get('inputs') and os.path.isfile(S['inputs'][0]): build_selection()
        refresh()
        input_msg.value = '✅ re-attached <b>%s</b>' % html.escape(', '.join(rebound))
        return True
    structures = [p for p in loaded if p.lower().endswith(('.pdb', '.ent', '.cif', '.mmcif'))]
    parm_files = [p for p in loaded if p.lower().endswith('.parm7')]
    smalls = [p for p in loaded if p.lower().endswith(('.xyz', '.gjf'))]
    csv_files = [p for p in loaded if p.lower().endswith('.csv')]
    parm = parm_files[0] if parm_files else None
    if csv_files:
        _clear_structure_bound_state()
        csv_path = csv_files[0]
        S.update(mode='utility', inputs=[csv_path], subcmd='scan3d')
        S.setdefault('advanced_overrides', {}).setdefault('scan3d', {})['csv_path'] = csv_path
        set_subcmd('scan3d')
        input_msg.value = '✅ 3D scan surface <b>%s</b> attached for plot-only mode.' % html.escape(csv_path)
        _render_input_queue(); _sync_capability_controls(); render_viewer(); refresh()
    elif structures:
        if S.get('inputs') and S.get('mode') not in (None, 'pdb', 'mmcif'):
            input_msg.value = '<div role="alert" style="color:#991b1b">Remove the current small-molecule files before adding structures.</div>'
            return False
        adding = bool(S.get('inputs'))
        load_pdb(structures, parm=(parm or S.get('parm')), append=adding)
        input_msg.value = '✅ %s <b>%s</b> (%s)' % (
            'appended' if adding else 'loaded', html.escape(', '.join(structures)), html.escape(source))
    elif smalls:
        if S.get('inputs') and S.get('mode') not in (None, 'small'):
            input_msg.value = '<div role="alert" style="color:#991b1b">Remove the current structures before adding small molecules.</div>'
            return False
        adding = bool(S.get('inputs'))
        if adding:
            merged = list(S.get('inputs', [])) + [p for p in smalls if p not in S.get('inputs', [])]
            _invalidate_last_run('Input files changed; validate and run the updated system.')
        else:
            merged = smalls; _clear_structure_bound_state()
        S.update(mode='small', inputs=merged, charge_explicit=False)
        if not adding and S.get('rep') == 'cartoon':
            S['rep'] = 'ball+stick'
            _rep_widget = globals().get('dd_rep')
            if _rep_widget is not None and _rep_widget.value == 'cartoon':
                _rep_widget.value = 'ball+stick'
        set_subcmd('opt' if len(merged) == 1 else 'all')
        _scc = globals().get('_sync_capability_controls')
        if _scc is not None: _scc()
        input_msg.value = '✅ %s <b>%s</b> (%s) — set total charge in Options.' % (
            'appended' if adding else 'loaded', html.escape(', '.join(smalls)), html.escape(source))
        _render_input_queue(); build_selection(); refresh()
    elif parm:
        _invalidate_last_run('Amber topology changed; validate and run again.')
        S['parm'] = parm; _render_input_queue(); refresh()
        input_msg.value = '✅ topology <b>%s</b> attached (%s)' % (html.escape(parm), html.escape(source))
    return bool(structures or smalls or csv_files or parm)

def _accept_upload_pairs(pairs, source):
    renamed, loaded = [], []
    try:
        for name, content in pairs:
            target = _save_upload(name, content); loaded.append(target)
            if target != os.path.basename(name): renamed.append('%s → %s' % (name, target))
        accepted = _ingest_saved_files(loaded, source)
    except Exception as exc:
        accepted = False
        input_msg.value = ('<div role="alert" style="color:#991b1b">Upload failed: %s; '
                           'existing files were kept.</div>' % html.escape(str(exc)))
    if not accepted: _discard_rejected_uploads(loaded)
    else: _remember_uploaded_paths(loaded)
    if accepted and renamed:
        input_msg.value += ('<br><small>Existing files kept; uploaded as %s</small>' %
                            html.escape(', '.join(renamed)))
    return bool(accepted)

_upload_guard = {'active': False}
def _on_upload(change):
    if _upload_guard['active']: return
    items = change.get('new', upl.value)
    if not items: return
    raw = list(items.items()) if isinstance(items, dict) else [(f['name'], f) for f in items]
    pairs = [(name, meta['content']) for name, meta in raw]
    try:
        _accept_upload_pairs(pairs, 'file picker')
    finally:
        _upload_guard['active'] = True
        try: _reset_file_upload(upl)
        finally: _upload_guard['active'] = False

upl.observe(_on_upload, names='value')

_EX = (['BezA methyltransferase - cluster MEP (R->P)',
        'Aromatic Claisen rearrangement - small molecule']
       if IS_CLUSTER else ['methyltransferase complex - scan mode'])
ex_choice = W.Dropdown(options=_EX, value=_EX[0], description='example',
                       style={'description_width': 'initial'},
                       layout=W.Layout(width='400px', max_width='100%'))
ex_btn = W.Button(description='Load example', icon='flask', layout=W.Layout(width='150px'))
def _load_example(_):
    sel = ex_choice.value
    if IS_CLUSTER and sel.startswith('Aromatic Claisen'):
        try:
            reactant = _example_file('aromatic_claisen/reactant.xyz')
            product = _example_file('aromatic_claisen/product.xyz')
        except Exception as exc:
            input_msg.value = '⚠️ could not fetch the example: %s' % exc
            return
        _queue_change(clear=True)
        _ingest_saved_files([reactant, product], 'example')
        S.update(charge=0, charge_explicit=True)
        set_subcmd('all'); all_mode.value = 'mep'
        _scc = globals().get('_sync_capability_controls')
        if _scc is not None: _scc()
        wq = globals().get('w_q')
        if wq is not None: wq.value = 0
        input_msg.value = '✅ Aromatic Claisen rearrangement ready (reactant → product; charge 0).'
        _render_input_queue(); refresh(); return
    if IS_CLUSTER:
        try:
            _r, _p = _example_file('1.R.pdb'), _example_file('3.P.pdb')
        except Exception as exc:
            input_msg.value = '⚠️ could not fetch the example: %s' % exc; return
        _queue_change(clear=True)
        input_msg.value = '⭐ BezA methyltransferase (R→P MEP).'
        load_pdb([_r, _p], center=['SAM', 'GPP', 'MG'], lcharge={'SAM': 1, 'GPP': -3})
    else:
        try:
            _c = _example_file('methyltransferase/complex.pdb')
            _pm = _example_file('methyltransferase/complex.parm7')
        except Exception as exc:
            input_msg.value = '⚠️ could not fetch the example: %s' % exc; return
        S['scan_target'] = 1.3
        input_msg.value = '⭐ Methyltransferase (scan mode, shipped parm7 → no AmberTools).'
        load_pdb([_c], parm=_pm,
                 center=['SAM', 'PHN'], lcharge={'SAM': 1, 'PHN': -1},
                 scan_preset="[('SAM 359 CS1','PHN 360 C8',1.3)]")
ex_btn.on_click(_load_example)
_input_formats = _drop_formats
_drop_prompt = W.HTML(
    '<div class="rxdrop-prompt"><b>Drop files here</b><br><small>%s</small></div>' %
    html.escape(_input_formats))
_DROP_STATE = {'generation': 0, 'seen': set()}
_drop_epoch = W.HTML(
    '<span class="rxdrop-epoch-value" data-generation="0" aria-hidden="true"></span>')
_drop_epoch.layout.display = 'none'

def _claim_drop_batch(batch, generation):
    batch = str(batch or '')
    try: generation = int(generation)
    except (TypeError, ValueError): return False, 'invalid generation'
    if generation != _DROP_STATE['generation']: return False, 'stale generation'
    if not batch: return False, 'missing batch id'
    if batch in _DROP_STATE['seen']: return False, 'duplicate batch'
    _DROP_STATE['seen'].add(batch)
    return True, ''

def _bump_drop_generation():
    _DROP_STATE['generation'] += 1
    _DROP_STATE['seen'].clear()
    _drop_epoch.value = (
        '<span class="rxdrop-epoch-value" data-generation="%d" '
        'aria-hidden="true"></span>' % _DROP_STATE['generation'])

_drop = W.VBox([_drop_prompt, upl, _drop_epoch],
               layout=W.Layout(width='100%', align_items='center', justify_content='center'))
_drop.add_class('rxdrop')
input_file_rows = W.VBox(layout=W.Layout(width='100%'))
input_order_note = W.HTML()

def _queue_change(path=None, delta=0, remove=False, clear=False, kind='input'):
    if kind == 'parm':
        _invalidate_last_run('Amber topology removed; validate and run again.')
        previous_parm = S.get('parm'); S['parm'] = None
        _delete_owned_uploads([previous_parm])
        input_msg.value = '<small>Topology removed.</small>'
        _render_input_queue(); refresh(); return
    if kind == 'model':
        clear_model = globals().get('_clear_uploaded_model')
        if clear_model is not None: clear_model()
        return
    old_paths = list(S.get('inputs', [])); paths = list(old_paths)
    old_aux_paths = [S.get('parm'), S.get('model_pdb')]
    old_view_index = max(0, min(int(S.get('_view_input_index', 0)),
                                max(0, len(old_paths) - 1)))
    old_view_path = old_paths[old_view_index] if old_paths else None
    if clear:
        _bump_drop_generation()
        paths = []; S['parm'] = None; S['model_pdb'] = None
    elif path in paths:
        i = paths.index(path)
        if remove: paths.pop(i)
        else:
            j = max(0, min(len(paths) - 1, i + delta))
            paths[i], paths[j] = paths[j], paths[i]
    primary_changed = old_paths[:1] != paths[:1]
    if old_view_path in paths:
        S['_view_input_index'] = paths.index(old_view_path)
        displayed_changed = False
    else:
        S['_view_input_index'] = min(old_view_index, max(0, len(paths) - 1))
        new_view_path = paths[S['_view_input_index']] if paths else None
        displayed_changed = old_view_path != new_view_path
    if primary_changed:
        _clear_structure_bound_state()
    else:
        _invalidate_last_run('Input order changed; validate and run again.')
        if displayed_changed:
            S.update(_last_pick=None, _pick_history=[], _last_pick_message='',
                     _last_pick_tone='ok')
    detached = [item for item in old_paths if item not in paths]
    if clear: detached.extend(old_aux_paths)
    _delete_owned_uploads(detached)
    S['inputs'] = paths
    input_msg.value = ('<small>Primary input changed — residue/atom selections, charge, and prepared model were cleared.</small>'
                       if primary_changed else
                       '<small>Input order updated; selections tied to input 1 were kept.</small>')
    if not paths:
        S.update(mode=None, _pdb_text='', _pdb_path=None, _view_format='pdb', _atom_meta=[], _atoms={})
    _render_input_queue()
    if paths and S.get('mode') in ('pdb', 'mmcif', 'small'): build_selection()
    else: render_viewer()
    _scc = globals().get('_sync_capability_controls')
    if _scc is not None: _scc()
    refresh()

def _file_row(path, role, index=None, total=None, kind='input'):
    label = W.HTML('<span style="display:inline-block;overflow:hidden;text-overflow:ellipsis;'
                   'white-space:nowrap;max-width:100%%" title="%s"><b>%s</b> · %s</span>' %
                   (html.escape(path), html.escape(role), html.escape(os.path.basename(path))))
    label.add_class('rxfilename')
    controls = []
    if kind == 'input':
        up = W.Button(description='Move earlier', tooltip='Move earlier', disabled=(index == 0),
                      layout=W.Layout(width='38px'))
        down = W.Button(description='Move later', tooltip='Move later', disabled=(index == total - 1),
                        layout=W.Layout(width='38px'))
        up.add_class('rxmove-earlier'); down.add_class('rxmove-later')
        up.on_click(lambda _, p=path: _queue_change(path=p, delta=-1))
        down.on_click(lambda _, p=path: _queue_change(path=p, delta=1))
        controls.extend([up, down])
    close = W.Button(description='Remove file', tooltip='Remove %s' % os.path.basename(path),
                     layout=W.Layout(width='38px'))
    close.add_class('rxremove-file')
    close.on_click(lambda _, p=path, k=kind: _queue_change(path=p, remove=True, kind=k))
    controls.append(close)
    row = W.HBox([label, W.HBox(controls, layout=W.Layout(flex_flow='row nowrap'))],
                 layout=W.Layout(width='100%', flex_flow='row nowrap', align_items='center'))
    row.add_class('rxfile')
    return row

def _render_input_queue():
    paths = list(S.get('inputs', [])); rows = []
    reaction_order = (_wv('dd_subcmd', S.get('subcmd', 'all')) in ('path-opt', 'path-search') or
                      (_wv('dd_subcmd', S.get('subcmd', 'all')) == 'all' and
                       _wv('all_mode', 'mep') == 'mep'))
    for i, path in enumerate(paths):
        role = ('Input' if len(paths) == 1 or not reaction_order else
                ('Reactant' if i == 0 else ('Product' if i == len(paths) - 1 else 'Intermediate %d' % i)))
        if len(paths) > 1 and not reaction_order: role = 'Input %d' % (i + 1)
        rows.append(_file_row(path, '%d · %s' % (i + 1, role), i, len(paths)))
    if S.get('parm'): rows.append(_file_row(S['parm'], 'topology', kind='parm'))
    if S.get('model_pdb'): rows.append(_file_row(S['model_pdb'], 'prepared model', kind='model'))
    input_file_rows.children = rows or [W.HTML('<small style="color:#64748b">No files attached yet.</small>')]
    if not paths:
        order = '<b>No structures loaded</b>'
    elif len(paths) == 1:
        order = '<b>1 structure</b> · single-structure input'
    else:
        order = ('<b>%d structures</b> · reaction order shown above' % len(paths) if reaction_order else
                 '<b>%d input files</b>' % len(paths))
    if S.get('parm'):
        order += ' · topology: <code>%s</code>' % html.escape(os.path.basename(S['parm']))
    input_order_note.value = '<small>%s</small>' % order
    clear_button = globals().get('b_clear_inputs')
    if clear_button is not None: clear_button.disabled = not bool(paths or S.get('parm') or S.get('model_pdb'))
    sync_view = globals().get('_sync_view_input_widget')
    if sync_view is not None: sync_view()
b_clear_inputs = W.Button(description='clear files', icon='eraser', layout=W.Layout(width='120px'))
b_clear_inputs.on_click(lambda _: _queue_change(clear=True))
_render_input_queue()
_input_box_children = [
    W.HTML('<b>Input files</b> <small>· reorder when the workflow uses reaction order; remove with ×</small>'),
    _drop, input_file_rows, W.HBox([b_clear_inputs]), input_order_note,
    W.HTML('<hr style="margin:6px 0">'), W.HBox([ex_choice, ex_btn]), input_msg]

# Colab renders ipywidgets' selection containers (Tab, Accordion) as an empty
# block, so every collapsible below is a Button + VBox that works everywhere.
def _info_markup(tip, revision=0):
    """Browser-native disclosure: click opens the panel; hover shows the same help."""
    plain = ' '.join(str(tip).split())
    safe = html.escape(plain, quote=True)
    return ('<details class="rxinfo-details" data-revision="%d">'
            '<summary aria-label="More information: %s" title="%s">&#9432;</summary>'
            '<div class="rxhelp-panel" role="note"><small>%s</small></div></details>' %
            (int(revision), safe, safe, safe))

_INFO_CONTROLS = weakref.WeakSet()
def _close_info(control=None):
    """Rebuild one or all disclosures in their closed state."""
    controls = [control] if control is not None else list(_INFO_CONTROLS)
    for item in controls:
        if item is not None and hasattr(item, '_rx_tip'):
            item._rx_info_revision = getattr(item, '_rx_info_revision', 0) + 1
            item.value = _info_markup(item._rx_tip, item._rx_info_revision)

def _info_control(tip, target=None):
    """A native details/summary info disclosure that works without Python clicks."""
    control = target if target is not None else W.HTML()
    control._rx_tip = str(tip)
    control._rx_info_revision = getattr(control, '_rx_info_revision', 0) + 1
    control.value = _info_markup(control._rx_tip, control._rx_info_revision)
    control.add_class('rxinfo')
    _INFO_CONTROLS.add(control)
    return control

def _set_info_text(control, tip):
    control._rx_tip = str(tip)
    control._rx_info_revision = getattr(control, '_rx_info_revision', 0) + 1
    control.value = _info_markup(control._rx_tip, control._rx_info_revision)

def _close_info_target(target):
    _close_info(target)

def _hdr(content, tip):
    row = W.HBox([W.HTML(content), _info_control(tip)],
                 layout=W.Layout(width='100%', flex_flow='row wrap', align_items='center'))
    row.add_class('rxhelp-row'); return row

def _flag_row(widget, tip, info_target=None):
    row = W.HBox([widget, _info_control(tip, info_target)],
                 layout=W.Layout(width='100%', flex_flow='row wrap', align_items='center'))
    row.layout.display = widget.layout.display or ''
    widget._rx_flag_row = row
    row.add_class('rxflagrow'); return row

def _set_flag_visible(widget, visible):
    display_value = '' if visible else 'none'
    widget.layout.display = display_value
    row = getattr(widget, '_rx_flag_row', None)
    if row is not None: row.layout.display = display_value


def _collapsible(title, child, on_open=None):
    _btn = W.Button(layout=W.Layout(width='auto'))
    _body = W.VBox([child])
    _st = {'open': False}
    def _sync():
        _body.layout.display = '' if _st['open'] else 'none'
        _btn.description = ('Hide ' if _st['open'] else 'Show ') + title
    def _click(_):
        _st['open'] = not _st['open']; _sync()
        if _st['open'] and on_open is not None: on_open()
    _btn.on_click(_click); _sync()
    _box = W.VBox([_btn, _body]); _box.add_class('rxfold')
    def _set_open(opened=True):
        changed = _st['open'] != bool(opened)
        _st['open'] = bool(opened); _sync()
        if changed and _st['open'] and on_open is not None: on_open()
    _box._rx_set_open = _set_open; _box._rx_button = _btn; _box._rx_body = _body
    return _box

if not IS_CLUSTER:
    _input_box_children.insert(5, _collapsible('Optional prepared ML-region model', model_upload_box))
input_box = W.VBox(_input_box_children)

# ============================================================== VIEWER (3D pick)
viewer_out = W.Output()
viewer_status = W.HTML()
view_input = W.Dropdown(options=[], description='viewing', style={'description_width': 'initial'},
                        layout=W.Layout(width='300px', max_width='100%'))
view_input_note = W.HTML()
_view_input_guard = {'active': False}
center_panel, charge_panel, scan_panel, freeze_panel = (W.VBox() for _ in range(4))
for _p in (center_panel, charge_panel, scan_panel, freeze_panel): _p.add_class('rxcard')
center_ids_html = W.HTML()
center_widget = None


def _center_values():
    """The center picker's options are (label, value) pairs so ligands can be
    flagged in the label; callers that match on residue names need the values."""
    if center_widget is None: return []
    return [o[1] if isinstance(o, tuple) else o for o in center_widget.options]
charge_rows = None
_PICK_ACTIONS = (
    ('Center residue (-c)', 'center', 'center'), ('Ligand charge (-l)', 'ligand', 'center'),
    ('Scan bond · atom A', 'scanA', 'scan'), ('Scan bond · atom B', 'scanB', 'scan'),
    ('Freeze pair · atom A', 'freezeA', 'freeze'), ('Freeze pair · atom B', 'freezeB', 'freeze'),
    ('Freeze atom (Cartesian)', 'freezeatom', 'freeze'),
)
pick_action = W.Dropdown(
    options=[(label, value) for label, value, _panel in _PICK_ACTIONS],
    value='center', description='Click action', style={'description_width': 'initial'},
    layout=W.Layout(width='360px', max_width='100%'))
_PICK_ACTION_STATE = {'user': False, 'sync': False}
def _remember_pick_action(change):
    if not _PICK_ACTION_STATE['sync'] and change.get('old') != change.get('new'):
        _PICK_ACTION_STATE['user'] = True
pick_action.observe(_remember_pick_action, names='value')
exact_atom = W.Text(value='', description='exact atom',
                    placeholder='1-based index or A:SAM:359:C1',
                    style={'description_width': 'initial'},
                    layout=W.Layout(width='360px', max_width='100%'))
exact_atom_btn = W.Button(description='set current pick', icon='crosshairs',
                          layout=W.Layout(width='165px'),
                          tooltip='Apply “Click sets” to this exact atom without using WebGL.')
exact_atom_msg = W.HTML()

_MOLSTAR_VERSION = '5.6.1'
_MOLSTAR_JS = ('https://cdn.jsdelivr.net/npm/molstar@%s/build/viewer/molstar.js'
               % _MOLSTAR_VERSION)
_MOLSTAR_CSS = ('https://cdn.jsdelivr.net/npm/molstar@%s/build/viewer/molstar.css'
                % _MOLSTAR_VERSION)
_VIEWER_GENERATION = {'value': 0}
_INCREMENTAL_PICK = {'active': False}

def _pick_text(pick=None):
    pick = pick or S.get('_last_pick') or {}
    residue = '%s:%s:%s' % (pick.get('chain'), pick.get('resn'), pick.get('resi')) \
        if pick.get('chain') else '%s:%s' % (pick.get('resn'), pick.get('resi'))
    suffix = ' · atom #%d' % (int(pick['index']) + 1) \
        if pick.get('index') is not None and int(pick['index']) >= 0 else ''
    return '%s · %s%s' % (residue, pick.get('atom') or '?', suffix)

def _pick_key(pick):
    if not pick: return None
    try: return ('index', int(pick.get('index')))
    except (TypeError, ValueError):
        return ('atom', str(pick.get('chain') or ''), str(pick.get('resn') or '').upper(),
                str(pick.get('resi') or ''), str(pick.get('atom') or '').upper())

def _remember_pick(pick):
    """Keep each transient Mol* pick once, in click order."""
    history = S.setdefault('_pick_history', [])
    key = _pick_key(pick)
    for position, previous in enumerate(history):
        if _pick_key(previous) == key:
            history[position] = pick
            return False
    history.append(pick)
    return True

def _molstar_document(source, fmt='pdb', *, interactive=False, generation=0,
                      show_water=False, show_sequence=None):
    """Build an isolated, pinned Mol* viewer using its standard interface."""
    callback_ns = 'pdb2reaction_gui'
    config = {
        'format': ('xyz' if fmt == 'xyz' else
                   ('mmcif' if fmt in ('cif', 'mmcif') else 'pdb')),
        'interactive': bool(interactive),
        'generation': int(generation),
        'callback': callback_ns if interactive else '',
        'showWater': bool(show_water),
        'showSequence': bool(fmt != 'xyz' if show_sequence is None else show_sequence),
    }
    template = r"""<!doctype html>
<html>
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<link rel="stylesheet" href="__MOLSTAR_CSS__">
<style>
html,body,#molstar-host{width:100%;height:100%;margin:0;overflow:hidden;background:#fff}
body{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Helvetica,Arial,sans-serif}
#molstar-host{position:absolute;inset:0}
#molstar-state{position:absolute;z-index:20;inset:0;display:flex;align-items:center;
 justify-content:center;background:#f8fafc;color:#475569;font-size:13px}
#molstar-state.error{color:#991b1b;padding:18px;text-align:center;box-sizing:border-box}
.msp-plugin{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Helvetica,Arial,sans-serif}
</style>
</head>
<body>
<div id="molstar-host"></div>
<div id="molstar-state" role="status">Loading structure…</div>
<script src="__MOLSTAR_JS__"></script>
<script>
(async function(){
  'use strict';
  const cfg=__CONFIG__, source=__SOURCE__;
  const stateNode=document.getElementById('molstar-state');
  let ready=false, viewer=null, ignoreStartupEmpty=true, clickQueue=Promise.resolve();

  function kernel(){
    const candidates=[window,window.parent,window.top];
    for(const candidate of candidates){
      try{
        if(candidate&&candidate.google&&candidate.google.colab&&candidate.google.colab.kernel)
          return candidate.google.colab.kernel;
      }catch(_error){}
    }
    return null;
  }
  function invoke(suffix,args){
    const target=kernel();
    if(!target||!cfg.callback)return Promise.resolve(null);
    return target.invokeFunction(cfg.callback+'.'+suffix,args,{});
  }
  function properties(loc){
    const SP=molstar.lib.structure.StructureProperties;
    return {
      sourceIndex:Number(SP.atom.sourceIndex(loc)),
      serial:Number(SP.atom.id(loc)),
      chain:String(SP.chain.auth_asym_id(loc)||'').trim(),
      seq:Number(SP.residue.auth_seq_id(loc)),
      ins:String(SP.residue.pdbx_PDB_ins_code(loc)||'').trim(),
      resn:String(SP.residue.auth_comp_id(loc)||SP.residue.label_comp_id(loc)||'').trim(),
      atom:String(SP.atom.auth_atom_id(loc)||SP.atom.label_atom_id(loc)||'').trim(),
      element:String(SP.atom.type_symbol(loc)||'').trim()
    };
  }
  async function setElementThemeAndCartoonAlpha(){
    const manager=viewer.plugin.managers.structure.component;
    const structures=viewer.plugin.managers.structure.hierarchy.current.structures;
    for(const structure of structures){
      await manager.updateRepresentationsTheme(structure.components,{
        color:'element-symbol',
        colorParams:{carbonColor:{name:'element-symbol',params:{}}}
      });
    }
    const state=viewer.plugin.state.data, update=state.build();
    let changed=false;
    for(const structure of structures){
      for(const component of structure.components){
        if(component.key!=='structure-component-static-polymer')continue;
        for(const representation of component.representations||[]){
          const params=representation.cell&&representation.cell.transform.params;
          if(!params||!params.values||params.values.type.name!=='cartoon')continue;
          update.to(representation.cell).update(old=>({
            ...old,type:{...old.type,params:{...old.type.params,alpha:0.55}}
          }));
          changed=true;
        }
      }
    }
    if(changed)await update.commit();
  }
  async function configureWater(){
    const hierarchy=viewer.plugin.managers.structure.hierarchy;
    for(const structure of hierarchy.current.structures){
      let component=structure.components.find(
        item=>item.key==='structure-component-static-water'
      );
      if(!component&&cfg.showWater){
        const water=await viewer.plugin.builders.structure.tryCreateComponentStatic(
          structure.cell,'water',{label:'Water'}
        );
        if(water)await viewer.plugin.builders.structure.representation.addRepresentation(
          water,{type:'ball-and-stick',color:'element-symbol'}
        );
        continue;
      }
      if(!component)continue;
      const hidden=!!(component.cell&&component.cell.state&&component.cell.state.isHidden);
      if(hidden===cfg.showWater)
        viewer.plugin.managers.structure.component.toggleVisibility([component]);
      if(cfg.showWater&&!component.representations.length)
        await viewer.plugin.managers.structure.component.addRepresentation(
          [component],'ball-and-stick'
        );
    }
  }


  try{
    if(!window.molstar||!molstar.Viewer)
      throw new Error('Mol* did not load. Check the network connection and rerun Launch GUI.');
    viewer=window.rxMolstarViewer=await molstar.Viewer.create('molstar-host',{
      layoutIsExpanded:false,
      layoutShowControls:true,
      layoutShowRemoteState:false,
      layoutShowSequence:cfg.showSequence,
      layoutShowLog:false,
      layoutShowLeftPanel:false,
      collapseLeftPanel:true,
      collapseRightPanel:true,
      viewportBackgroundColor:'#ffffff'
    });
    await viewer.loadStructureFromData(source,cfg.format,{dataLabel:'input structure'});
    await setElementThemeAndCartoonAlpha();
    await configureWater();
    viewer.plugin.managers.camera.reset({},0);
    ready=true;stateNode.style.display='none';

    if(cfg.interactive){
      async function handleClick(event){
        if(!ready)return;
        const loci=event&&event.current&&event.current.loci;
        if(!loci||loci.kind==='empty-loci'){
          if(ignoreStartupEmpty){ignoreStartupEmpty=false;return;}
          await invoke('clear_highlights',[cfg.generation]);
          return;
        }
        ignoreStartupEmpty=false;
        if(loci.kind!=='element-loci')return;
        const SE=molstar.lib.structure.StructureElement;
        const loc=SE.Loci.getFirstLocation(loci);
        if(!loc)return;
        const item=properties(loc), exact=SE.Loci.size(loci)===1;
        await invoke('on_click',[
          String(item.sourceIndex),item.resn,String(item.seq),item.chain,item.atom,
          String(item.serial),item.ins,true,cfg.generation,exact
        ]);
      }
      viewer.subscribe(viewer.plugin.behaviors.interaction.click,event=>{
        clickQueue=clickQueue.then(()=>handleClick(event)).catch(error=>console.error(error));
      });
    }
    window.addEventListener('resize',()=>viewer&&viewer.handleResize());
  }catch(error){
    console.error(error);
    stateNode.className='error';
    stateNode.textContent=String(error&&error.message||error);
  }
})();
</script>
</body>
</html>"""
    return (template
            .replace('__MOLSTAR_CSS__', _MOLSTAR_CSS)
            .replace('__MOLSTAR_JS__', _MOLSTAR_JS)
            .replace('__CONFIG__', json.dumps(config, separators=(',', ':')).replace('<', '\\u003c'))
            .replace('__SOURCE__', json.dumps(str(source)).replace('<', '\\u003c')))

def _molstar_iframe(source, fmt='pdb', **kwargs):
    document = _molstar_document(source, fmt, **kwargs)
    return ('<iframe class="rxmolstar-frame" title="Mol* molecular viewer" '
            'srcdoc="%s" allow="fullscreen" '
            'style="width:100%%;aspect-ratio:4/3;border:1px solid #dfe6ef;'
            'border-radius:10px;background:#fff;"></iframe>' %
            html.escape(document, quote=True))

def render_viewer():
    _VIEWER_GENERATION['value'] += 1
    generation = _VIEWER_GENERATION['value']
    if not S.get('_pdb_text'):
        viewer_status.layout.display = ''
        viewer_status.value = (
            '<div role="status" style="min-height:180px;display:flex;align-items:center;'
            'justify-content:center;text-align:center;border:1px dashed #cbd5e1;'
            'border-radius:12px;background:#f8fafc;color:#64748b;padding:16px;">'
            '<div><b>No structure loaded</b><br><small>Load a structure in Input, '
            'then use Mol* here.</small></div></div>')
        with viewer_out: clear_output()
        return
    viewer_status.value = ''
    viewer_status.layout.display = 'none'
    with viewer_out:
        clear_output()
        display(HTML(_molstar_iframe(
            S['_pdb_text'], S.get('_view_format', 'pdb'),
            interactive=True, generation=generation,
            show_water=S.get('show_water', False))))

def _render_center_ids():
    ids = S.get('center_ids', [])
    center_ids_html.value = ('<small>exact residues: <code>%s</code></small>' % ','.join(ids)) if ids else ''
    center_ids_html.layout.display = '' if ids else 'none'

def _resolve_click_meta(atom_index, serial='', resn='', resi='', chain='', atom='', icode=''):
    """Map a browser atom to retained input metadata without trusting source-index order."""
    metadata = S.get('_atom_meta', [])
    if S.get('_view_format') == 'xyz':
        try: return metadata[int(atom_index)]
        except (IndexError, TypeError, ValueError): return None
    norm = lambda value: str(value or '').strip()
    try: serial_int = int(serial)
    except (TypeError, ValueError): serial_int = None
    if serial_int is not None:
        found = [meta for meta in metadata if meta.get('serial') == serial_int]
        if len(found) == 1: return found[0]
    expected = (norm(chain), norm(resn).upper(), norm(resi), norm(icode), norm(atom).upper())
    if any(expected):
        found = [meta for meta in metadata
                 if (norm(meta.get('chain')), norm(meta.get('resname')).upper(),
                     norm(meta.get('resseq')), norm(meta.get('icode')),
                     norm(meta.get('name')).upper()) == expected]
        if len(found) == 1: return found[0]
        return None
    try: return metadata[int(atom_index)]
    except (IndexError, TypeError, ValueError): return None

def on_click(atom_index, resn='', resi='', chain='', atom='', serial='', icode='',
             live_marked=False, viewer_generation=None, exact=True):
    if viewer_generation is not None:
        try: current_generation = int(viewer_generation)
        except (TypeError, ValueError): return
        if current_generation != _VIEWER_GENERATION['value']: return
    meta = _resolve_click_meta(atom_index, serial, resn, resi, chain, atom, icode)
    if meta is None:
        exact_atom_msg.value = ('<small role="alert" style="color:#991b1b">Could not map this viewer atom '
                                'to the input; workflow selections were not changed.</small>')
        return
    try: viewer_index = int(atom_index)
    except (TypeError, ValueError): viewer_index = -1
    index = int(meta['index'])
    resn = str(meta.get('resname') or resn).strip()
    resi = str(meta.get('resseq') if meta.get('resseq') is not None else resi)
    icode = str(meta.get('icode') or icode).strip(); resi += icode
    chain = str(meta.get('chain') or chain).strip()
    atom = str(meta.get('name') or atom).strip()
    serial = meta.get('serial', serial); xyz = meta.get('xyz')
    exact_atom_msg.value = ''
    act = pick_action.value
    if exact in (False, 0, 'false', 'False') and act not in ('center', 'ligand'):
        S['_last_pick'] = {'chain': chain, 'resn': resn, 'resi': resi, 'atom': 'residue',
                           'serial': serial, 'icode': icode, 'xyz': xyz, 'index': index,
                           'viewer_index': viewer_index, 'action': act}
        S['_last_pick_message'] = ('Mol* focused this residue. Click one atom in its '
                                   'ball-and-stick view to assign the active role.')
        S['_last_pick_tone'] = 'warn'
        renderer = globals().get('_render_last_pick_status')
        if renderer is not None: renderer()
        refresh()
        return
    pick = {'chain': chain, 'resn': resn, 'resi': resi, 'atom': atom,
            'serial': serial, 'icode': icode, 'xyz': xyz, 'index': index,
            'viewer_index': viewer_index, 'action': act}
    message, tone = '', 'ok'
    if S.get('_view_input_index', 0) and not S.get('_view_mapping_ok', False):
        message, tone = 'view-only: atom identifiers differ from the first input; return to R/input 1 to edit selections', 'warn'
    elif act == 'center':
        rid = ('%s:%s:%s' % (chain, resn, resi)) if chain else str(resi)
        narrowed = False
        if center_widget is not None and resn in set(center_widget.value):
            _INCREMENTAL_PICK['active'] = True
            try:
                center_widget.value = tuple(value for value in center_widget.value if value != resn)
            finally:
                _INCREMENTAL_PICK['active'] = False
            narrowed = True
        elif resn in S.get('center', []):
            S['center'] = [value for value in S['center'] if value != resn]
            narrowed = True
        if rid not in S['center_ids']:
            S['center_ids'].append(rid); _render_center_ids()
            message = ('replaced the %s name selection with this exact center (-c)' % resn
                       if narrowed else 'added as an exact center (-c)')
        else:
            message = 'already selected as an exact center (-c)'
    elif act == 'ligand' and charge_rows is not None and resn in charge_rows:
        row = charge_rows[resn]
        if row.get('auto'):
            message = 'uses the built-in %s charge (%+g)' % (resn, row['val'].value)
        else:
            row['use'].value = True; message = 'enabled its ligand-charge (-l) row'
    elif act == 'ligand':
        message, tone = 'not a detected ligand/cofactor; choose a hetero residue', 'warn'
    elif act in ('scanA', 'scanB'):
        slot = 0 if act == 'scanA' else 1
        picked = {'chain': chain, 'resn': resn, 'resi': resi, 'atom': atom, 'xyz': xyz, 'index': index}
        if slot == 1 and _same_atom(S['scan_atoms'][0], picked):
            message, tone = 'scan atom B must differ from atom A', 'warn'
        else:
            S['scan_atoms'][slot] = picked
            message = 'set scan atom %s' % ('A' if slot == 0 else 'B')
            if slot == 0:
                pick_action.value = 'scanB'; message += '; next click sets atom B'
        _render_scan_panel()
    elif act in ('freezeA', 'freezeB'):
        slot = 0 if act == 'freezeA' else 1
        picked = {'chain': chain, 'resn': resn, 'resi': resi, 'atom': atom, 'xyz': xyz, 'index': index}
        if slot == 1 and _same_atom(S['freeze_buf'][0], picked):
            message, tone = 'restraint endpoint B must differ from endpoint A', 'warn'
        else:
            S['freeze_buf'][slot] = picked
            message = 'set restraint endpoint %s' % ('A' if slot == 0 else 'B')
            if slot == 0:
                pick_action.value = 'freezeB'; message += '; next click sets endpoint B'
        _render_freeze_panel()
    elif act == 'freezeatom':
        idx = index + 1 if index >= 0 else None
        if idx is not None and idx not in S['freeze_atoms']:
            S['freeze_atoms'].append(idx); _render_freeze_panel(); message = 'added frozen atom #%d' % idx
        else:
            message = 'atom is already frozen' if idx is not None else 'could not resolve the atom index'
            tone = 'warn'
    S['_last_pick'] = pick
    _remember_pick(pick)
    S['_last_pick_message'] = message or 'selected'
    S['_last_pick_tone'] = tone
    _rlp = globals().get('_render_last_pick_status')
    if _rlp is not None: _rlp()
    # Mol* owns live click rendering in the browser. Only the manual exact-atom
    # fallback rebuilds the iframe from Python.
    if not live_marked:
        render_viewer()
    refresh()

def _clear_highlights_from_browser(viewer_generation=None):
    if viewer_generation is not None:
        try: current_generation = int(viewer_generation)
        except (TypeError, ValueError): return
        if current_generation != _VIEWER_GENERATION['value']: return
    S.update(_last_pick=None, _pick_history=[], _last_pick_message='', _last_pick_tone='ok')
    renderer = globals().get('_render_last_pick_status')
    if renderer is not None: renderer()
    refresh()

try:
    from google.colab import output as _co
    _co.register_callback('pdb2reaction_gui.on_click', on_click)
    _co.register_callback('pdb2reaction_gui.clear_highlights', _clear_highlights_from_browser)
except Exception:
    _co = None              # click->Python only registers inside Google Colab

def _render_scan_panel():
    a, b = S['scan_atoms']
    def fmt(atom): return _aspec(atom) if atom else 'not set'
    distance = scan_distance()
    pair_ready = bool(a and b and not _same_atom(a, b))
    pair_state = ('<span style="color:#166534"><b>Ready</b> · current distance %.2f Å</span>' % distance
                  if pair_ready and distance is not None else
                  '<span style="color:#92400e">Pick atom A, then atom B.</span>')
    def _choose(slot):
        def activate(_):
            if not _view_is_editable(): return
            _PICK_ACTION_STATE['sync'] = True
            try: pick_action.value = 'scanA' if slot == 0 else 'scanB'
            finally: _PICK_ACTION_STATE['sync'] = False
            _render_scan_panel()
        return activate
    pick_a = W.Button(description='1  Pick atom A', layout=W.Layout(width='128px'))
    pick_b = W.Button(description='2  Pick atom B', layout=W.Layout(width='128px'))
    pick_a.button_style = 'primary' if pick_action.value == 'scanA' else ''
    pick_b.button_style = 'primary' if pick_action.value == 'scanB' else ''
    pick_a.on_click(_choose(0)); pick_b.on_click(_choose(1))
    def _clear_pair(_):
        if not _view_is_editable(): return
        S['scan_atoms'] = [None, None]; S['scan_preset'] = ''
        _PICK_ACTION_STATE['sync'] = True
        try: pick_action.value = 'scanA'
        finally: _PICK_ACTION_STATE['sync'] = False
        _render_scan_panel(); render_viewer(); refresh()
    clear_pair = W.Button(description='Clear pair', layout=W.Layout(width='105px'))
    clear_pair.on_click(_clear_pair)
    pair_controls = W.HBox([pick_a, pick_b, clear_pair], layout=W.Layout(flex_flow='row wrap'))
    pair_html = W.HTML(
        '<small>A: <code>%s</code><br>B: <code>%s</code><br>%s</small>' %
        (html.escape(fmt(a)), html.escape(fmt(b)), pair_state))
    sub_now = _wv('dd_subcmd', S['subcmd'])
    if sub_now in ('scan2d', 'scan3d'):
        need = 2 if sub_now == 'scan2d' else 3
        low = W.BoundedFloatText(value=1.2, min=0.3, max=6.0, step=0.05,
                                 description='3  Low Å', layout=W.Layout(width='158px'))
        high = W.BoundedFloatText(value=3.0, min=0.3, max=8.0, step=0.05,
                                  description='High Å', layout=W.Layout(width='150px'))
        axis_note = W.HTML()
        def _add_axis(_):
            if not _view_is_editable(): return
            if not pair_ready:
                axis_note.value = '<small role="alert" style="color:#991b1b">Pick two different atoms first.</small>'; return
            if low.value >= high.value:
                axis_note.value = '<small role="alert" style="color:#991b1b">Low must be smaller than high.</small>'; return
            if len(S['scan_axes']) >= need:
                axis_note.value = '<small role="alert" style="color:#991b1b">All axes are already defined.</small>'; return
            key = tuple(sorted((a.get('index'), b.get('index'))))
            existing = {tuple(sorted((axis['a'].get('index'), axis['b'].get('index'))))
                        for axis in S['scan_axes']}
            if key in existing:
                axis_note.value = '<small role="alert" style="color:#991b1b">That atom pair is already an axis.</small>'; return
            S['scan_axes'].append({'a': a, 'b': b, 'lo': low.value, 'hi': high.value})
            _render_scan_panel(); refresh()
        def _remove_axis(_):
            if not _view_is_editable(): return
            if S['scan_axes']: S['scan_axes'].pop()
            _render_scan_panel(); refresh()
        def _clear_axes(_):
            if not _view_is_editable(): return
            S['scan_axes'] = []; _render_scan_panel(); refresh()
        add_axis = W.Button(description='4  Add axis', button_style='info',
                            layout=W.Layout(width='120px'), disabled=len(S['scan_axes']) >= need)
        remove_axis = W.Button(description='Remove last', layout=W.Layout(width='120px'),
                               disabled=not S['scan_axes'])
        clear_axes = W.Button(description='Clear axes', layout=W.Layout(width='105px'))
        add_axis.on_click(_add_axis); remove_axis.on_click(_remove_axis); clear_axes.on_click(_clear_axes)
        axes = '<br>'.join(
            'axis %d: %s ↔ %s &nbsp; %.3g–%.3g Å' %
            (index + 1, html.escape(_aspec(axis['a'])), html.escape(_aspec(axis['b'])),
             axis['lo'], axis['hi'])
            for index, axis in enumerate(S['scan_axes'])
        ) or 'No axes added.'
        colour = '#166534' if len(S['scan_axes']) == need else '#92400e'
        scan_panel.children = [
            W.HTML('<b>%s coordinate grid</b> <small>· define %d atom-pair axes</small>' %
                   (sub_now, need)),
            pair_controls, pair_html, W.HBox([low, high]),
            W.HBox([add_axis, remove_axis, clear_axes]), axis_note,
            W.HTML('<small style="color:%s"><b>Axes %d/%d</b><br>%s</small>' %
                   (colour, len(S['scan_axes']), need, axes))]
        return
    target = W.BoundedFloatText(value=float(S['scan_target']), min=0.3, max=6.0, step=0.05,
                                description='3  Target Å', layout=W.Layout(width='178px'))
    def _set_target(_):
        if not _view_is_editable(): return
        S['scan_target'] = target.value; refresh()
    target.observe(_set_target, names='value')
    def _add_stage(_):
        if not _view_is_editable() or not pair_ready: return
        S['scan_stages'].append([{'a': a, 'b': b, 't': S['scan_target']}])
        _render_scan_panel(); refresh()
    def _add_concerted(_):
        if not _view_is_editable() or not pair_ready: return
        if not S['scan_stages']: S['scan_stages'].append([])
        S['scan_stages'][-1].append({'a': a, 'b': b, 't': S['scan_target']})
        _render_scan_panel(); refresh()
    def _clear_stages(_):
        if not _view_is_editable(): return
        S['scan_stages'] = []; _render_scan_panel(); refresh()
    add_stage = W.Button(description='Add sequential stage', layout=W.Layout(width='165px'))
    add_concerted = W.Button(description='Add to current stage', layout=W.Layout(width='165px'))
    clear_stages = W.Button(description='Clear stages', layout=W.Layout(width='110px'))
    add_stage.disabled = add_concerted.disabled = not pair_ready
    add_stage.on_click(_add_stage); add_concerted.on_click(_add_concerted)
    clear_stages.on_click(_clear_stages)
    stages = '<br>'.join(
        'stage %d: %s' %
        (index + 1, ' &amp; '.join(
            '%s ↔ %s → %.3g Å' %
            (html.escape(_aspec(bond['a'])), html.escape(_aspec(bond['b'])), bond['t'])
            for bond in stage))
        for index, stage in enumerate(S['scan_stages'])
    ) or 'No extra stages.'
    stage_more = _collapsible(
        'Multiple bonds / stages',
        W.VBox([W.HBox([add_stage, add_concerted, clear_stages]),
                W.HTML('<small>%s</small>' % stages)]))
    stage_more._rx_set_open(bool(S['scan_stages']))
    note = W.HTML('' if len(S.get('inputs', [])) == 1 else
                  '<small style="color:#92400e">A scan coordinate is used only by a single-structure scan.</small>')
    scan_panel.children = [
        W.HTML('<b>Scan coordinate</b> <small>· A–B distance</small>'),
        pair_controls, pair_html, target, note, stage_more]

def _render_freeze_panel():
    fa, fb = S['freeze_buf']
    def fmt(x): return _aspec(x) if x else '— pick —'
    def _addpair(_):
        if fa and fb and not _same_atom(fa, fb):
            S['freeze_pairs'].append({'a': fa, 'b': fb, 't': None}); S['freeze_buf'] = [None, None]
            _render_freeze_panel(); refresh()
    def _clrpairs(_): S['freeze_pairs'] = []; _render_freeze_panel(); refresh()
    def _clratoms(_): S['freeze_atoms'] = []; _render_freeze_panel(); render_viewer(); refresh()
    b_ap = W.Button(description='add freeze pair', icon='plus', layout=W.Layout(width='160px'))
    b_cp = W.Button(description='clear pairs', layout=W.Layout(width='110px'))
    b_ca = W.Button(description='clear atoms', layout=W.Layout(width='110px'))
    b_ap.on_click(_addpair); b_cp.on_click(_clrpairs); b_ca.on_click(_clratoms)
    pairs = '; '.join('%s↔%s' % (_aspec(p['a']), _aspec(p['b'])) for p in S['freeze_pairs']) or '(none)'
    atoms = ','.join(str(i) for i in S['freeze_atoms']) or '(none)'
    freeze_panel.children = [
        W.HTML('<b>Distance restraints</b> <small>(<code>--dist-freeze</code>, opt) — set pick mode to '
               '“Freeze pair · A/B”, click two atoms, then add</small>'),
        W.HTML('A: <code>%s</code> &nbsp; B: <code>%s</code>' % (fmt(fa), fmt(fb))),
        W.HBox([b_ap, b_cp]),
        W.HTML('<small>pairs: <code>%s</code></small>' % pairs),
        W.HTML('<hr style="margin:5px 0"><b>Frozen atoms</b> <small>(<code>--freeze-atoms</code>; '
               'this panel appears only for accepting subcommands) — pick mode “Freeze atom”, click atoms</small>'),
        W.HBox([b_ca]),
        W.HTML('<small>1-based indices: <code>%s</code></small>' % atoms)]

def _view_role(index, total):
    reaction_order = (_wv('dd_subcmd', S.get('subcmd', 'all')) in ('path-opt', 'path-search') or
                      (_wv('dd_subcmd', S.get('subcmd', 'all')) == 'all' and
                       _wv('all_mode', 'mep') == 'mep'))
    if total <= 1 or not reaction_order: return 'Input %d' % (index + 1)
    if index == 0: return 'Reactant'
    if index == total - 1: return 'Product'
    return 'Intermediate %d' % index

def _sync_view_input_widget():
    paths = list(S.get('inputs', []))
    index = max(0, min(int(S.get('_view_input_index', 0)), max(0, len(paths) - 1)))
    S['_view_input_index'] = index
    opts = [('%s · %s' % (_view_role(i, len(paths)), os.path.basename(path)), i)
            for i, path in enumerate(paths)]
    _view_input_guard['active'] = True
    try:
        view_input.options = opts
        if opts: view_input.value = index
    finally:
        _view_input_guard['active'] = False

def _atom_signatures(metadata):
    return [(str(m.get('chain') or ''), str(m.get('resname') or '').upper(),
             str(m.get('resseq')), str(m.get('icode') or ''), str(m.get('name') or '').upper())
            for m in metadata]

def _resolve_atom_query(query):
    query = str(query or '').strip()
    if not query: raise ValueError('Enter a 1-based atom index or CHAIN:RESNAME:RESSEQ:ATOM.')
    metadata = S.get('_atom_meta', [])
    if query.isdigit():
        index = int(query) - 1
        if 0 <= index < len(metadata): return index
        raise ValueError('Atom index %s is outside 1–%d.' % (query, len(metadata)))
    parts = [part.strip() for part in query.split(':')]
    if len(parts) == 4: chain, resn, resi, atom = parts
    elif len(parts) == 3: chain, (resn, resi, atom) = '', parts
    else: raise ValueError('Use CHAIN:RESNAME:RESSEQ[ICODE]:ATOM (or omit CHAIN).')
    found = []
    for meta in metadata:
        meta_resi = str(meta.get('resseq')) + str(meta.get('icode') or '')
        if (str(meta.get('chain') or '') == chain and
                str(meta.get('resname') or '').upper() == resn.upper() and
                meta_resi == resi and str(meta.get('name') or '').upper() == atom.upper()):
            found.append(int(meta['index']))
    if len(found) == 1: return found[0]
    if len(found) > 1: raise ValueError('That atom is ambiguous; include the chain ID.')
    raise ValueError('No retained atom matches %s.' % query)

def _apply_exact_atom(_=None):
    try:
        index = _resolve_atom_query(exact_atom.value)
    except ValueError as exc:
        exact_atom_msg.value = '<small role="alert" style="color:#991b1b">%s</small>' % html.escape(str(exc))
        return
    exact_atom_msg.value = ''
    on_click(str(index))
exact_atom_btn.on_click(_apply_exact_atom)

def _view_is_editable():
    return int(S.get('_view_input_index', 0)) == 0 or bool(S.get('_view_mapping_ok'))

def _set_primary_editor_enabled(enabled):
    disabled = not bool(enabled)
    if center_widget is not None: center_widget.disabled = disabled
    if charge_rows is not None:
        for row in charge_rows.values():
            row['use'].disabled = disabled or row.get('auto', False)
            row['val'].disabled = disabled or row.get('auto', False)

def _set_widget_tree_disabled(widget, disabled):
    for child in getattr(widget, 'children', ()):
        if hasattr(child, 'disabled'): child.disabled = bool(disabled)
        _set_widget_tree_disabled(child, disabled)

def _set_selection_editor_enabled(enabled):
    _set_primary_editor_enabled(enabled)
    for panel in (scan_panel, freeze_panel): _set_widget_tree_disabled(panel, not enabled)
    for button in getattr(chips_box, 'children', ()):
        if hasattr(button, 'disabled'): button.disabled = not enabled
    for name in ('b_extract', 'b_clear'):
        button = globals().get(name)
        if button is not None: button.disabled = not enabled

def build_selection():
    """Load and commit one view atomically; primary editors remain primary-owned."""
    global center_widget, charge_rows
    if S['mode'] not in ('pdb', 'mmcif', 'small') or not S['inputs']:
        render_viewer()
        return False
    _sync_view_input_widget()
    view_index = int(S.get('_view_input_index', 0))
    path = S['inputs'][view_index]
    is_small = S.get('mode') == 'small'
    try:
        if is_small:
            text, metadata, viewer_path = _load_small_view_structure(path)
            allr, het = [], []
            atoms = {(str(meta['index']),): meta['xyz'] for meta in metadata}
        else:
            text, metadata, viewer_path = _load_view_structure(path)
            allr, het = parse_residues(text, metadata)
            atoms = parse_atoms(text, metadata)
        signatures = _atom_signatures(metadata)
    except Exception as exc:
        with viewer_out: clear_output(); print('Could not prepare structure for the viewer:', exc)
        input_msg.value = '❌ structure load failed: <code>%s</code>' % html.escape(str(exc))
        return False
    S.update(_pdb_path=viewer_path, _pdb_text=text,
             _view_format='xyz' if is_small else 'pdb', _atom_meta=metadata,
             _hetero=het, _atoms=atoms)
    if view_index == 0:
        S['_primary_atom_signatures'] = signatures
        S['_primary_atom_meta'] = [dict(meta) for meta in metadata]
        S['_view_mapping_ok'] = True
        view_input_note.value = ''
    else:
        S['_view_mapping_ok'] = bool(S.get('_primary_atom_signatures')) and signatures == S['_primary_atom_signatures']
        view_input_note.value = (('<small style="color:#166534">Atom identifiers match input 1; '
                                  'picks update the shared workflow selection.</small>')
                                 if S['_view_mapping_ok'] else
                                 ('<small role="alert" style="color:#92400e">View-only: atom identifiers/order '
                                  'differ from input 1. Clicks still highlight atoms and residues, but cannot '
                                  'change the workflow; return to R/input 1 to edit selections.</small>'))
    if view_index == 0 and not is_small:
        initial_center = tuple(x for x in S.get('center', []) if x in het)
        residue_copies = {}
        for meta in metadata:
            rn = str(meta.get('resname') or '').upper()
            if rn in het:
                residue_copies.setdefault(rn, set()).add(
                    (str(meta.get('chain') or ''), str(meta.get('resseq')),
                     str(meta.get('icode') or '')))
        _center_opts = [('%s · all %d copies' % (r, len(residue_copies[r])), r)
                        if len(residue_copies.get(r, ())) > 1 else (r, r) for r in het]
        center_widget = W.SelectMultiple(options=_center_opts, value=initial_center,
                                         rows=min(5, max(1, len(het))), layout=W.Layout(width='230px'))
        charge_rows = {}; rows = []
        for rn in het:
            auto = rn in _ION_CHARGES
            pre = _ION_CHARGES[rn] if auto else S.get('lcharge', {}).get(rn, 0)
            use = W.Checkbox(value=(auto or rn in S.get('lcharge', {})),
                             description=('auto %s' if auto else 'use %s') % rn,
                             disabled=auto, indent=False, layout=W.Layout(width='105px'))
            val = W.BoundedFloatText(value=float(pre), min=-9, max=9, step=1,
                                     description='%s charge' % rn, style={'description_width': 'initial'},
                                     disabled=auto, layout=W.Layout(width='165px'))
            charge_rows[rn] = {'use': use, 'val': val, 'auto': auto}; rows.append(W.HBox([use, val]))
        def _sync_center(change=None):
            if not _view_is_editable(): return
            S['center'] = list(center_widget.value)
            _render_center_ids()
            if not _INCREMENTAL_PICK['active']: render_viewer()
            refresh()
        def _sync_charge_rows(_=None):
            if not _view_is_editable(): return
            S['lcharge'] = {r: x['val'].value for r, x in charge_rows.items()
                            if x['use'].value and not x.get('auto')}
            refresh()
        center_widget.observe(_sync_center, names='value')
        for x in charge_rows.values():
            x['use'].observe(_sync_charge_rows, names='value'); x['val'].observe(_sync_charge_rows, names='value')
        center_picker = (center_widget if het else
                         W.HTML('<small>No ligand names found — click an exact residue in 3D.</small>'))
        center_panel.children = [
            _hdr('<b>center <code>-c</code></b>',
                 'Choose a ligand/cofactor name for all matching copies, or click one exact residue in 3D.'),
            center_picker, center_ids_html]
        charge_list = (W.VBox(rows, layout=W.Layout(max_height='140px', overflow='auto', width='100%'))
                       if rows else W.HTML('<i>no hetero ligands</i>'))
        charge_panel.children = [
            _hdr('<b>ligand charges <code>-l</code></b>',
                 'Known ions use the built-in locked charge (for example MG is +2). -l sets only unknown ligand residue names; -q sets the total system charge.'),
            charge_list]
        _sync_charge_rows()
    _render_center_ids(); _render_scan_panel(); _render_freeze_panel()
    _set_selection_editor_enabled(_view_is_editable())
    if view_index == 0 or S.get('_view_mapping_ok'):
        _remap_stored_atom_coordinates(metadata)
    render_viewer()
    return True

def _on_view_input(change):
    if _view_input_guard['active'] or change.get('new') is None: return
    previous = {key: S.get(key) for key in ('_view_input_index', '_last_pick', '_pick_history',
                                            '_last_pick_message', '_last_pick_tone')}
    previous_note = view_input_note.value
    S.update(_view_input_index=int(change['new']), _last_pick=None, _pick_history=[],
             _last_pick_message='', _last_pick_tone='ok')
    if not build_selection():
        S.update(previous); view_input_note.value = previous_note
        _sync_view_input_widget(); render_viewer()
    _render_last_pick_status()
view_input.observe(_on_view_input, names='value')

# Mol* owns representation, colour, camera, measurement, and screenshot controls.
# These hidden widgets keep older saved settings loadable without duplicating that UI.
dd_rep = W.Dropdown(options=['cartoon', 'sticks', 'ball+stick', 'spheres', 'line'],
                    value='cartoon')
dd_col = W.Dropdown(options=['element', 'chain', 'spectrum'], value='element')
cb_water = W.Checkbox(value=False, description='water', indent=False,
                      layout=W.Layout(width='82px'),
                      tooltip='Show or hide Mol*’s Water component.')
def _on_water(change):
    S['show_water'] = bool(change['new'])
    if not _SESSION_APPLY['active']: render_viewer()
cb_water.observe(_on_water, names='value')
cb_surf = W.Checkbox(value=False, description='surface')
cb_spin = W.Checkbox(value=False, description='spin')
dd_size = W.Dropdown(options=[640, 720, 800], value=720)
for _legacy_view_widget in (dd_rep, dd_col, cb_surf, cb_spin, dd_size):
    _legacy_view_widget.layout.display = 'none'
view_controls = W.HBox([
    cb_water,
    _info_control('Mol* starts with water hidden. The standard Components panel can also '
                  'show or hide Water and change every molecular representation.')
], layout=W.Layout(flex_flow='row wrap', align_items='center'))

_sel_help = ''
summary_html = W.HTML()
def _render_summary():
    cw = center_widget
    cen = ','.join(list(cw.value) if cw is not None else S.get('center', []))
    ids = ','.join(S.get('center_ids', []))
    lc = ','.join('%s:%g' % (k, v) for k, v in S['lcharge'].items())
    auto_lc = ','.join('%s:%+g' % (name, row['val'].value) for name, row in (charge_rows or {}).items()
                       if row.get('auto'))
    sc = ' ; '.join(scan_literals())
    fp = '; '.join('%s↔%s' % (_aspec(p['a']), _aspec(p['b'])) for p in S['freeze_pairs'])
    fa = ','.join(str(i) for i in S['freeze_atoms'])
    rows = []
    sub = _wv('dd_subcmd', S.get('subcmd', 'all'))
    if cen or ids:
        center_label = 'center -c' if sub in ('all', 'extract') else 'preparation center'
        rows.append('%s: <code>%s</code>%s%s' %
                    (center_label, html.escape(cen or '—'),
                     (' · exact <code>%s</code>' % html.escape(ids)) if ids else '',
                     ' · use Prepare below' if sub not in ('all', 'extract') else ''))
    if lc: rows.append('charges -l: <code>%s</code>' % html.escape(lc))
    if auto_lc: rows.append('built-in ion charge: <code>%s</code>' % html.escape(auto_lc))
    if sc: rows.append('scan -s: <code>%s</code>' % html.escape(sc))
    if fp or fa: rows.append('freeze: <code>%s%s</code>' %
                             (html.escape(fp), (' · atoms ' + html.escape(fa)) if fa else ''))
    summary_html.value = (
        '<div style="border:1px solid #cdd;border-radius:8px;padding:5px 7px;background:#f7fbff;">'
        '<b>Selection summary</b><br><small>%s</small></div>' %
        ('<br>'.join(rows) if rows else 'No selections yet.'))

chips_box = W.HBox()
chips_box.add_class('rxchip')
def _render_chips():
    def mk(kind, key):
        def _rm(_):
            if kind == 'center' and center_widget is not None:
                center_widget.value = tuple(x for x in center_widget.value if x != key)
                return
            elif kind == 'id':
                S['center_ids'] = [x for x in S['center_ids'] if x != key]
                _render_center_ids()
            elif kind == 'atom':
                S['freeze_atoms'] = [x for x in S['freeze_atoms'] if x != key]
                _render_freeze_panel()
            _render_chips(); render_viewer(); refresh()
        return _rm
    btns = []
    cur = list(center_widget.value) if center_widget is not None else S.get('center', [])
    editable = _view_is_editable()
    for rn in cur:
        b = W.Button(description='%s ✕' % rn, button_style='info', disabled=not editable,
                     layout=W.Layout(width='auto'))
        b.on_click(mk('center', rn)); btns.append(b)
    for rid in S.get('center_ids', []):
        b = W.Button(description='%s ✕' % rid, button_style='warning', disabled=not editable,
                     layout=W.Layout(width='auto'))
        b.on_click(mk('id', rid)); btns.append(b)
    for fa in S.get('freeze_atoms', []):
        b = W.Button(description='⚓%d ✕' % fa, disabled=not editable, layout=W.Layout(width='auto'))
        b.on_click(mk('atom', fa)); btns.append(b)
    chips_box.children = btns
    chips_box.layout.display = '' if btns else 'none'

freeze_acc = _collapsible('Freezing', freeze_panel)
viewer_out.layout = W.Layout(width='100%')
pick_hint = W.HTML()
last_pick_html = W.HTML()
last_pick_info = _info_control('')
last_pick_html.layout = W.Layout(flex='1 1 auto', min_width='0')
last_pick_row = W.HBox([last_pick_html],
                       layout=W.Layout(width='100%', flex_flow='row wrap', align_items='center'))
_PICK_HINT = {
    'center': 'Click a residue to add an exact -c center.',
    'ligand': 'Click a ligand/cofactor to inspect or enable its charge.',
    'scanA': 'Click the first atom of the scan coordinate.', 'scanB': 'Click the second atom of the scan coordinate.',
    'freezeA': 'Click endpoint A of an opt distance restraint.', 'freezeB': 'Click endpoint B of an opt distance restraint.',
    'freezeatom': 'Click atoms to freeze by 1-based Cartesian index.',
}
def _active_pick_hint():
    return _PICK_HINT.get(pick_action.value, 'Choose a click action.')
def _pick_info_text():
    return (_active_pick_hint() +
            ' Mol* keeps its standard focus, selection, and empty-canvas behavior.')
def _render_pick_hint(_=None):
    pick_hint.value = ('<div style="background:#eff6ff;border:1px solid #bfdbfe;border-radius:9px;'
                       'padding:6px 9px;"><b>Active click:</b> %s</div>' % _active_pick_hint())
    _set_info_text(last_pick_info, _pick_info_text())
    status_renderer = globals().get('_render_last_pick_status')
    if status_renderer is not None: status_renderer()
pick_action.observe(_render_pick_hint, names='value'); _render_pick_hint()
def _render_last_pick_status():
    pick = S.get('_last_pick')
    _set_info_text(last_pick_info, _pick_info_text())
    if not pick:
        last_pick_html.value = ''
        return
    tone = '#92400e' if S.get('_last_pick_tone') == 'warn' else '#166534'
    last_pick_html.value = (
        '<div role="status" aria-live="polite" style="border-left:4px solid #f59e0b;'
        'background:#fffbeb;padding:5px 8px;color:%s"><small><b>Last click:</b> '
        '<code>%s</code> — %s</small></div>' %
        (tone, _pick_text(), S.get('_last_pick_message') or 'selected'))
_render_last_pick_status()
viewer_more = _collapsible('Exact atom input', W.VBox([
    W.HBox([exact_atom, exact_atom_btn], layout=W.Layout(flex_flow='row wrap')),
    exact_atom_msg]))
viewer_col = W.VBox([
    W.HBox([view_input, pick_action, last_pick_info]), view_input_note, view_controls,
    viewer_more, viewer_status, viewer_out, last_pick_row])
viewer_col.add_class('rxviewer')
# Compute commands operate on the current model. Prepare a cluster here when a
# full protein was loaded; `all` can instead perform the same extraction itself.
extract_msg = W.HTML()
b_extract = W.Button(description='Extract cluster model', icon='scissors',
                     button_style='info', layout=W.Layout(width='220px'),
                     tooltip='Run `extract` with the picked center (-c) and radius (-r), then use its '
                             'output as the input for opt / tsopt / freq / irc / path-opt.')
prep_radius = W.FloatText(value=2.6, description='radius Å',
                          style={'description_width': '58px'},
                          layout=W.Layout(width='115px'),
                          tooltip='Extraction radius in Å; the CLI default is 2.6.')
def _do_extract(_):
    try:
        if not S['inputs']: raise ValueError('Load a structure in the Input tab first.')
        cen = _center_cli_selectors()
        if not cen: raise ValueError('Pick at least one center residue in the 3D view (-c is required).')
        r = float(prep_radius.value or 0.0)
        model_dir = _runtime_path('prepared_models')
        os.makedirs(model_dir, exist_ok=True)
        outs = [_unique_path(os.path.join(model_dir, '%02d_%s_cluster.pdb' % (i + 1, Path(path).stem)))
                for i, path in enumerate(S['inputs'])]
        cmd = [CLI, 'extract', '-i', *S['inputs'], '-o', *outs, '-c', ','.join(cen)]
        if r and r > 0: cmd += ['-r', str(r)]
        lc = ','.join('%s:%g' % (k, v) for k, v in S['lcharge'].items())
        if lc: cmd += ['-l', lc]
        extract_msg.value = '<small>running: <code>%s</code></small>' % ' '.join(cmd)
        p = subprocess.run(cmd, capture_output=True, text=True)
        if p.returncode != 0 or not all(os.path.exists(path) for path in outs):
            raise RuntimeError((p.stdout + p.stderr)[-400:] or 'extract failed')
        previous = {'inputs': list(S['inputs']), 'mode': S['mode'], 'parm': S.get('parm')}
        load_pdb(outs, None, keep_subcmd=True)
        # load_pdb deliberately invalidates an older preparation transaction;
        # install this new transaction only after the new inputs finish loading.
        S['_pre_extract'] = previous
        b_revert.layout.display = ''
        input_msg.value = '✅ <b>%s</b> (extracted cluster model%s)' % (
            ', '.join(outs), 's' if len(outs) != 1 else '')
        extract_msg.value = ('<small style="color:#181">✅ %d model(s) prepared outside the run output; '
                             'they are now the ordered inputs.</small>' % len(outs))
    except Exception as e:
        extract_msg.value = '<small style="color:#a00">extract failed: %s</small>' % e
b_extract.on_click(_do_extract)
b_revert = W.Button(description='Revert', icon='undo', layout=W.Layout(width='110px'),
                    tooltip='Restore the structure that was loaded before the last extraction.')
b_revert.layout.display = 'none'
def _do_revert(_):
    prev = S.get('_pre_extract')
    if not prev: return
    load_pdb(list(prev['inputs']), prev.get('parm'), mode=prev['mode'], keep_subcmd=True)
    input_msg.value = '↩ reverted to <b>%s</b>' % ', '.join(prev['inputs'])
    extract_msg.value = '<small>reverted to the pre-extraction structure</small>'
b_revert.on_click(_do_revert)
extract_panel = W.VBox([
    _hdr('<b>Cluster model</b> <small>— prepare a full protein for this compute</small>',
         'all can do this internally. For a standalone compute, pick the center residues above, '
         'set the extraction radius here, then press Extract. '
         'Revert puts the original structure back.'),
    W.HBox([prep_radius, b_extract, b_revert], layout=W.Layout(flex_flow='row wrap')),
    extract_msg])
extract_panel.add_class('rxcard')
selection_inspector = W.VBox([
    summary_html, chips_box,
    center_panel, charge_panel, scan_panel, extract_panel])
selection_inspector.add_class('rxinspector')
workspace = W.HBox([viewer_col, selection_inspector]); workspace.add_class('rxworkspace')
selection_help = W.HTML(_sel_help)
selection_help.layout.display = 'none'
selection_route = W.HTML()
selection_route.layout.display = 'none'
def _sync_select_availability():
    has_files = bool(S.get('inputs'))
    is_utility = S.get('mode') == 'utility'
    viewable = has_files and not is_utility
    workspace.layout.display = '' if viewable else 'none'
    selection_inspector.layout.display = '' if viewable else 'none'
    selection_route.layout.display = 'none' if viewable else ''
    selection_route.value = (
        '<div class="rxcard"><b>Utility input</b><br><small>No 3D selection is needed.</small></div>'
        if is_utility else
        '<div class="rxcard"><b>No structure loaded</b></div>')
selection_box = W.VBox([selection_route, workspace, freeze_acc])

# ============================================================== OPTIONS
cb_advsub = W.Checkbox(value=False, description='Show utility workflows', indent=False)
dd_subcmd = W.Dropdown(options=_sub_options(BASIC_SUBS), value='all', description='workflow',
                       style={'description_width': 'initial'},
                       layout=W.Layout(width='320px', max_width='100%'))
subreq = W.HTML(); subcmd_note = W.HTML()
def _workflow_contract(sub):
    """Resolve the visible input/output contract for the active mode and stages."""
    spec = SPEC.get(sub, {})
    req, outputs = spec.get('req', '(complete the command line)'), list(spec.get('out', ()))
    if sub == 'all':
        mode = _wv('all_mode', 'mep')
        if mode == 'mep':
            req = '2 or more structures in reaction order'
            outputs = ['summary.log', 'mep.pdb', 'energy_diagram_MEP.png', 'segments/seg_NN/']
        elif mode == 'scan':
            req = '1 structure + a scan bond picked in Viewer'
            outputs = ['summary.log', 'scan/', 'segments/seg_NN/']
        else:
            req = '1 TS-candidate structure'
            outputs = ['summary.log', 'segments/seg_01/{reactant,ts,product}', 'ts/', 'irc/']
        if _wv('w_ts', False) and mode != 'tsonly': outputs += ['ts/', 'irc/']
        if _wv('w_th', False): outputs += ['thermoanalysis.yaml', 'energy_diagram_Gibbs.png']
        if _wv('adv_dft', False): outputs += ['dft/', 'energy_diagram_DFT.png']
    return req, tuple(dict.fromkeys(outputs))

def _render_workflow_contract():
    sub = _wv('dd_subcmd', S.get('subcmd', 'all')) or 'all'
    req, outputs = _workflow_contract(sub)
    subreq.value = '<small><b>needs:</b> %s</small>' % html.escape(req)
    target = globals().get('outputs_html')
    if target is not None:
        target.value = ('<small><b>produces:</b> %s</small>' %
                        ' · '.join('<code>%s</code>' % html.escape(o) for o in outputs))

def _on_sub(_):
    sub = dd_subcmd.value; S['subcmd'] = sub
    _render_workflow_contract()
    subcmd_note.value = ('' if (sub in COMPUTE or sub == 'extract') else
                         '<small style="color:#166534">utility template filled from the current input; review it below</small>'
                         if sub in AUTOFILL_UTILS else
                         '<small style="color:#7c5c00">utility subcommand — finish it in the command line below</small>')
    _rsp = globals().get('_render_scan_panel')
    if _rsp is not None: _rsp()                          # scan vs scan2d/3d builder
    _scc = globals().get('_sync_capability_controls')
    if _scc is not None: _scc()
    refresh()
dd_subcmd.observe(_on_sub, names='value')
def _on_advsub(_):
    keep = dd_subcmd.value
    names = SUBS if cb_advsub.value else BASIC_SUBS
    dd_subcmd.options = _sub_options(names)
    dd_subcmd.value = keep if keep in names else 'all'
cb_advsub.observe(_on_advsub, names='value')
_on_sub(None)   # initialise the requirement hint for the default subcommand
all_mode = W.ToggleButtons(
    options=[('MEP ≥2', 'mep'), ('Scan 1', 'scan'), ('TS-only', 'tsonly')],
    value='mep', style={'button_width': '74px', 'description_width': '0px'},
    layout=W.Layout(width='240px'))
_ALL_MODE_STATE = {'last': 'mep', 'tsopt_before_tsonly': False, 'user': False, 'sync': False}
_SESSION_APPLY = {'active': False}
def _on_mode(change):
    if _SESSION_APPLY['active']:
        _ALL_MODE_STATE['last'] = all_mode.value
        return
    old = change.get('old', _ALL_MODE_STATE['last']) if isinstance(change, dict) else _ALL_MODE_STATE['last']
    new = all_mode.value
    if not _ALL_MODE_STATE.get('sync') and old != new: _ALL_MODE_STATE['user'] = True
    if new == 'tsonly' and old != 'tsonly':
        _ALL_MODE_STATE['tsopt_before_tsonly'] = bool(w_ts.value)
        w_ts.value = True
    elif old == 'tsonly' and new != 'tsonly':
        w_ts.value = _ALL_MODE_STATE['tsopt_before_tsonly']
    _ALL_MODE_STATE['last'] = new
    sync = globals().get('_sync_capability_controls')
    if sync is not None: sync()
    refresh()
all_mode.observe(_on_mode, names='value')

_bk0 = BACKEND if BACKEND in MODELS else 'mace'
dd_backend = W.Dropdown(options=[('%s · installed' % _bk0, _bk0)], value=_bk0,
                        description='backend', disabled=True,
                        style={'description_width': 'initial'},
                        layout=W.Layout(width='190px'))
dd_model = W.Dropdown(options=MODELS[_bk0], value=S.get('model', DEFAULT_MODEL[_bk0]),
                      description='model', style={'description_width': 'initial'},
                      layout=W.Layout(width='230px'))
def _bk(_):
    changed = S.get('backend') != dd_backend.value
    S['backend'] = dd_backend.value
    dd_model.options = MODELS[dd_backend.value]
    if _SESSION_APPLY['active']: return
    dd_model.value = DEFAULT_MODEL[dd_backend.value]
    S['model'] = dd_model.value
    if changed: _invalidate_last_run('Backend changed; validate and run again.')
    refresh()
def _mdl(_):
    changed = S.get('model') != dd_model.value
    S['model'] = dd_model.value
    if _SESSION_APPLY['active']: return
    if changed: _invalidate_last_run('Backend model changed; validate and run again.')
    refresh()
dd_backend.observe(_bk, names='value'); dd_model.observe(_mdl, names='value')
w_ts = W.Checkbox(value=False, description='--tsopt (TS + IRC)', indent=False)
w_th = W.Checkbox(value=False, description='--thermo (freq + ΔG)', indent=False)
w_out = W.Text(value='result', description='out dir', layout=W.Layout(width='260px'))
w_reuse = W.Checkbox(value=False, description='reuse non-empty out dir', indent=False,
                     tooltip='Enable only to resume or intentionally add to an existing output directory.')
output_note = W.HTML()
w_q = W.IntText(value=0, description='system charge (-q)', style={'description_width': 'initial'},
                layout=W.Layout(width='220px'))
w_charge_ok = W.Checkbox(value=False, description='charge verified', indent=False,
                         tooltip='Confirm that -q is the net charge of the complete input system.')
def _render_output_note():
    out = _effective_out_dir() if '_effective_out_dir' in globals() else (w_out.value or 'result')
    occupied = os.path.isdir(out) and bool(os.listdir(out))
    output_note.value = ('<small style="color:#a60">⚠ non-empty output exists; enable reuse to run into it</small>'
                         if occupied and not w_reuse.value else '')
def _sync_run(_=None):
    S['tsopt'] = w_ts.value; S['thermo'] = w_th.value
    S['out_dir'] = w_out.value or 'result'; S['charge'] = w_q.value
    if _SESSION_APPLY['active']: return
    sync = globals().get('_sync_capability_controls')
    if sync is not None: sync()
    refresh()
def _sync_charge(_=None):
    S['charge'] = w_q.value
    if _SESSION_APPLY['active']: return
    S['charge_explicit'] = False
    if w_charge_ok.value: w_charge_ok.value = False
    refresh()
def _sync_charge_ok(change):
    S['charge_explicit'] = bool(change['new'])
    if not _SESSION_APPLY['active']: refresh()
for w in (w_ts, w_th, w_out, w_reuse): w.observe(_sync_run, names='value')
w_q.observe(_sync_charge, names='value')
w_charge_ok.observe(_sync_charge_ok, names='value')

# Advanced flags are derived from this repository's live Click command.
adv_mult = W.IntText(value=1, description='-m mult', style={'description_width': 'initial'}, layout=W.Layout(width='160px'))
adv_prec = W.Dropdown(options=['auto', 'fp32', 'fp64'], value='auto', description='--precision', style={'description_width': 'initial'}, layout=W.Layout(width='190px'))
adv_det = W.Checkbox(value=False, description='--deterministic', indent=False)
adv_mep = W.Dropdown(options=['(default)', 'gsm'] + (['dmf'] if DMF_READY else []), value='(default)', description='--mep-mode', style={'description_width': 'initial'}, layout=W.Layout(width='200px'))
adv_dmf = W.Dropdown(options=['(default)', 'gpu', 'cpu'], value='(default)', description='--dmf-backend', style={'description_width': 'initial'}, layout=W.Layout(width='220px'))
adv_thresh = W.Dropdown(options=['(default)', 'gau_loose', 'gau', 'gau_tight', 'gau_vtight', 'baker'], value='(default)', description='--thresh', style={'description_width': 'initial'}, layout=W.Layout(width='230px'))
adv_radius = W.FloatText(value=2.6, description='-r radius Å', style={'description_width': 'initial'}, layout=W.Layout(width='190px'))
def _sync_radius_widgets(change):
    target = adv_radius if change['owner'] is prep_radius else prep_radius
    if target.value != change['new']: target.value = change['new']
prep_radius.observe(_sync_radius_widgets, names='value')
adv_radius.observe(_sync_radius_widgets, names='value')
adv_dft = W.Checkbox(value=False, description='--dft (DFT single-point)', indent=False)
adv_dftfb = W.Text(value='', description='--dft-func-basis', placeholder='wb97x-d3,def2-tzvp', style={'description_width': 'initial'}, layout=W.Layout(width='330px'))
adv_flatten = W.Checkbox(value=False, description='--flatten', indent=False,
                         tooltip='Opt-in Cartesian flattening of near-zero modes; helps a stubborn TS converge.')
adv_refine = W.Checkbox(value=False, description='--refine-path', indent=False,
                        tooltip='all only: switch MEP from single-segment path-opt to recursive multi-segment path-search.')
adv_maxcyc = W.IntText(value=0, description='--max-cycles (0=default)', style={'description_width': 'initial'},
                       layout=W.Layout(width='250px'))

adv_search = W.Text(value='', placeholder='filter flags…', description='search',
                    style={'description_width': 'initial'}, layout=W.Layout(width='360px', max_width='100%'))
adv_count = W.HTML()
adv_rows_box = W.VBox(layout=W.Layout(max_height='420px', overflow='auto', width='100%',
                                      border='1px solid #e2e8f0', padding='4px'))

def _option_help(name, sub='all', fallback='See the command help for this option.'):
    for param in _advanced_options(sub):
        if getattr(param, 'name', None) == name and getattr(param, 'help', None): return param.help
    return fallback

def _set_advanced_override(sub, name, value):
    by_sub = S.setdefault('advanced_overrides', {}).setdefault(sub, {})
    if value in (None, ''): by_sub.pop(name, None)
    else: by_sub[name] = value
    refresh()

def _advanced_widget(sub, param):
    flag = _advanced_flag(param); saved = S.setdefault('advanced_overrides', {}).setdefault(sub, {}).get(param.name)
    is_bool = param.is_bool_flag or isinstance(param.type, click.types.BoolParamType)
    if is_bool:
        widget = W.Dropdown(options=[('CLI default', None), ('on', True), ('off', False)], value=saved,
                            description=flag, style={'description_width': 'initial'},
                            layout=W.Layout(width='360px', max_width='92%'))
    elif isinstance(param.type, click.Choice):
        choices = list(param.type.choices)
        widget = W.Dropdown(options=[('CLI default', None)] + [(str(choice), choice) for choice in choices],
                            value=saved, description=flag, style={'description_width': 'initial'},
                            layout=W.Layout(width='420px', max_width='92%'))
    else:
        default = param.default
        placeholder = 'CLI default%s' % ((': %s' % default) if default not in (None, '', ()) else '')
        if param.multiple: placeholder += ' · quote each repeated value'
        widget = W.Text(value='' if saved is None else str(saved), description=flag, placeholder=placeholder,
                        style={'description_width': 'initial'}, layout=W.Layout(width='520px', max_width='92%'))
    widget.observe(lambda change, s=sub, n=param.name: _set_advanced_override(s, n, change['new']), names='value')
    row = _flag_row(widget, param.help or 'No additional help is supplied by this command.')
    row._rx_search = ('%s %s %s' % (flag, param.name, param.help or '')).lower()
    return row

def _render_advanced_rows(_=None):
    _close_info()
    sub = _wv('dd_subcmd', S.get('subcmd', 'all')) or 'all'
    query = adv_search.value.strip().lower()
    rows = [_advanced_widget(sub, param) for param in _advanced_options(sub)
            if _advanced_status(sub, param) == 'rendered' and _advanced_semantic_applicable(sub, param.name)]
    shown = [row for row in rows if not query or query in row._rx_search]
    adv_rows_box.children = shown or [W.HTML('<small>No matching advanced flags.</small>')]
    adv_count.value = ('<small><b>%d option%s</b>%s</small>' %
                       (len(shown), '' if len(shown) == 1 else 's',
                        (' · %d total' % len(rows)) if query else ''))

adv_search.observe(_render_advanced_rows, names='value')

def _sync_capability_controls(_=None):
    sub = dd_subcmd.value
    adv_mep.disabled = sub not in TOOL_CAPABILITIES['mep_mode']
    adv_thresh.disabled = sub not in TOOL_CAPABILITIES['threshold']
    _set_flag_visible(adv_dft, sub == 'all' and DFT_READY)
    radius_applies = (sub == 'extract' or
                      (sub == 'all' and S.get('mode') in ('pdb', 'mmcif')))
    adv_radius.disabled = not radius_applies
    _set_flag_visible(adv_radius, radius_applies)
    # Every surfaced flag is shown only where the CLI accepts it (FLAG_SUBS).
    for _wn, _subs in FLAG_SUBS.items():
        _w = globals().get(_wn)
        if _w is not None and _wn not in ('adv_dft', 'adv_radius'):
            _set_flag_visible(_w, sub in _subs)
    _set_flag_visible(adv_dmf, sub in FLAG_SUBS['adv_dmf'] and adv_mep.value == 'dmf')
    mode = _wv('all_mode', 'mep') if sub == 'all' else None
    if sub == 'all':
        path_active = mode != 'tsonly'
        for _w in (adv_mep, adv_thresh, adv_refine, adv_maxcyc):
            _set_flag_visible(_w, path_active)
        _set_flag_visible(adv_dmf, path_active and adv_mep.value == 'dmf')
        if mode == 'tsonly' and not w_ts.value: w_ts.value = True
        w_ts.disabled = mode == 'tsonly'
        _set_flag_visible(adv_flatten, mode == 'tsonly' or w_ts.value)
    else:
        w_ts.disabled = False
    # "all workflow" mode and the depth switches only exist on `all`; hide them for
    # every other subcommand instead of showing controls the command cannot use.
    for _name in ('all_mode_box', 'depth_box'):
        _b = globals().get(_name)
        if _b is not None: _b.layout.display = '' if sub == 'all' else 'none'
    # Select-tab panels follow the same table.
    _panels = SPEC.get(sub, {}).get('panels', ())
    if sub == 'all' and _wv('all_mode', 'mep') != 'scan':
        _panels = tuple(panel for panel in _panels if panel != 'scan')
    _atom_selectable = bool(S.get('inputs')) and S.get('mode') in ('pdb', 'mmcif', 'small')
    _residue_selectable = bool(S.get('inputs')) and S.get('mode') in ('pdb', 'mmcif')
    _prep_active = sub in _PREP_SUBS and _residue_selectable
    _select_panels = set(_panels) if _atom_selectable else set()
    if not _residue_selectable: _select_panels.discard('center')
    if _prep_active: _select_panels.add('center')
    for _name, _key in (('scan_panel', 'scan'), ('freeze_acc', 'freeze'),
                        ('center_panel', 'center'), ('charge_panel', 'center')):
        _b = globals().get(_name)
        if _b is not None: _b.layout.display = '' if _key in _select_panels else 'none'
    _extract_panel = globals().get('extract_panel')
    if _extract_panel is not None: _extract_panel.layout.display = '' if _prep_active else 'none'
    _sync_select_availability()
    _current_pick = pick_action.value
    _pick_options = [(label, value) for label, value, panel in _PICK_ACTIONS
                     if panel is None or panel in _select_panels]
    _valid_picks = {value for _label, value in _pick_options}
    _PICK_ACTION_STATE['sync'] = True
    try:
        pick_action.options = _pick_options
        if _PICK_ACTION_STATE['user'] and _current_pick in _valid_picks:
            pick_action.value = _current_pick
        elif 'scan' in _select_panels and 'scanA' in _valid_picks:
            pick_action.value = 'scanA'
        elif 'center' in _select_panels and 'center' in _valid_picks:
            pick_action.value = 'center'
        elif _pick_options:
            pick_action.value = _pick_options[0][1]
    finally:
        _PICK_ACTION_STATE['sync'] = False
    _kb = globals().get('key_opts_box')
    _bb = globals().get('backend_box')
    if _bb is not None: _bb.layout.display = '' if sub in MLIP_COMPUTE else 'none'
    _charge_row = globals().get('charge_run_row')
    if _charge_row is not None: _charge_row.layout.display = '' if sub in COMPUTE else 'none'
    _output_row = globals().get('output_run_row')
    _file_output = sub in COMPUTE or sub == 'extract' or sub in AUTOFILL_UTILS - {'bond-summary'}
    if _output_row is not None: _output_row.layout.display = '' if _file_output else 'none'
    w_out.layout.display = '' if sub in COMPUTE or sub == 'extract' else 'none'
    _hint = globals().get('utility_output_hint')
    if _hint is not None:
        _hint.layout.display = '' if _file_output and sub not in COMPUTE and sub != 'extract' else 'none'
    _run_box = globals().get('run_settings_box')
    if _run_box is not None: _run_box.layout.display = '' if (sub in COMPUTE or _file_output) else 'none'
    adv_dftfb.description = '--func-basis' if sub == 'dft' else '--dft-func-basis'
    _dftfb_applicable = DFT_READY and (sub == 'dft' or (sub == 'all' and adv_dft.value))
    adv_dftfb.disabled = not _dftfb_applicable
    _set_flag_visible(adv_dftfb, _dftfb_applicable)
    if _kb is not None:
        _key_widgets = (adv_mep, adv_dmf, adv_thresh, adv_maxcyc, adv_prec, adv_det,
                        adv_flatten, adv_refine, adv_mult, adv_radius, adv_dftfb)
        _kb.layout.display = '' if any(getattr(w, '_rx_flag_row').layout.display != 'none'
                                       for w in _key_widgets) else 'none'
    _render_workflow_contract()
    queue_renderer = globals().get('_render_input_queue')
    if queue_renderer is not None: queue_renderer()
    label = globals().get('depth_label')
    if label is not None:
        label.value = ('<b>Stages</b> · TS optimization required'
                       if sub == 'all' and mode == 'tsonly' else
                       '<b>Optional stages</b>')
    render_advanced = globals().get('_render_advanced_rows')
    if render_advanced is not None: render_advanced()
    _set_selection_editor_enabled(_view_is_editable())

for w in (adv_mult, adv_prec, adv_det, adv_mep, adv_dmf, adv_thresh, adv_radius, adv_dft, adv_dftfb,
          adv_flatten, adv_refine, adv_maxcyc):
    w.observe(refresh, names='value')
adv_dft.observe(_sync_capability_controls, names='value')
adv_mep.observe(_sync_capability_controls, names='value')
_sync_capability_controls()
adv_box = W.VBox([W.HBox([adv_search, adv_count]), adv_rows_box])
adv_acc = _collapsible('Advanced flags', adv_box)
all_mode_box = all_mode
depth_label = W.HTML('<b>Optional stages</b>')
depth_box = W.VBox([depth_label, W.HBox([
    _flag_row(w_ts, _option_help('tsopt', fallback='Run TS optimization and IRC after path generation.')),
    _flag_row(w_th, _option_help('thermo', fallback='Run frequency thermochemistry for the refined structure.')),
    _flag_row(adv_dft, _option_help('dft', fallback='Add a DFT single-point correction.'))])])
outputs_html = W.HTML()
subreq.layout = W.Layout(flex='1 1 220px', min_width='0')
outputs_html.layout = W.Layout(flex='3 1 560px', min_width='0')
key_opts_content = W.VBox([
    W.HBox([_flag_row(adv_mep, _option_help('mep_mode')), _flag_row(adv_dmf, _option_help('dmf_backend')),
            _flag_row(adv_thresh, _option_help('thresh')), _flag_row(adv_maxcyc, _option_help('max_cycles'))]),
    W.HBox([_flag_row(adv_prec, _option_help('precision')), _flag_row(adv_det, _option_help('deterministic')),
            _flag_row(adv_flatten, _option_help('flatten')), _flag_row(adv_refine, _option_help('refine_path'))]),
    W.HBox([_flag_row(adv_mult, _option_help('spin')), _flag_row(adv_radius, _option_help('radius')),
            _flag_row(adv_dftfb, _option_help('dft_func_basis', fallback='DFT functional,basis override.'))])])
key_opts_box = _collapsible('Key options', key_opts_content)
_missing_features = []
if not DFT_READY: _missing_features.append('DFT is hidden (rerun Setup with install_dft enabled)')
if not DMF_READY: _missing_features.append('DMF is hidden (pydmf + cyipopt are not installed)')
dependency_note = W.HTML('<small style="color:#92400e">%s</small>' % ' · '.join(_missing_features)
                         if _missing_features else '')
backend_box = W.VBox([
    _hdr('<b>MLIP backend &amp; model</b>',
         'Setup installed this backend. Restart the runtime and rerun Setup to switch; auto precision uses its default.'),
    W.HBox([dd_backend, dd_model]), dependency_note])
backend_box.add_class('rxcard')
workflow_controls = W.HBox([dd_subcmd, cb_advsub, all_mode_box],
                            layout=W.Layout(flex_flow='row wrap', align_items='center'))
workflow_contract_row = W.HBox([subreq, outputs_html],
                                layout=W.Layout(flex_flow='row wrap'))
workflow_controls.add_class('rxworkflow-controls')
workflow_contract_row.add_class('rxworkflow-contract')
workflow_box = W.VBox([
    _hdr('<b>Workflow</b>',
         'all runs the end-to-end pipeline; the other entries run one stage on prepared inputs.'),
    workflow_controls, workflow_contract_row, subcmd_note])
workflow_box.add_class('rxcard')
viewer_box = W.VBox([workflow_box, selection_box])
charge_info = _info_control('The system charge is passed as -q when it cannot be derived from selected ligand charges or a GJF input.')
charge_run_row = W.HBox([w_q, w_charge_ok, charge_info], layout=W.Layout(flex_flow='row wrap'))
utility_output_hint = W.HTML('<small>The utility output path is shown in the generated command.</small>')
output_run_row = W.HBox([w_out, w_reuse, utility_output_hint], layout=W.Layout(flex_flow='row wrap'))
run_settings_box = W.VBox([
    _hdr('<b>Run &amp; output</b>',
         'Confirm the complete-system charge, choose a fresh output directory, and enable reuse only intentionally.'),
    charge_run_row, output_run_row, output_note])
run_settings_box.add_class('rxcard')
options_box = W.VBox([backend_box, depth_box, key_opts_box, run_settings_box, adv_acc])
_sync_capability_controls()   # apply all-only visibility now that the boxes exist

# ============================================================== RESULTS tab
res_out = W.Output()
traj_out = W.Output(layout={'width': '100%', 'min_width': '0', 'flex': '1 1 440px'})
plot_out = W.Output(layout={'width': '100%', 'min_width': '0', 'flex': '1 1 440px'})
result_context, traj_label, frame_state, trajectory_intro = W.HTML(), W.HTML(), W.HTML(), W.HTML()
artifact_out = W.Output()
artifact_choice = W.Dropdown(options=[], description='File',
                             style={'description_width': 'initial'},
                             layout=W.Layout(width='620px', max_width='100%'))
traj_choice = W.Dropdown(options=[], description='Path',
                         style={'description_width': 'initial'},
                         layout=W.Layout(width='620px', max_width='100%'))
_result_pick_guard = {'active': False}
frame_prev = W.Button(description='Previous', disabled=True,
                      tooltip='Show the preceding trajectory frame.',
                      layout=W.Layout(width='92px'))
frame_play = W.Play(value=0, min=0, max=0, step=1, interval=900,
                    description='Play', disabled=True,
                    layout=W.Layout(width='112px'))
frame_next = W.Button(description='Next', disabled=True,
                      tooltip='Show the next trajectory frame.',
                      layout=W.Layout(width='76px'))
frame_slider = W.IntSlider(min=0, max=0, value=0, description='Path position',
                           continuous_update=False, readout=False, disabled=True,
                           style={'description_width': 'initial'},
                           layout=W.Layout(width='100%', min_width='190px'))
_frame_link = W.jslink((frame_play, 'value'), (frame_slider, 'value'))
_TRAJ = {'frames': [], 'energies': [], 'path': None, 'semantics': {}}

def _step_frame(delta):
    if frame_slider.disabled: return
    frame_slider.value = max(frame_slider.min, min(frame_slider.max, frame_slider.value + delta))

frame_prev.on_click(lambda _: _step_frame(-1))
frame_next.on_click(lambda _: _step_frame(1))

def _parse_trj(path):
    frames, energies = [], []
    lines = open(path).read().splitlines()
    i, n_lines = 0, len(lines)
    while i < n_lines:
        if not lines[i].strip():
            i += 1; continue
        try: n = int(lines[i].strip())
        except ValueError:
            i += 1; continue
        comment = lines[i + 1] if i + 1 < n_lines else ''
        frames.append('\n'.join(lines[i:i + 2 + n]))
        try: energies.append(float(comment.strip().split()[0]))
        except (ValueError, IndexError): energies.append(None)
        i += 2 + n
    return frames, energies

def _rel_kcal():
    es = _TRAJ['energies']
    base = es[0] if (es and es[0] is not None) else next((e for e in es if e is not None), None)
    if base is None: return None
    return [((e - base) * 627.509) if e is not None else None for e in es]

def _trajectory_semantics(sub=None, path=''):
    """Describe a trajectory without assigning reaction semantics to generic runs."""
    sub = str(sub or S.get('_last_subcmd') or S.get('subcmd') or '').lower()
    raw = str(path or '').replace('\\', '/').lower()
    name = os.path.basename(raw)
    if 'imag_' in name or 'mode_' in name or '/vib/' in raw:
        return {'title': 'Vibrational-mode animation', 'start': 'negative phase', 'end': 'positive phase',
                'x': 'phase frame', 'extrema': False}
    if 'scan' in name or sub in ('scan', 'scan2d', 'scan3d'):
        return {'title': 'Scan trajectory', 'start': 'scan start', 'end': 'scan end',
                'x': 'scan frame', 'extrema': False}
    if 'irc' in name or sub == 'irc':
        return {'title': 'IRC trajectory', 'start': 'IRC start', 'end': 'IRC end',
                'x': 'IRC frame', 'extrema': False}
    if 'tsopt' in name or sub == 'tsopt':
        return {'title': 'TS-refinement trajectory', 'start': 'initial candidate', 'end': 'refined candidate',
                'x': 'optimization step', 'extrema': False}
    if ('mep' in name or 'path' in name or 'segment' in name or
            sub in ('all', 'path-opt', 'path-search')):
        return {'title': 'Reaction-path trajectory', 'start': 'R', 'end': 'P',
                'x': 'image', 'extrema': True}
    if 'opt' in name or sub == 'opt':
        return {'title': 'Optimization trajectory', 'start': 'initial', 'end': 'optimized',
                'x': 'optimization step', 'extrema': False}
    return {'title': 'Trajectory', 'start': 'initial', 'end': 'final',
            'x': 'frame', 'extrema': False}

def _stationary(ys, semantics=None):
    """Energy-only profile candidates; no frequency or IRC certification is implied."""
    semantics = semantics or _TRAJ.get('semantics') or _trajectory_semantics()
    n = len(ys)
    if not n: return []
    if n < 2: return [(0, semantics['start'])]
    pts = [(0, semantics['start'])]
    if semantics.get('extrema'):
        for k in range(1, n - 1):
            a, b, c = ys[k - 1], ys[k], ys[k + 1]
            if not (a == a and b == b and c == c): continue                   # skip NaN
            if b > a and b >= c and (b - min(a, c)) > 0.3: pts.append((k, 'peak candidate'))
            elif b < a and b <= c and (max(a, c) - b) > 0.3: pts.append((k, 'minimum candidate'))
    pts.append((n - 1, semantics['end']))
    return pts

def _show_frame(i):
    fr = _TRAJ['frames']
    if not fr: return
    i = max(0, min(i, len(fr) - 1))
    with traj_out:
        clear_output()
        display(HTML(_molstar_iframe(fr[i], 'xyz', show_sequence=False)))
    with plot_out:
        clear_output()
        rk = _rel_kcal()
        semantics = _TRAJ.get('semantics') or _trajectory_semantics(path=_TRAJ.get('path'))
        if rk is None:
            frame_state.value = ('<div role="status" aria-live="polite" aria-atomic="true">'
                                 '<b>Frame %d/%d</b> · energy unavailable</div>' % (i + 1, len(fr)))
            print('(no per-frame energies in the trajectory)'); return
        import matplotlib.pyplot as plt
        plt.rcParams.update({'font.size': 10, 'figure.dpi': 120, 'savefig.dpi': 120,
                             'axes.edgecolor': '#cbd5e1', 'axes.linewidth': 0.9,
                             'xtick.color': '#64748b', 'ytick.color': '#64748b',
                             'axes.labelcolor': '#475569', 'text.color': '#33404d'})
        xs = list(range(1, len(rk) + 1)); ys = [y if y is not None else float('nan') for y in rk]
        stat = _stationary(ys, semantics)
        current_label = dict(stat).get(i, '')
        LINE, HALO, EMPH, MUTE = '#4C72B0', '#F4A259', '#E07B39', '#cfd6dd'
        fig, ax = plt.subplots(figsize=(4.8, 3.6))
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        ax.grid(True, axis='y', color='#edf0f4', lw=0.9, zorder=0)
        for (si, lab) in stat:                                               # horizontal state levels
            if not (ys[si] == ys[si]): continue
            on = (si == i)
            ax.axhline(ys[si], color=(EMPH if on else MUTE), lw=(2.4 if on else 1.0),
                       ls=('-' if on else (0, (4, 3))), alpha=(0.95 if on else 0.85), zorder=1)
            ax.annotate(lab, xy=(xs[si], ys[si]), xytext=(0, 9),
                        textcoords='offset points', va='bottom', ha='center', fontsize=9,
                        color=('#b5601f' if on else '#64748b'),
                        fontweight=('bold' if on else 'normal'))
        ax.plot(xs, ys, '-', color=LINE, lw=2.0, zorder=2)
        ax.plot(xs, ys, 'o', color=LINE, ms=3.6, mec='white', mew=0.5, zorder=3)
        ax.axvline(xs[i], color=EMPH, lw=1.1, alpha=0.40, zorder=1)
        if ys[i] == ys[i]:                                                   # halo + solid center (no harsh dot)
            ax.plot([xs[i]], [ys[i]], 'o', ms=22, color=HALO, alpha=0.30, mec='none', zorder=4)
            ax.plot([xs[i]], [ys[i]], 'o', ms=9.5, color=EMPH, mec='white', mew=1.0, zorder=5)
        ax.set_xlabel(semantics['x'], fontsize=10); ax.set_ylabel('ΔE (kcal/mol)', fontsize=10)
        energy_text = ('ΔE = %.1f kcal/mol' % ys[i]) if ys[i] == ys[i] else 'ΔE unavailable'
        label_text = (' · ' + current_label) if current_label else ''
        frame_state.value = ('<div role="status" aria-live="polite" aria-atomic="true">'
                             '<b>Frame %d of %d</b>%s · %s</div>' %
                             (i + 1, len(rk), label_text, energy_text))
        ax.margins(x=0.10, y=0.16)
        fig.tight_layout(); plt.show(); plt.close(fig)

def _on_frame_change(change):
    frame_prev.disabled = frame_slider.disabled or frame_slider.value <= frame_slider.min
    frame_next.disabled = frame_slider.disabled or frame_slider.value >= frame_slider.max
    _show_frame(int(frame_slider.value))

frame_slider.observe(_on_frame_change, names='value')

def _summary_html(summary_path=None):
    if not summary_path or not os.path.exists(summary_path): return ''
    try:
        with open(summary_path) as fh: d = json.load(fh)
    except (OSError, ValueError) as exc:
        return '<div role="alert" style="color:#991b1b">Could not parse <code>%s</code>: %s</div>' % (html.escape(os.path.basename(summary_path)), html.escape(str(exc)))
    def val(x):
        try: return '%.1f' % float(x)
        except (TypeError, ValueError): return '—'
    scientific = str(d.get('scientific_status') or '').lower()
    complete = scientific in ('success', 'complete', 'pass', 'passed')
    partial = bool(scientific) and not complete
    reasons = d.get('scientific_status_reasons') or []
    if not isinstance(reasons, (list, tuple)): reasons = [reasons]
    badge_text = scientific or 'scientific status unavailable'
    badge_color = '#166534' if complete else ('#92400e' if partial else '#475569')
    status_html = ('<span style="display:inline-block;background:%s;color:white;border-radius:999px;'
                   'padding:2px 9px;margin-bottom:6px">%s</span>' %
                   (badge_color, html.escape(badge_text)))
    if reasons:
        status_html += (' <details style="display:inline-block"><summary>status details</summary><ul>%s</ul></details>' %
                        ''.join('<li>%s</li>' % html.escape(str(reason)) for reason in reasons))
    post = {row.get('index'): row for row in (d.get('post_segments') or [])}
    rows = ''
    for k, seg in enumerate(d.get('segments', []) or []):
        idx = seg.get('index', k + 1); ps = post.get(idx, {})
        mlip = ps.get('mlip') if isinstance(ps.get('mlip'), dict) else {}
        gibbs = ps.get('gibbs_mlip') if isinstance(ps.get('gibbs_mlip'), dict) else {}
        refined = mlip.get('barrier_kcal') is not None
        barrier = mlip.get('barrier_kcal') if refined else seg.get('barrier_kcal')
        delta = mlip.get('delta_kcal') if refined else seg.get('delta_kcal')
        method = 'refined MLIP' if refined else 'MEP band'
        imag = (ps.get('ts_imag') or {}).get('n_imag')
        rows += ('<tr><td>seg %02d</td><td>%s</td><td align=right>%s</td>'
                 '<td align=right>%s</td><td align=right>%s</td><td align=right>%s</td></tr>') % (
                     idx, method, val(barrier), val(gibbs.get('barrier_kcal')), val(delta),
                     '—' if imag is None else str(imag))
    rls = d.get('rate_limiting_step') or {}; rl = rls.get('barrier_kcal')
    table = ('<table style="border-collapse:collapse;" border=1 cellpadding=5>'
         '<caption style="text-align:left;font-weight:600">Current-run segment summary (kcal/mol)</caption>'
         '<tr><th scope="col">segment</th><th scope="col">source</th><th scope="col">ΔE‡</th><th scope="col">ΔG‡</th>'
         '<th scope="col">ΔE</th><th scope="col">n<sub>imag</sub></th></tr>%s</table>' % rows)
    h = status_html + '<div style="max-width:100%%;overflow-x:auto">%s</div>' % table
    if rl is not None:
        barrier_label = 'Provisional barrier' if partial else ('Rate-limiting barrier' if complete else 'Reported barrier')
        h += ('<br><b style="font-size:15px;">%s: %s kcal/mol</b> '
              '<small>(%s)</small>' % (barrier_label, val(rl), rls.get('method') or 'method not recorded'))
    h += '<br><small>status: %s · images: %s · backend/model: %s / %s</small>' % (
        d.get('status'), d.get('n_images'), d.get('mlip_backend'), d.get('mlip_model'))
    return h

def _current_run_files(out):
    """Return only files changed by the last GUI-launched run for this root."""
    if not S.get('_last_out_dir') or os.path.abspath(out) != os.path.abspath(S['_last_out_dir']):
        return []
    return sorted(path for path in S.get('_last_files', []) if os.path.isfile(path))

def _select_status_json(current, sub):
    """Prefer the aggregate summary for composite workflows, otherwise a leaf result."""
    order = ('summary.json', 'result.json') if sub in ('all', 'path-search') else ('result.json', 'summary.json')
    for name in order:
        candidates = sorted((p for p in current if os.path.basename(p) == name),
                            key=lambda p: (p.count(os.sep), p))
        if candidates: return candidates[0]
    return None

def _result_context_html(out):
    sub = S.get('_last_subcmd') or S.get('subcmd') or 'all'
    current = _current_run_files(out)
    status_bits = []
    status_json = _select_status_json(current, sub)
    if status_json:
        try:
            with open(status_json) as fh: data = json.load(fh)
            for key in ('execution_status', 'scientific_status', 'status', 'converged',
                        'n_imaginary_modes', 'n_imag'):
                if key in data: status_bits.append('%s=<b>%s</b>' % (key, html.escape(str(data[key]))))
            reasons = data.get('scientific_status_reasons') or []
            if reasons:
                if not isinstance(reasons, (list, tuple)): reasons = [reasons]
                status_bits.append('<details><summary>scientific status details</summary><ul>%s</ul></details>' %
                                   ''.join('<li>%s</li>' % html.escape(str(reason)) for reason in reasons))
        except Exception:
            status_bits.append('%s could not be parsed' % os.path.basename(status_json))
    manifest = S.get('_last_manifest') or {}
    if manifest:
        status_bits.insert(0, 'run=<b>%s</b>%s' % (
            html.escape(str(manifest.get('status') or 'recorded')),
            (' (exit %s)' % html.escape(str(manifest['exit_code']))) if manifest.get('exit_code') is not None else ''))
    rels = [os.path.relpath(path, out) for path in current]
    shown = rels[:16]
    stdout_only = bool((S.get('_last_manifest') or {}).get('stdout_only'))
    artifacts = ('standard output' if stdout_only else
                 (' · '.join('<code>%s</code>' % html.escape(p) for p in shown)
                  or 'no files recorded'))
    if len(rels) > len(shown): artifacts += ' · … +%d files' % (len(rels) - len(shown))
    return ('<div style="border:1px solid #dbeafe;border-radius:11px;padding:7px 9px;background:#f8fbff;">'
            '<b>%s</b> · <small>%s<br>%s</small></div>' %
            (html.escape(sub), ' · '.join(status_bits) or 'status unavailable', artifacts))

_ARTIFACT_KINDS = {'.png': 'image', '.jpg': 'image', '.jpeg': 'image',
                   '.svg': 'SVG', '.html': 'interactive HTML',
                   '.csv': 'CSV table', '.pdf': 'PDF', '.json': 'JSON',
                   '.yaml': 'YAML', '.yml': 'YAML', '.txt': 'text',
                   '.log': 'text', '.out': 'text', '.md': 'text',
                   '.pdb': 'structure', '.ent': 'structure', '.cif': 'structure',
                   '.mmcif': 'structure'}
_TEXT_PREVIEW_LIMIT = 512 * 1024
_STRUCTURE_PREVIEW_LIMIT = 5 * 1024 * 1024

def _artifact_kind(path):
    path = Path(path)
    if path.suffix.lower() == '.xyz':
        return None if 'trj' in path.name.lower() else 'structure'
    return _ARTIFACT_KINDS.get(path.suffix.lower())

def _csv_preview_html(path, max_rows=50, max_cols=20):
    rows = []
    with open(path, newline='', encoding='utf-8', errors='replace') as fh:
        reader = csv.reader(fh)
        for index, row in enumerate(reader):
            if index > max_rows: break
            rows.append(row[:max_cols])
    if not rows: return '<i>empty CSV</i>'
    head, body = rows[0], rows[1:max_rows + 1]
    table = '<table border="1" cellpadding="4" style="border-collapse:collapse;max-width:100%;">'
    table += '<thead><tr>%s</tr></thead>' % ''.join('<th>%s</th>' % html.escape(cell) for cell in head)
    table += '<tbody>%s</tbody></table>' % ''.join(
        '<tr>%s</tr>' % ''.join('<td>%s</td>' % html.escape(cell) for cell in row) for row in body)
    if len(rows) > max_rows: table += '<small>Preview truncated after %d rows.</small>' % max_rows
    return '<div style="max-width:100%%;overflow-x:auto">%s</div>' % table

def _text_preview_html(path, kind):
    size = os.path.getsize(path)
    with open(path, encoding='utf-8', errors='replace') as fh:
        source = fh.read(_TEXT_PREVIEW_LIMIT + 1)
    truncated = len(source) > _TEXT_PREVIEW_LIMIT or size > _TEXT_PREVIEW_LIMIT
    source = source[:_TEXT_PREVIEW_LIMIT]
    if kind == 'JSON':
        try: source = json.dumps(json.loads(source), indent=2, ensure_ascii=False)
        except (ValueError, TypeError): pass
    note = ('<small>Preview truncated at 512 KiB; download the result for the complete file.</small>'
            if truncated else '')
    return ('<pre style="max-height:520px;max-width:100%%;overflow:auto;white-space:pre-wrap;'
            'word-break:break-word;background:#0f172a;color:#e2e8f0;padding:10px;border-radius:8px;">%s</pre>%s'
            % (html.escape(source), note))

def _structure_preview(path):
    """Show one bounded molecular structure without treating trajectories as files."""
    size = os.path.getsize(path)
    if size > _STRUCTURE_PREVIEW_LIMIT:
        print('Structure preview skipped (%.1f MiB); download the result for the complete file.' %
              (size / 1048576.0))
        return
    suffix = Path(path).suffix.lower()
    fmt = ('pdb' if suffix in ('.pdb', '.ent') else
           ('mmcif' if suffix in ('.cif', '.mmcif') else 'xyz'))
    with open(path, encoding='utf-8', errors='replace') as fh: source = fh.read()
    display(HTML(_molstar_iframe(source, fmt, show_sequence=(fmt != 'xyz'))))

def _render_artifact(_=None):
    if _result_pick_guard['active']: return
    path = artifact_choice.value
    out = S.get('_last_out_dir') or _effective_result_root()
    with artifact_out:
        clear_output()
        if not path or not os.path.isfile(path): return
        rel = os.path.relpath(path, out)
        display(W.HTML('<div><b>Displayed artifact:</b> <code>%s</code></div>' % html.escape(rel)))
        kind = _artifact_kind(path)
        if kind == 'image':
            display(Image(filename=path)); return
        try:
            if kind == 'structure':
                _structure_preview(path)
            elif kind in ('SVG', 'interactive HTML'):
                with open(path, encoding='utf-8', errors='replace') as fh: source = fh.read()
                title = '%s: %s' % (kind, os.path.basename(path))
                display(HTML('<iframe sandbox="allow-scripts" title="%s" srcdoc="%s" '
                             'style="width:100%%;height:560px;border:1px solid #e6eaf0;'
                             'border-radius:10px;"></iframe>' %
                             (html.escape(title, quote=True), html.escape(source, quote=True))))
            elif kind == 'CSV table':
                display(HTML(_csv_preview_html(path)))
            elif kind in ('JSON', 'YAML', 'text'):
                display(HTML(_text_preview_html(path, kind)))
            elif kind == 'PDF':
                size = os.path.getsize(path)
                if size > 5 * 1024 * 1024:
                    print('PDF preview skipped (%.1f MiB); use the results/diagnostics download.' % (size / 1048576.0))
                else:
                    with open(path, 'rb') as fh: data = base64.b64encode(fh.read()).decode('ascii')
                    display(HTML('<iframe title="PDF result" src="data:application/pdf;base64,%s" '
                                 'style="width:100%%;height:560px;border:1px solid #e6eaf0;'
                                 'border-radius:10px;"></iframe>' % data))
            else:
                print('No inline preview is available for this file type.')
        except OSError as exc:
            print('could not embed artifact:', exc)

def _load_trajectory(path=None, out=None):
    path = path or traj_choice.value
    out = out or S.get('_last_out_dir') or _effective_result_root()
    _TRAJ.update(frames=[], energies=[], path=path, semantics={})
    if path and os.path.isfile(path):
        try:
            _TRAJ['frames'], _TRAJ['energies'] = _parse_trj(path)
        except OSError as exc:
            traj_label.value = '<small role="alert" style="color:#991b1b">Could not read trajectory: %s</small>' % html.escape(str(exc))
    if _TRAJ['frames']:
        semantics = _trajectory_semantics(S.get('_last_subcmd'), path)
        _TRAJ['semantics'] = semantics
        trajectory_intro.value = ('<div><span class="rxpath-title">%s</span>'
                                  '<span class="rxpath-subtitle">linked structure + energy</span></div>' %
                                  html.escape(semantics['title']))
        traj_label.value = '<small><code>%s</code></small>' % html.escape(os.path.relpath(path, out))
        last = max(0, len(_TRAJ['frames']) - 1)
        frame_play.max = last; frame_play.value = 0
        frame_slider.max = last; frame_slider.value = 0
        frame_slider.disabled = False; frame_play.disabled = last == 0
        frame_prev.disabled = True; frame_next.disabled = last == 0
        results_empty.layout.display = 'none'; trajectory_box.layout.display = ''
        _show_frame(0)
    else:
        frame_play.max = 0; frame_play.value = 0; frame_play.disabled = True
        frame_slider.max = 0; frame_slider.value = 0; frame_slider.disabled = True
        frame_prev.disabled = True; frame_next.disabled = True
        frame_state.value = ''
        trajectory_box.layout.display = 'none'
        with traj_out: clear_output()
        with plot_out: clear_output()

def _on_traj_choice(_=None):
    if not _result_pick_guard['active']: _load_trajectory()
artifact_choice.observe(_render_artifact, names='value')
traj_choice.observe(_on_traj_choice, names='value')

def _results(out=None):
    out = out or S.get('_last_out_dir') or _effective_result_root()
    current = _current_run_files(out)
    manifest = S.get('_last_manifest') or {}
    result_context.value = _result_context_html(out)
    with res_out:
        clear_output()
        if not current and manifest.get('stdout_only'):
            print('This utility writes to standard output; inspect the run log below.')
        elif not current and manifest.get('status') == 'failed':
            print('Command failed (exit %s). Inspect the run log; no current-run artifact was produced.' % manifest.get('exit_code', '?'))
        elif not current and manifest.get('status') == 'cancelled':
            print('Command was cancelled (exit 130). Inspect the run log; no current-run artifact was produced.')
        elif not current and manifest:
            print('The current run finished without a file artifact. Inspect the run log and command status.')
        elif not current:
            print('No current GUI run is recorded for', out)
            print('Run a workflow first; older files in a reused directory are intentionally not shown.')
        sub = S.get('_last_subcmd') or S.get('subcmd') or 'all'
        def _preview_priority(path):
            name = os.path.basename(path).lower(); kind = _artifact_kind(path)
            if sub in ('all', 'path-opt', 'path-search', 'scan', 'scan2d', 'scan3d') and kind == 'image': return 0
            if sub in ('opt', 'tsopt', 'extract', 'fix-altloc', 'add-elem-info') and kind == 'structure': return 0
            if sub == 'freq' and ('frequenc' in name or 'thermo' in name): return 0
            return {'result.json': 2, 'result.yaml': 3, 'result.yml': 3,
                    'thermoanalysis.yaml': 4, 'frequencies_cm-1.txt': 5,
                    'summary.json': 20}.get(name, 10)
        visuals = sorted((p for p in current if _artifact_kind(p)),
                         key=lambda p: (_preview_priority(p), p))
        summaries = sorted((p for p in current if os.path.basename(p) == 'summary.json'),
                           key=lambda p: p.count(os.sep))
        sh = _summary_html(summaries[0] if summaries else None)
        if sh: display(W.HTML(sh))
    prefer_irc = ('--tsopt' in S.get('_last_argv', []) or
                  (S.get('_last_subcmd') or '') in ('tsopt', 'irc'))
    def _trajectory_rank(path):
        name = os.path.basename(path).lower()
        preferred = 'finished_irc_trj' in name if prefer_irc else 'mep_trj' in name
        return (not preferred, 'mep_trj' not in name, 'finished_irc_trj' not in name, path)
    cand = sorted((p for p in current if Path(p).suffix.lower() == '.xyz' and 'trj' in os.path.basename(p).lower()),
                  key=_trajectory_rank)
    _result_pick_guard['active'] = True
    try:
        artifact_choice.options = [('%s · %s' % (_artifact_kind(p), os.path.relpath(p, out)), p)
                                   for p in visuals]
        artifact_choice.disabled = not visuals
        if visuals: artifact_choice.value = visuals[0]
        traj_choice.options = [('%s · %s' % (_trajectory_semantics(S.get('_last_subcmd'), p)['title'],
                                             os.path.relpath(p, out)), p) for p in cand]
        traj_choice.disabled = not cand
        if cand: traj_choice.value = cand[0]
    finally:
        _result_pick_guard['active'] = False
    artifact_fold.layout.display = '' if visuals else 'none'
    artifact_fold._rx_set_open(bool(visuals and not cand))
    _render_artifact()
    _load_trajectory(cand[0] if cand else None, out)
    if not cand:
        results_empty.layout.display = ''
        state = manifest.get('status')
        results_empty.value = ('<div role="status" style="padding:10px;border:1px dashed #64748b;border-radius:10px;">%s</div>' %
                               ('No trajectory was produced; the generated file preview is below.' if current and visuals else
                                'This result has no XYZ trajectory. Inspect its status and files above.' if current else
                                'The command failed before producing a trajectory; inspect the run log.' if state == 'failed' else
                                'The command was cancelled before producing a trajectory.' if state == 'cancelled' else
                                'This run produced no XYZ trajectory.' if manifest else
                                'Run a workflow to inspect its current trajectory here.'))
        traj_label.value = '<small>no XYZ trajectory found for this result</small>'
    res_btn.description = 'Refresh'
    res_btn.disabled = False
    dl_btn.disabled = not bool(current or manifest or S.get('_last_log'))
res_btn = W.Button(description='Show results', icon='bar-chart', disabled=True,
                   layout=W.Layout(width='200px'))
res_btn.on_click(lambda _: _results(S.get('_last_out_dir') or _effective_result_root()))
dl_btn = W.Button(description='Download current run (.zip)', icon='download', disabled=True,
                  tooltip='Current-run files plus the command manifest and captured log; available after failures too.',
                  layout=W.Layout(width='300px', max_width='100%'))
def _download(_):
    out = S.get('_last_out_dir') or _effective_result_root()
    current = _current_run_files(out)
    manifest = S.get('_last_manifest') or {}
    transcript = S.get('_last_log') or ''
    if not (current or manifest or transcript):
        with res_out: clear_output(); print('No current run is available to bundle.'); return
    stem = Path(os.path.abspath(out)).name or 'results'
    z = _unique_path(_runtime_path('downloads', stem + '_current_run.zip'))
    with zipfile.ZipFile(z, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        external_index = 0
        for path in current:
            rel = os.path.relpath(path, out)
            if rel.startswith('..'):
                external_index += 1
                arcname = 'other_outputs/%02d_%s' % (external_index, os.path.basename(path))
            else:
                arcname = rel
            archive.write(path, arcname)
        archive.writestr('colab_run.json', json.dumps(manifest, indent=2))
        if transcript: archive.writestr('colab_run.log', transcript)
    try:
        from google.colab import files as _f; _f.download(z)
    except Exception as e:
        with res_out: print('Download needs Colab; zip saved at', z, '(', e, ')')
dl_btn.on_click(_download)
results_empty = W.HTML(
    '<div role="status" style="padding:10px;border:1px dashed #64748b;border-radius:10px;">'
    'No results yet.</div>')
artifact_box = W.VBox([artifact_choice, artifact_out])
artifact_fold = _collapsible('Generated file preview', artifact_box)
artifact_fold.layout.display = 'none'
artifact_fold.add_class('rxartifact')
trajectory_head = W.HBox([trajectory_intro, frame_state],
                          layout=W.Layout(width='100%', justify_content='space-between',
                                          align_items='center'))
trajectory_head.add_class('rxpath-head')
frame_controls = W.HBox([frame_prev, frame_play, frame_next, frame_slider],
                         layout=W.Layout(width='100%', align_items='center'))
frame_controls.add_class('rxpath-controls')
structure_panel = W.VBox([
    W.HTML('<div class="rxpath-panel-title">Structure</div>'), traj_out])
energy_panel = W.VBox([
    W.HTML('<div class="rxpath-panel-title">Energy profile</div>'), plot_out])
structure_panel.add_class('rxpath-panel'); energy_panel.add_class('rxpath-panel')
path_grid = W.HBox([structure_panel, energy_panel], layout=W.Layout(width='100%'))
path_grid.add_class('rxpath-grid')
trajectory_box = W.VBox([
    trajectory_head, traj_choice, traj_label, frame_controls, path_grid])
trajectory_box.layout.display = 'none'
trajectory_box.add_class('rxresults'); trajectory_box.add_class('rxpath')
results_box = W.VBox([
    W.HBox([res_btn, dl_btn], layout=W.Layout(flex_flow='row wrap')),
    result_context, res_out, results_empty, trajectory_box, artifact_fold])
results_box.add_class('rxresults')

# ============================================================== command line + run (bottom, PyMOL-style)
logbox = W.Output(layout={'border': '1px solid #ccc', 'max_height': '380px', 'overflow': 'auto'})
def _stop_child(process):
    if process.poll() is not None: return
    try:
        if os.name == 'posix': os.killpg(os.getpgid(process.pid), signal.SIGTERM)
        else: process.terminate()
    except ProcessLookupError:
        return
    try: process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        try:
            if os.name == 'posix': os.killpg(os.getpgid(process.pid), signal.SIGKILL)
            else: process.kill()
        except ProcessLookupError:
            pass
        process.wait()

def _stream(cmd):
    command_line = '$ ' + ' '.join(shlex.quote(c) for c in cmd) + '\n\n'
    transcript = [command_line]
    with logbox:
        clear_output(); print(command_line, end='')
        try:
            p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                 encoding='utf-8', errors='replace', bufsize=1,
                                 start_new_session=(os.name == 'posix'))
        except OSError as exc:
            line = '[launch failed] %s\n[exit 127]\n' % exc
            transcript.append(line); print(line, end='')
            return 127, ''.join(transcript)
        try:
            for line in p.stdout:
                transcript.append(line); print(line, end='')
            p.wait()
        except KeyboardInterrupt:
            line = '\n[interrupt] stopping the child process …\n'
            transcript.append(line); print(line, end='')
            _stop_child(p)
            transcript.append('[exit 130 · cancelled]\n'); print('[exit 130 · cancelled]')
            return 130, ''.join(transcript)
        except Exception as exc:
            _stop_child(p)
            line = '\n[stream failed] %s\n[exit 125]\n' % exc
            transcript.append(line); print(line, end='')
            return 125, ''.join(transcript)
        finally:
            if p.stdout is not None: p.stdout.close()
        trailer = '\n[exit %d]\n' % p.returncode
        transcript.append(trailer); print(trailer, end='')
    return p.returncode, ''.join(transcript)
def _argv():
    line = cmd_box.value.strip()
    if not line or line.startswith('#'):
        with logbox: clear_output(); print('No command yet — load an input (Input tab).')
        return None
    try:
        return shlex.split(line)   # execute exactly what the editable command line shows
    except ValueError as exc:
        with logbox: clear_output(); print('Invalid command line:', exc)
        _set_run_status('✗ invalid command', 'error', 'invalid')
        _open_run_log()
        return None

def _normalized_scope_argv(argv):
    """Apply this root CLI's argv normalization for output classification."""
    args = list(argv or [])
    if len(args) < 2: return args
    try:
        from pdb2reaction.cli.bool_compat import normalize_argv_option_names
        args = [args[0], *normalize_argv_option_names(args[1:])]
    except Exception:
        pass
    first = args[1]
    if first in ('-h', '--help', '--version'): return args
    try:
        import click
        from pdb2reaction.cli import cli as root_cli
        root_ctx = click.Context(root_cli, info_name=CLI, resilient_parsing=True)
        if root_cli.get_command(root_ctx, first) is None:
            top_level = set()
            for param in root_cli.params:
                top_level.update(getattr(param, 'opts', ()) or ())
                top_level.update(getattr(param, 'secondary_opts', ()) or ())
            if first.startswith('--'):
                is_top_level = first.split('=', 1)[0] in top_level
            elif first.startswith('-') and len(first) >= 2:
                is_top_level = first[:2] in top_level
            else:
                is_top_level = False
            if first.startswith('-') and not is_top_level:
                args.insert(1, 'all')
        if len(args) > 1 and not args[1].startswith('-'):
            command_name = args[1]
            value_opts, toggle_opts, negative_aliases, single_flags = (
                root_cli._resolve_bool_options(root_ctx, command_name))
            normalized, _legacy = root_cli._normalize_bool_argv(
                args[1:], {command_name: value_opts},
                {command_name: toggle_opts}, {command_name: negative_aliases},
                {command_name: single_flags})
            args = [args[0], *normalized]
    except Exception:
        pass
    return args

def _effective_out_dir(argv=None):
    """Return the last output path in the editable command, with GUI fallback."""
    if argv is None:
        line = cmd_box.value.strip()
        try: argv = shlex.split(line) if line and not line.startswith('#') else []
        except ValueError: argv = []
    argv = _normalized_scope_argv(argv)
    out = None
    flags = ('-o', '--out', '--out-dir', '--output')
    for i, token in enumerate(argv):
        if token in flags and i + 1 < len(argv):
            out = argv[i + 1]
        elif token.startswith('-o') and not token.startswith('--') and token != '-o':
            out = token[2:]
        else:
            for flag_name in flags:
                if token.startswith(flag_name + '='):
                    out = token.split('=', 1)[1]
    return out or _click_output_default(argv) or S.get('out_dir') or 'result'

def _click_subcommand_params(argv):
    """Parse the pinned command without invoking callbacks or validating files."""
    try:
        argv = _normalized_scope_argv(argv)
        import click
        from pdb2reaction.cli import cli as root_cli
        root_ctx = click.Context(root_cli, info_name=CLI, resilient_parsing=True)
        command = root_cli.get_command(root_ctx, argv[1])
        if command is None: return {}
        sub_ctx = click.Context(command, info_name=argv[1], parent=root_ctx,
                                resilient_parsing=True)
        parsed, _remaining, _order = command.make_parser(sub_ctx).parse_args(
            args=list(argv[2:]))
        return parsed
    except Exception:
        return {}

def _click_output_default(argv):
    """Read the selected command's own output-option default from Click."""
    try:
        import click
        from pdb2reaction.cli import cli as root_cli
        argv = _normalized_scope_argv(argv)
        root_ctx = click.Context(root_cli, info_name=CLI, resilient_parsing=True)
        command = root_cli.get_command(root_ctx, argv[1])
        output_flags = {'-o', '--out', '--out-dir', '--output'}
        for param in command.params if command is not None else ():
            if output_flags.intersection(getattr(param, 'opts', ()) or ()):
                value = getattr(param, 'default', None)
                if isinstance(value, (str, os.PathLike)) and str(value):
                    return str(value)
    except Exception:
        pass
    return None

def _trj2fig_output_targets(argv):
    """Parse every trj2fig output accepted by this pinned CLI release."""
    parsed = _click_subcommand_params(argv)
    outs = parsed.get('outs')
    extra_outs = parsed.get('extra_outs')
    parsed_outputs = (list(outs) if isinstance(outs, (list, tuple)) else [])
    parsed_outputs += (list(extra_outs) if isinstance(extra_outs, (list, tuple)) else [])
    if parsed_outputs:
        return list(dict.fromkeys(str(path) for path in parsed_outputs))
    value_flags = {'-v', '--verbose', '-i', '--input', '--unit', '-r', '--reference',
                   '-q', '--charge', '-m', '--multiplicity', '-b', '--backend',
                   '--solvent', '--solvent-model', '--backend-model', '--precision'}
    output_exts = {'.png', '.jpg', '.jpeg', '.html', '.svg', '.pdf', '.csv'}
    flagged, positional = [], []
    args = list(argv[2:])
    positional_only = False
    i = 0
    while i < len(args):
        token = args[i]
        if token == '--':
            positional_only = True; i += 1; continue
        if not positional_only and token in ('-o', '--out'):
            if i + 1 < len(args): flagged.append(args[i + 1])
            i += 2; continue
        if not positional_only and token.startswith('--out='):
            flagged.append(token.split('=', 1)[1]); i += 1; continue
        if not positional_only and token.startswith('-o') and token != '-o':
            flagged.append(token[2:]); i += 1; continue
        if not positional_only and token in value_flags:
            i += 2; continue
        if not positional_only and any(token.startswith(flag + '=') for flag in value_flags if flag.startswith('--')):
            i += 1; continue
        if positional_only or (not token.startswith('-') and Path(token).suffix.lower() in output_exts):
            positional.append(token)
        i += 1
    found = flagged + positional
    if not found: found = ['energy.png']
    return list(dict.fromkeys(found))

def _flag_enabled(argv, positive, negative):
    enabled = False
    for token in argv:
        if token == positive: enabled = True
        elif token == negative: enabled = False
    return enabled

def _force_dry_run(argv):
    """Append the validating flag before a positional-only ``--`` marker."""
    args = list(argv)
    index = args.index('--') if '--' in args else len(args)
    args.insert(index, '--dry-run')
    return args

def _grouped_option_values(argv, flags, unique=True):
    """Collect repeated or space-grouped values for legacy variadic options."""
    found = []
    i = 2
    while i < len(argv):
        token = argv[i]
        if token in flags:
            i += 1
            while i < len(argv) and not argv[i].startswith('-'):
                found.append(argv[i]); i += 1
            continue
        matched = False
        for flag in flags:
            if flag.startswith('--') and token.startswith(flag + '='):
                found.append(token.split('=', 1)[1]); matched = True; break
        if not matched and '-o' in flags and token.startswith('-o') and token != '-o':
            found.append(token[2:]); matched = True
        i += 1
    return list(dict.fromkeys(found)) if unique else found

def _parsed_path(parsed, key):
    """Return one explicitly parsed CLI path, excluding Click sentinels."""
    value = parsed.get(key)
    return str(value) if isinstance(value, (str, os.PathLike)) and str(value) else None

def _input_needs_cif_companion(inputs):
    """Match the product's mmCIF/PDB-overflow normalization boundary."""
    for value in inputs:
        path = Path(value)
        if path.suffix.lower() in ('.cif', '.mmcif'): return True
        if path.suffix.lower() == '.pdb' and path.is_file():
            try:
                from pdb2reaction.io.structure_formats import pdb_requires_normalization
                if pdb_requires_normalization(path): return True
            except Exception:
                pass
    return False

def _exact_output_scope(outputs, include_json=False, companions=(), expand_user=False):
    """Build a current-run scope from the exact files a command may own."""
    def _absolute(path):
        value = os.path.expanduser(str(path)) if expand_user else str(path)
        return os.path.abspath(value)
    paths = [_absolute(path) for path in outputs]
    paths = list(dict.fromkeys(paths))
    root = os.path.dirname(paths[0]) or '.'
    tracked = list(paths)
    tracked.extend(_absolute(path) for path in companions)
    if include_json:
        tracked.extend((os.path.join(root, 'result.json'),
                        os.path.join(root, 'summary.json')))
    return {'target': paths[0], 'targets': paths, 'root': root,
            'shallow': True, 'prefix': None,
            'exact_targets': list(dict.fromkeys(tracked)),
            'direct_current': True}

def _output_scope(argv=None):
    """Resolve exact file targets or the directory owned by this argv."""
    if argv is None:
        try: argv = shlex.split(cmd_box.value.strip())
        except ValueError: argv = []
    argv = _normalized_scope_argv(argv)
    # Click's eager information flags exit after printing and never own output
    # files. Stop scanning at ``--`` so a positional token is not misread.
    visible = argv[1:argv.index('--')] if '--' in argv else argv[1:]
    if any(token in ('-h', '--help', '--help-advanced', '--version')
           for token in visible):
        return {'target': 'standard output', 'targets': [], 'root': os.getcwd(),
                'shallow': True, 'prefix': None, 'exact_targets': [],
                'direct_current': True, 'stdout_only': True}
    target = _effective_out_dir(argv)
    sub = argv[1] if len(argv) > 1 else S.get('subcmd')
    if sub == 'bond-summary':
        return {'target': 'standard output', 'targets': [], 'root': os.getcwd(),
                'shallow': True, 'prefix': None, 'exact_targets': [],
                'direct_current': True, 'stdout_only': True}
    if sub == 'trj2fig':
        outputs = [os.path.abspath(os.path.expanduser(path))
                   for path in _trj2fig_output_targets(argv)]
        return _exact_output_scope(
            outputs, _flag_enabled(argv, '--out-json', '--no-out-json'),
            expand_user=True)
    if sub == 'extract':
        raw_outputs = _grouped_option_values(argv, ('-o', '--output'), unique=False)
        inputs = _grouped_option_values(argv, ('-i', '--input'), unique=False)
        if len(inputs) == 1 and raw_outputs:
            raw_outputs = raw_outputs[:1]
        if not raw_outputs and inputs:
            raw_outputs = (['model.pdb'] if len(inputs) == 1 else
                           ['model_%s.pdb' % Path(path).stem for path in inputs])
        if raw_outputs:
            outputs = list(raw_outputs)
            companions = []
            if len(outputs) == len(inputs) and len(outputs) > 1:
                companions = [Path(output).with_suffix('.cif')
                              for output, input_path in zip(outputs, inputs)
                              if _input_needs_cif_companion([input_path])]
            elif outputs and inputs and _input_needs_cif_companion([inputs[0]]):
                companions = [Path(outputs[0]).with_suffix('.cif')]
            return _exact_output_scope(
                outputs, _flag_enabled(argv, '--out-json', '--no-out-json'),
                companions)
    parsed = _click_subcommand_params(argv)
    if sub == 'energy-diagram':
        output = _parsed_path(parsed, 'output_path') or 'energy_diagram.png'
        if not Path(output).suffix: output += '.png'
        return _exact_output_scope(
            [output], _flag_enabled(argv, '--out-json', '--no-out-json'))
    if sub == 'add-elem-info':
        input_path = _parsed_path(parsed, 'in_pdb')
        output = _parsed_path(parsed, 'out_pdb')
        if input_path and not output:
            source = Path(input_path)
            if _flag_enabled(argv, '--overwrite', '--no-overwrite'):
                output = str(source)
            elif source.name.lower().endswith('.pdb'):
                output = str(source.with_name(source.name[:-4] + '_add_elem.pdb'))
            else:
                output = str(source.with_name(source.name + '_add_elem.pdb'))
        if output: return _exact_output_scope([output])
    if sub == 'fix-altloc':
        input_path = _parsed_path(parsed, 'input_path')
        output = _parsed_path(parsed, 'out')
        inplace = _flag_enabled(argv, '--inplace', '--no-inplace')
        if input_path and os.path.isdir(input_path):
            source = os.path.abspath(input_path)
            owned = source if inplace else (
                os.path.abspath(output) if output else
                str(Path(source).with_name(Path(source).name + '_clean')))
            return {'target': owned, 'targets': [], 'root': owned,
                    'shallow': False, 'prefix': None, 'exact_targets': [],
                    'direct_current': False}
        if input_path:
            source = Path(input_path)
            if inplace:
                return _exact_output_scope(
                    [source, source.with_suffix(source.suffix + '.bak')])
            if output:
                candidate = Path(output)
                output = candidate if candidate.suffix.lower() == '.pdb' else candidate / source.name
            else:
                output = source.with_name(source.stem + '_clean.pdb')
            return _exact_output_scope([output])
    file_output_subs = {'extract', 'fix-altloc', 'add-elem-info', 'energy-diagram',
                        'trj2fig', 'oniom-export', 'oniom-import'}
    if sub in file_output_subs and Path(target).suffix:
        absolute = os.path.abspath(target)
        tracked = [absolute]
        if _flag_enabled(argv, '--out-json', '--no-out-json'):
            parent = os.path.dirname(absolute) or '.'
            tracked.extend((os.path.join(parent, 'result.json'),
                            os.path.join(parent, 'summary.json')))
        return {'target': target, 'targets': [absolute],
                'root': os.path.dirname(absolute) or '.', 'shallow': True,
                'prefix': None, 'exact_targets': tracked, 'direct_current': True}
    return {'target': target, 'targets': [], 'root': target, 'shallow': False,
            'prefix': None, 'exact_targets': [], 'direct_current': False}

def _effective_result_root(argv=None):
    """Resolve the primary directory Results should inspect for this argv."""
    return _output_scope(argv)['root']

def _matches_output_scope(path, scope):
    exact = scope.get('exact_targets') or []
    return not exact or os.path.abspath(path) in set(exact)

def _snapshot_files(root, shallow=False):
    root = os.path.abspath(root)
    if not os.path.isdir(root): return {}
    paths = glob.glob(os.path.join(root, '*')) if shallow else glob.glob(os.path.join(root, '**', '*'), recursive=True)
    snap = {}
    for path in paths:
        if not os.path.isfile(path): continue
        st = os.stat(path); snap[os.path.abspath(path)] = (st.st_size, st.st_mtime_ns)
    return snap

def _snapshot_output_scope(scope):
    if scope.get('stdout_only'): return {}
    exact = scope.get('exact_targets') or []
    if not exact: return _snapshot_files(scope['root'], shallow=scope['shallow'])
    snap = {}
    for path in exact:
        if not os.path.isfile(path): continue
        st = os.stat(path); snap[os.path.abspath(path)] = (st.st_size, st.st_mtime_ns)
    return snap

def _output_scope_collision(scope):
    if scope.get('stdout_only'): return False
    exact = scope.get('exact_targets') or []
    if exact: return any(os.path.lexists(path) for path in exact)
    root = scope['root']
    return os.path.isdir(root) and bool(os.listdir(root))

def _structured_current_paths(root, changed):
    """Use machine-readable claims when present; otherwise use the file delta."""
    root = os.path.abspath(root); claimed = set(); status_json = set(); has_claims = False
    changed_abs = {os.path.abspath(path) for path in changed}
    def _add(value, base=None):
        if not isinstance(value, str) or not value: return
        if os.path.isabs(value): candidates = [value]
        else:
            candidates = []
            if base is not None: candidates.append(os.path.join(base, value))
            candidates.extend((os.path.join(root, value), os.path.abspath(value)))
        candidates = list(dict.fromkeys(os.path.abspath(path) for path in candidates))
        path = next((path for path in candidates if path in changed_abs and os.path.isfile(path)), None)
        if path is None: path = next((path for path in candidates if os.path.isfile(path)), None)
        if path is not None: claimed.add(path)
    for path in list(changed):
        if os.path.basename(path) not in ('result.json', 'summary.json'): continue
        status_json.add(os.path.abspath(path))
        try:
            with open(path) as fh: payload = json.load(fh)
        except Exception:
            continue
        if 'current_output_paths' in payload:
            has_claims = True
            for value in payload.get('current_output_paths') or []: _add(value)
        if 'output_files' in payload:
            has_claims = True
            for value in payload.get('output_files') or []: _add(value)
        if 'files' in payload:
            has_claims = True
            values = payload.get('files') or {}
            values = values.values() if isinstance(values, dict) else values
            for value in values: _add(value)
        if 'key_output_files' in payload:
            has_claims = True
            for key, value in (payload.get('key_output_files') or {}).items():
                if isinstance(value, dict):
                    base = os.path.join(root, 'segments', key) if str(key).startswith('seg_') else root
                    for rel in value.get('files') or []: _add(rel, base)
                else:
                    _add(key)
    return sorted((claimed | status_json) if has_claims else set(changed))

def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for block in iter(lambda: fh.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()

def _command_input_option_flags(argv):
    """Derive existing-file option arity from pdb2reaction's selected command."""
    fallback_multi = {
        '-i', '--input', '--ref-pdb', '--ref-full-pdb', '-s', '--scan-lists',
    }
    fallback_single = {'--config', '--calc-file', '--ref-mode', '--csv'}
    args = _normalized_scope_argv(argv)
    try:
        import click
        from pdb2reaction.cli import cli as root_cli
        if len(args) < 2 or args[1].startswith('-'):
            return fallback_multi, fallback_single
        root_ctx = click.Context(root_cli, info_name=CLI, resilient_parsing=True)
        command = root_cli.get_command(root_ctx, args[1])
        if command is None:
            return fallback_multi, fallback_single
        multi, single = set(), set()
        for param in command.params:
            value_type = getattr(param, 'type', None)
            flags = set(getattr(param, 'opts', ()) or ())
            flags.update(getattr(param, 'secondary_opts', ()) or ())
            # scan-lists deliberately accepts either an inline literal or a YAML file,
            # so Click exposes it as STRING; hash only values that resolve to real files.
            if flags.intersection({'-s', '--scan-lists'}):
                multi.update(flags)
                continue
            if not (isinstance(value_type, click.Path) and value_type.exists and value_type.file_okay):
                continue
            if getattr(param, 'multiple', False) or getattr(param, 'nargs', 1) != 1:
                multi.update(flags)
            else:
                single.update(flags)
        return (multi, single) if (multi or single) else (fallback_multi, fallback_single)
    except Exception:
        return fallback_multi, fallback_single

def _click_parsed_input_files(argv):
    """Read option and positional input paths through Click's parser, without invoking."""
    args = _normalized_scope_argv(argv)
    try:
        import click
        from pdb2reaction.cli import cli as root_cli
        if len(args) < 2 or args[1].startswith('-'): return []
        root_ctx = click.Context(root_cli, info_name=CLI, resilient_parsing=True)
        command = root_cli.get_command(root_ctx, args[1])
        if command is None: return []
        command_ctx = click.Context(command, info_name=args[1], parent=root_ctx,
                                    resilient_parsing=True)
        parsed, _remaining, _order = command.make_parser(command_ctx).parse_args(args[2:])
        found = []
        for param in command.params:
            value_type = getattr(param, 'type', None)
            if not (isinstance(value_type, click.Path) and value_type.exists and value_type.file_okay):
                continue
            value = parsed.get(param.name)
            values = value if isinstance(value, (tuple, list)) else (value,)
            for path in values:
                if path is not None and os.path.isfile(os.fspath(path)):
                    found.append(os.path.abspath(os.fspath(path)))
        return found
    except Exception:
        return []

def _command_input_files(argv):
    argv = _normalized_scope_argv(argv)
    multi, single = _command_input_option_flags(argv)
    found = []
    i = 0
    while i < len(argv):
        token = argv[i]
        if token in multi:
            i += 1
            while i < len(argv) and not argv[i].startswith('-'):
                if os.path.isfile(argv[i]): found.append(os.path.abspath(argv[i]))
                i += 1
            continue
        if token in single and i + 1 < len(argv):
            if os.path.isfile(argv[i + 1]): found.append(os.path.abspath(argv[i + 1]))
            i += 2; continue
        for flag in multi | single:
            if token.startswith(flag + '=') and os.path.isfile(token.split('=', 1)[1]):
                found.append(os.path.abspath(token.split('=', 1)[1]))
            elif (flag.startswith('-') and not flag.startswith('--') and token != flag
                  and token.startswith(flag) and os.path.isfile(token[len(flag):])):
                found.append(os.path.abspath(token[len(flag):]))
        i += 1
    found.extend(_click_parsed_input_files(argv))
    return sorted(set(found))

def _validation_fingerprint(argv):
    files = []
    for path in _command_input_files(argv):
        try: digest = _sha256(path)
        except OSError: digest = '<missing>'
        files.append((os.path.abspath(path), digest))
    return (_command_fingerprint(cmd_box.value), tuple(files))
b_rebuild = W.Button(description='Rebuild', icon='magic', layout=W.Layout(width='120px'),
                     tooltip='Regenerate the command from the current GUI selections.')
b_rebuild.on_click(lambda _: (_auto.__setitem__('on', True), refresh()))
b_copy = W.Button(description='Copy', icon='copy', layout=W.Layout(width='100px'), tooltip='Copy the command line.')
def _copy(_):
    try:
        from google.colab import output as _o
        _o.eval_js('navigator.clipboard.writeText(%s)' % json.dumps(cmd_box.value))
        toast.value = '<small style="color:#1f7a3d">copied ✓</small>'
    except Exception as e:
        toast.value = '<small>copy needs Colab (%s)</small>' % e
b_copy.on_click(_copy)
b_clear = W.Button(description='Clear selections', icon='eraser', layout=W.Layout(width='160px'))
def _clear_sel(_):
    if not _view_is_editable(): return
    S['center'] = []; S['lcharge'] = {}
    if center_widget is not None: center_widget.value = ()
    if charge_rows is not None:
        for x in charge_rows.values(): x['use'].value = bool(x.get('auto'))
    S.update(center_ids=[], scan_atoms=[None, None], scan_preset='', scan_stages=[], scan_axes=[],
             freeze_buf=[None, None], freeze_pairs=[], freeze_atoms=[], measure_atoms=[],
             _last_pick=None, _pick_history=[], _last_pick_message='', _last_pick_tone='ok')
    _render_center_ids(); _render_scan_panel(); _render_freeze_panel()
    _render_last_pick_status()
    render_viewer(); refresh()
b_clear.on_click(_clear_sel)
b_validate = W.Button(description='Validate', button_style='info', icon='check', layout=W.Layout(width='130px'),
                      tooltip='Run the displayed compute command with --dry-run appended; no MLIP stage runs.')
b_run = W.Button(description='Run', button_style='danger', icon='play', layout=W.Layout(width='130px'),
                 tooltip='Execute exactly the command shown above. This may start a costly GPU calculation.')
def _set_running(value):
    _ACTION_STATE['running'] = bool(value)
    for button in (b_rebuild, b_clear): button.disabled = value
    cmd_box.disabled = value
    _sync_action_enabled()
def _open_run_log():
    fold = globals().get('run_log_fold')
    if fold is not None: fold._rx_set_open(True)
def _do_validate(_):
    _sync_run(); a = _argv()
    if not a: return
    fingerprint = _validation_fingerprint(a)
    effective = _normalized_scope_argv(a)
    sub = effective[1] if len(effective) > 1 else ''
    if sub not in COMPUTE:
        with logbox: clear_output(); print('Validate is available for compute subcommands; use the command help for this utility.')
        _set_run_status('validation is for compute workflows', 'warn', 'utility')
        _open_run_log()
        return
    # A final canonical flag wins over legacy ``--dry-run False`` and an
    # explicit ``--no-dry-run``.  Insert before ``--`` so Click parses it.
    a = _force_dry_run(a)
    _set_running(True)
    try:
        _set_run_status('🔎 validating…', 'info', 'validating')
        rc, validation_log = _stream(a)
        _RUN_STATE['validation_log'] = validation_log
        if rc == 0:
            _RUN_STATE['validated_fingerprint'] = fingerprint
            _set_run_status('✓ valid', 'ok', 'valid')
        else:
            _RUN_STATE['validated_fingerprint'] = None
            _set_run_status('✗ validation failed', 'error', 'invalid')
            _open_run_log()
    finally:
        _set_running(False)
b_validate.on_click(_do_validate)
def _do_run(_):
    _sync_run(); a = _argv()
    if not a: return
    effective = _normalized_scope_argv(a)
    sub = effective[1] if len(effective) > 1 else ''
    validated = _RUN_STATE.get('validated_fingerprint')
    if sub in COMPUTE and validated and validated != _validation_fingerprint(a):
        _RUN_STATE['validated_fingerprint'] = None
        _set_run_status('validated inputs changed · validate again', 'warn', 'changed')
        with logbox: clear_output(); print('The command or an input file changed after validation. Validate this exact run again.')
        _open_run_log()
        return
    if sub not in COMPUTE and sub != 'extract' and sub not in AUTOFILL_UTILS and _auto['on']:
        with logbox:
            clear_output(); print('This utility needs command-specific arguments.')
            print('Edit the command line using its --help, then run the exact edited command.')
        _set_run_status('◆ finish utility command', 'warn', 'utility')
        _open_run_log()
        return
    scope = _output_scope(a)
    target = scope['target']; out = scope['root']
    collision = _output_scope_collision(scope)
    if collision and not w_reuse.value:
        _set_run_status('✗ output exists', 'error', 'collision')
        with logbox:
            clear_output(); print('Refusing to overwrite or mix an existing output:', target if scope['shallow'] else out)
            print('Choose a new out dir, or enable "reuse non-empty out dir" in Options.')
        _open_run_log()
        _tab_go(2)
        return
    before = _snapshot_output_scope(scope)
    started = time.time()
    input_files = _command_input_files(a)
    real_run = not _flag_enabled(effective, '--dry-run', '--no-dry-run')
    if real_run:
        S.update(_last_out_dir=out, _last_subcmd=(sub or S.get('subcmd')),
                 _last_argv=list(a), _last_files=[], _last_log='',
                 _last_manifest={'tool': TOOL, 'subcommand': sub, 'argv': list(a),
                                 'status': 'running', 'output_root': out,
                                 'stdout_only': bool(scope.get('stdout_only'))})
        dl_btn.disabled = True
    _set_running(True)
    try:
        _RUN_STATE['validated_fingerprint'] = None
        _set_run_status('⏳ running…', 'warn', 'running')
        rc, transcript = _stream(a)
        if real_run: S['_last_log'] = transcript
        else: _RUN_STATE['validation_log'] = transcript
        if rc == 0: _set_run_status('✓ done', 'ok', 'done')
        elif rc == 130: _set_run_status('■ cancelled', 'warn', 'cancelled')
        else: _set_run_status('✗ failed (exit %d)' % rc, 'error', 'failed')
        if rc != 0: _open_run_log()
        if real_run:
            after = _snapshot_output_scope(scope)
            changed = sorted(path for path, stat in after.items()
                             if before.get(path) != stat and _matches_output_scope(path, scope))
            current = changed if scope.get('direct_current') else _structured_current_paths(out, changed)
            try:
                from importlib.metadata import version as _version
                package = 'pdb2reaction'
                installed = _version(package)
            except Exception:
                installed = None
            S['_last_out_dir'] = out
            S['_last_subcmd'] = sub or S.get('subcmd')
            S['_last_argv'] = list(a)
            S['_last_files'] = current
            S['_last_manifest'] = {
                'tool': TOOL, 'version': installed, 'subcommand': S['_last_subcmd'],
                'argv': list(a), 'started_unix': started, 'finished_unix': time.time(),
                'output_root': out, 'exit_code': rc,
                'stdout_only': bool(scope.get('stdout_only')),
                'status': 'success' if rc == 0 else ('cancelled' if rc == 130 else 'failed'),
                'inputs': [{'path': path, 'sha256': _sha256(path)} for path in input_files],
                'current_files': [os.path.relpath(path, out) for path in current],
            }
            _results(out); _tab_go(3)       # every real attempt replaces prior Results state
    finally:
        _set_running(False)
        if real_run: w_reuse.value = False
b_run.on_click(_do_run)
def _session_number(value, name, integer=False, minimum=None, maximum=None):
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(float(value)):
        raise ValueError('%s must be a finite number.' % name)
    if integer and int(value) != value: raise ValueError('%s must be an integer.' % name)
    value = int(value) if integer else float(value)
    if minimum is not None and value < minimum: raise ValueError('%s is below %s.' % (name, minimum))
    if maximum is not None and value > maximum: raise ValueError('%s is above %s.' % (name, maximum))
    return value

def _session_strings(value, name):
    if not isinstance(value, list) or not all(isinstance(item, str) and item for item in value):
        raise ValueError('%s must be a list of non-empty strings.' % name)
    return list(value)

def _session_atom(value, name):
    if not isinstance(value, dict): raise ValueError('%s must be an atom record.' % name)
    atom = dict(value)
    for field in ('chain', 'resn', 'resi', 'atom'):
        if field in atom and not isinstance(atom[field], str):
            raise ValueError('%s.%s must be text.' % (name, field))
    if 'index' in atom and atom['index'] is not None:
        atom['index'] = _session_number(atom['index'], name + '.index', integer=True, minimum=0)
    xyz = atom.get('xyz')
    if xyz is not None:
        if not isinstance(xyz, (list, tuple)) or len(xyz) != 3:
            raise ValueError('%s.xyz must contain three coordinates.' % name)
        atom['xyz'] = tuple(_session_number(item, name + '.xyz') for item in xyz)
    return atom

def _validate_and_normalize_session(payload):
    if not isinstance(payload, dict): raise ValueError('Settings JSON must contain one object.')
    d = dict(payload)
    if d.get('tool') not in (None, TOOL):
        raise ValueError('These settings belong to %s, not %s.' % (d.get('tool'), TOOL))
    version = d.get('schema_version', 1)
    if version != 1: raise ValueError('Unsupported settings schema version: %s.' % version)
    d['tool'] = TOOL; d['schema_version'] = 1
    backend = d.get('backend', S.get('backend', BACKEND))
    if backend not in MODELS: raise ValueError('Unknown backend: %s.' % backend)
    if backend != BACKEND:
        raise ValueError('Saved backend %s is not installed; restart Setup with that backend.' % backend)
    model = d.get('model', DEFAULT_MODEL[backend])
    if model not in MODELS[backend]: raise ValueError('Model %s is not available for %s.' % (model, backend))
    d['backend'] = backend; d['model'] = model
    sub = d.get('subcmd', 'all')
    if sub not in SUBS: raise ValueError('Subcommand %s is unavailable in this runtime.' % sub)
    d['subcmd'] = sub
    all_kind = d.get('all_mode', 'mep')
    if all_kind not in ('mep', 'scan', 'tsonly'): raise ValueError('Unknown all mode: %s.' % all_kind)
    d['all_mode'] = all_kind
    d['inputs'] = _session_strings(d.get('inputs', []), 'inputs')
    for name in ('parm', 'model_pdb'):
        value = d.get(name)
        if value is not None and not isinstance(value, str): raise ValueError('%s must be a path or null.' % name)
        d[name] = value
    mode = d.get('mode')
    if mode not in (None, 'pdb', 'mmcif', 'small', 'utility'): raise ValueError('Unknown input mode: %s.' % mode)
    d['mode'] = mode
    d['center'] = _session_strings(d.get('center', []), 'center')
    d['center_ids'] = _session_strings(d.get('center_ids', []), 'center_ids')
    charges = d.get('lcharge', {})
    if not isinstance(charges, dict) or not all(isinstance(name, str) and name for name in charges):
        raise ValueError('lcharge must map residue names to charges.')
    d['lcharge'] = {name: _session_number(value, 'lcharge.' + name, minimum=-9, maximum=9)
                    for name, value in charges.items()}
    scan_atoms = d.get('scan_atoms', [None, None])
    if not isinstance(scan_atoms, list) or len(scan_atoms) != 2:
        raise ValueError('scan_atoms must contain A and B.')
    d['scan_atoms'] = [None if atom is None else _session_atom(atom, 'scan_atoms') for atom in scan_atoms]
    stages = d.get('scan_stages', [])
    if not isinstance(stages, list): raise ValueError('scan_stages must be a list.')
    normalized_stages = []
    for si, stage in enumerate(stages):
        if not isinstance(stage, list): raise ValueError('scan_stages[%d] must be a list.' % si)
        normalized_stage = []
        for bi, bond in enumerate(stage):
            if not isinstance(bond, dict): raise ValueError('scan stage bond must be an object.')
            normalized_stage.append({'a': _session_atom(bond.get('a'), 'scan stage A'),
                                     'b': _session_atom(bond.get('b'), 'scan stage B'),
                                     't': _session_number(bond.get('t'), 'scan target', minimum=0.3, maximum=8.0)})
        normalized_stages.append(normalized_stage)
    d['scan_stages'] = normalized_stages
    axes = d.get('scan_axes', [])
    if not isinstance(axes, list): raise ValueError('scan_axes must be a list.')
    d['scan_axes'] = [
        {'a': _session_atom(axis.get('a'), 'scan axis A'),
         'b': _session_atom(axis.get('b'), 'scan axis B'),
         'lo': _session_number(axis.get('lo'), 'scan low', minimum=0.3, maximum=8.0),
         'hi': _session_number(axis.get('hi'), 'scan high', minimum=0.3, maximum=8.0)}
        for axis in axes if isinstance(axis, dict)
    ]
    if len(d['scan_axes']) != len(axes): raise ValueError('Every scan axis must be an object.')
    pairs = d.get('freeze_pairs', [])
    if not isinstance(pairs, list): raise ValueError('freeze_pairs must be a list.')
    d['freeze_pairs'] = [
        {'a': _session_atom(pair.get('a'), 'restraint A'),
         'b': _session_atom(pair.get('b'), 'restraint B'),
         't': (None if pair.get('t') is None else _session_number(pair.get('t'), 'restraint target', minimum=0.0))}
        for pair in pairs if isinstance(pair, dict)
    ]
    if len(d['freeze_pairs']) != len(pairs): raise ValueError('Every freeze pair must be an object.')
    frozen = d.get('freeze_atoms', [])
    if not isinstance(frozen, list): raise ValueError('freeze_atoms must be a list.')
    d['freeze_atoms'] = [_session_number(value, 'freeze atom', integer=True, minimum=1) for value in frozen]
    measured = d.get('measure_atoms', [])
    if not isinstance(measured, list) or len(measured) > 4: raise ValueError('measure_atoms accepts up to four atoms.')
    d['measure_atoms'] = [_session_atom(atom, 'measure atom') for atom in measured]
    d['scan_target'] = _session_number(d.get('scan_target', 1.6), 'scan_target', minimum=0.3, maximum=8.0)
    preset = d.get('scan_preset', '')
    if not isinstance(preset, str): raise ValueError('scan_preset must be text.')
    d['scan_preset'] = preset
    for name in ('tsopt', 'thermo', 'charge_explicit', 'show_water', 'surface', 'spin'):
        value = d.get(name, False)
        if not isinstance(value, bool): raise ValueError('%s must be true or false.' % name)
        d[name] = value
    d['charge'] = _session_number(d.get('charge', 0), 'charge', integer=True)
    out_dir = d.get('out_dir', 'result')
    if not isinstance(out_dir, str) or not out_dir.strip(): raise ValueError('out_dir must be non-empty text.')
    d['out_dir'] = out_dir
    if d.get('rep', 'cartoon') not in ('cartoon', 'sticks', 'ball+stick', 'spheres', 'line'):
        raise ValueError('Unknown representation.')
    if d.get('color', 'element') not in ('element', 'chain', 'spectrum'):
        raise ValueError('Unknown colour scheme.')
    d['rep'] = d.get('rep', 'cartoon'); d['color'] = d.get('color', 'element')
    width = _session_number(d.get('viewer_width', 720), 'viewer_width', integer=True)
    if width not in (640, 720, 800): raise ValueError('viewer_width must be 640, 720, or 800.')
    d['viewer_width'] = width
    overrides = d.get('advanced_overrides', {})
    if not isinstance(overrides, dict): raise ValueError('advanced_overrides must be an object.')
    normalized_overrides = {}
    for command_name, values in overrides.items():
        if command_name not in SUBS or not isinstance(values, dict):
            raise ValueError('Invalid advanced overrides for %s.' % command_name)
        params = {param.name: param for param in _advanced_options(command_name)}
        normalized_values = {}
        for name, value in values.items():
            param = params.get(name)
            if param is None or _advanced_status(command_name, param) != 'rendered':
                raise ValueError('Unknown editable advanced option %s.%s.' % (command_name, name))
            is_bool = param.is_bool_flag or isinstance(param.type, click.types.BoolParamType)
            if is_bool and value not in (True, False, None):
                raise ValueError('%s.%s must be true, false, or default.' % (command_name, name))
            if isinstance(param.type, click.Choice) and value is not None and value not in param.type.choices:
                raise ValueError('Invalid choice for %s.%s.' % (command_name, name))
            if not is_bool and not isinstance(param.type, click.Choice) and not isinstance(value, str):
                raise ValueError('%s.%s must be text.' % (command_name, name))
            normalized_values[name] = value
        normalized_overrides[command_name] = normalized_values
    d['advanced_overrides'] = normalized_overrides
    advanced = d.get('advanced', {})
    if not isinstance(advanced, dict): raise ValueError('advanced must be an object.')
    d['advanced'] = {
        'mult': _session_number(advanced.get('mult', 1), 'multiplicity', integer=True, minimum=1),
        'precision': advanced.get('precision', 'auto'), 'deterministic': advanced.get('deterministic', False),
        'mep_mode': advanced.get('mep_mode', '(default)'), 'dmf_backend': advanced.get('dmf_backend', '(default)'),
        'thresh': advanced.get('thresh', '(default)'),
        'radius': _session_number(advanced.get('radius', 2.6), 'radius', minimum=0.0),
        'flatten': advanced.get('flatten', False), 'refine_path': advanced.get('refine_path', False),
        'max_cycles': _session_number(advanced.get('max_cycles', 0), 'max_cycles', integer=True, minimum=0),
        'dft': advanced.get('dft', False), 'dft_func_basis': advanced.get('dft_func_basis', ''),
    }
    if d['advanced']['precision'] not in ('auto', 'fp32', 'fp64'): raise ValueError('Invalid precision.')
    if d['advanced']['mep_mode'] not in tuple(adv_mep.options): raise ValueError('MEP mode is unavailable.')
    if d['advanced']['dmf_backend'] not in tuple(adv_dmf.options): raise ValueError('DMF backend is unavailable.')
    if d['advanced']['thresh'] not in tuple(adv_thresh.options): raise ValueError('Invalid threshold.')
    for name in ('deterministic', 'flatten', 'refine_path', 'dft'):
        if not isinstance(d['advanced'][name], bool): raise ValueError('%s must be true or false.' % name)
    if not isinstance(d['advanced']['dft_func_basis'], str): raise ValueError('dft_func_basis must be text.')
    if d['advanced']['dft'] and not DFT_READY: raise ValueError('DFT support is not installed in this runtime.')
    existing_structures = [p for p in d['inputs'] if os.path.isfile(p) and Path(p).suffix.lower() in ('.pdb','.ent','.cif','.mmcif')]
    _preflight_structures(existing_structures)
    _assert_distinct_session_pairs = []
    a, b = d['scan_atoms']
    if a and b: _assert_distinct_session_pairs.append((a, b))
    for stage in d['scan_stages']:
        _assert_distinct_session_pairs.extend((bond['a'], bond['b']) for bond in stage)
    _assert_distinct_session_pairs.extend((axis['a'], axis['b']) for axis in d['scan_axes'])
    _assert_distinct_session_pairs.extend((pair['a'], pair['b']) for pair in d['freeze_pairs'])
    if any(_same_atom(first, second) for first, second in _assert_distinct_session_pairs):
        raise ValueError('Saved scan/restraint pairs must use two different atoms.')
    return d

def _session_dict():
    if center_widget is not None: S['center'] = list(center_widget.value)
    if charge_rows is not None:
        S['lcharge'] = {r: x['val'].value for r, x in charge_rows.items()
                        if x['use'].value and not x.get('auto')}
    keys = ['tool', 'backend', 'model', 'subcmd', 'inputs', 'parm', 'model_pdb', 'mode', 'center', 'center_ids',
            'lcharge', 'scan_atoms', 'scan_stages', 'scan_axes', 'scan_target', 'scan_preset',
            'freeze_pairs', 'freeze_atoms', 'measure_atoms', 'charge', 'charge_explicit', 'tsopt', 'thermo',
            'out_dir', 'rep', 'color', 'show_water', 'surface', 'spin', 'viewer_width', 'advanced_overrides']
    d = {k: S.get(k) for k in keys}; d['schema_version'] = 1
    d['all_mode'] = _wv('all_mode', 'mep')
    if d['all_mode'] == 'tsonly':
        d['tsopt'] = bool(_ALL_MODE_STATE.get('tsopt_before_tsonly', False))
    d['advanced'] = {'mult': _wv('adv_mult', 1), 'precision': _wv('adv_prec', 'auto'),
                     'deterministic': _wv('adv_det', False), 'mep_mode': _wv('adv_mep', '(default)'),
                     'dmf_backend': _wv('adv_dmf', '(default)'),
                     'thresh': _wv('adv_thresh', '(default)'), 'radius': _wv('adv_radius', 0.0),
                     'flatten': _wv('adv_flatten', False), 'refine_path': _wv('adv_refine', False),
                     'max_cycles': _wv('adv_maxcyc', 0),
                     'dft': _wv('adv_dft', False), 'dft_func_basis': _wv('adv_dftfb', '')}
    return d
def _save_session(_):
    session_path = _runtime_path('session.json')
    with open(session_path, 'w') as fh: json.dump(_session_dict(), fh, indent=1)
    toast.value = '<small style="color:#1f7a3d">saved session.json</small>'
    try:
        from google.colab import files as _f; _f.download(session_path)
    except Exception:
        toast.value = '<small>saved session.json (working dir)</small>'
def _apply_session(payload):
    d = _validate_and_normalize_session(payload)
    _clear_structure_bound_state()
    S.update(_last_pick=None, _pick_history=[],
             _last_pick_message='', _last_pick_tone='ok',
             _view_input_index=0, _view_mapping_ok=True)
    _SESSION_APPLY['active'] = True
    try:
        for key in ('backend', 'model', 'subcmd', 'out_dir', 'tsopt', 'thermo', 'charge', 'rep', 'color',
                    'show_water', 'surface', 'spin', 'viewer_width', 'scan_target', 'scan_preset',
                    'parm', 'model_pdb', 'mode'):
            S[key] = d[key]
        S['inputs'] = list(d['inputs']); S['center'] = list(d['center']); S['center_ids'] = list(d['center_ids'])
        S['lcharge'] = dict(d['lcharge']); S['scan_atoms'] = list(d['scan_atoms'])
        S['scan_stages'] = list(d['scan_stages']); S['scan_axes'] = list(d['scan_axes'])
        S['freeze_pairs'] = list(d['freeze_pairs']); S['freeze_atoms'] = list(d['freeze_atoms'])
        S['measure_atoms'] = list(d['measure_atoms']); S['advanced_overrides'] = dict(d['advanced_overrides'])
        S['charge_explicit'] = d['charge_explicit']
        dd_backend.value = d['backend']; dd_model.options = MODELS[d['backend']]; dd_model.value = d['model']
        if d['subcmd'] not in BASIC_SUBS: cb_advsub.value = True
        dd_subcmd.value = d['subcmd']
        all_mode.value = d['all_mode']; _ALL_MODE_STATE['last'] = d['all_mode']; _ALL_MODE_STATE['user'] = True
        _ALL_MODE_STATE['tsopt_before_tsonly'] = bool(d['tsopt'])
        w_ts.value = True if d['all_mode'] == 'tsonly' else bool(d['tsopt'])
        w_th.value = bool(d['thermo']); w_out.value = d['out_dir']
        w_q.value = int(d['charge']); w_charge_ok.value = bool(d['charge_explicit'])
        dd_rep.value = d['rep']; dd_col.value = d['color']; cb_water.value = d['show_water']
        cb_surf.value = d['surface']; cb_spin.value = d['spin']; dd_size.value = d['viewer_width']
        adv = d['advanced']
        adv_mult.value = int(adv['mult']); adv_prec.value = adv['precision']; adv_det.value = adv['deterministic']
        adv_mep.value = adv['mep_mode']; adv_dmf.value = adv['dmf_backend']; adv_thresh.value = adv['thresh']
        adv_radius.value = float(adv['radius']); adv_flatten.value = adv['flatten']
        adv_refine.value = adv['refine_path']; adv_maxcyc.value = int(adv['max_cycles'])
        adv_dft.value = adv['dft']; adv_dftfb.value = adv['dft_func_basis']
    finally:
        _SESSION_APPLY['active'] = False
    _auto['on'] = True
    _render_advanced_rows()
    loaded_primary = False
    if S['mode'] in ('pdb', 'mmcif', 'small') and S['inputs'] and os.path.exists(S['inputs'][0]):
        loaded_primary = build_selection()
    if loaded_primary:
        if center_widget is not None:
            center_widget.value = tuple(v for v in _center_values() if v in set(S['center']))
        if charge_rows is not None:
            for rn, row in charge_rows.items():
                row['use'].value = bool(row.get('auto')) or rn in S['lcharge']
                if row.get('auto'): row['val'].value = float(_ION_CHARGES[rn])
                elif rn in S['lcharge']: row['val'].value = float(S['lcharge'][rn])
    elif not S.get('_pdb_text'):
        render_viewer()
    _render_input_queue(); _sync_view_input_widget(); _render_freeze_panel()
    _render_last_pick_status(); _sync_capability_controls(); refresh()
    referenced = list(S.get('inputs', [])) + [p for p in (S.get('parm'), S.get('model_pdb')) if p]
    return [p for p in referenced if not os.path.isfile(p)]
up_sess = W.FileUpload(accept='.json', multiple=False, description='load session', layout=W.Layout(width='200px'))
def _on_load_sess(change):
    items = up_sess.value
    pairs = list(items.items()) if isinstance(items, dict) else [(f['name'], f) for f in items]
    try:
        for name, meta in pairs:
            c = meta['content']
            try:
                text = c if isinstance(c, str) else bytes(c).decode('utf-8')
                missing = _apply_session(json.loads(text))
                toast.value = ('<small role="alert" style="color:#92400e">loaded settings; re-upload: %s</small>' %
                               html.escape(', '.join(os.path.basename(p) for p in missing)) if missing else
                               '<small style="color:#1f7a3d">loaded %s</small>' % html.escape(name))
            except Exception as e:
                toast.value = '<small style="color:#a00">load failed: %s</small>' % html.escape(str(e))
    finally:
        _reset_file_upload(up_sess)
up_sess.observe(_on_load_sess, names='value')
b_save = W.Button(description='save settings', icon='save', layout=W.Layout(width='150px'),
                  tooltip='Save GUI settings only; input structures and topology files are not included.'); b_save.on_click(_save_session)
session_row = W.HBox([W.HTML('<small>settings (files excluded):</small>'), b_save, up_sess],
                     layout=W.Layout(flex_flow='row wrap'))
utility_bar = W.HBox([b_clear])
primary_bar = W.HBox([b_validate, b_run]); primary_bar.add_class('rxrun')
run_bar = W.HBox([utility_bar, primary_bar],
                 layout=W.Layout(width='100%', justify_content='space-between'))
interrupt_note = W.HTML('<small>To cancel a long run, use <b>Runtime → Interrupt execution</b>. '
                        'The GUI terminates and reaps the active child process.</small>')
run_log_fold = _collapsible('Run log', logbox)
session_more = _collapsible('Session & cancellation', W.VBox([interrupt_note, session_row]))
command_editor = W.VBox([
    W.HTML('<b>Command line</b>'), cmd_box, W.HBox([b_rebuild, b_copy])])
command_editor.add_class('rxcard')
cmdline_box = W.VBox([
    W.HBox([ready_chip, run_status, toast]),
    command_editor, run_bar, run_log_fold, session_more])

# ============================================================== assemble
# Colab's widget frontend renders ipywidgets' Tab as an empty block, so the tab
# strip is built from plain Buttons + a swapping VBox — widgets Colab draws
# reliably. _goto()/_tab_go() drive navigation.
_TAB_PAGES = [('1 Input', input_box), ('2 Viewer', viewer_box),
              ('3 Options', options_box), ('4 Results', results_box)]
_tab_body = W.VBox([page for _label, page in _TAB_PAGES])
_tab_status = W.HTML()
_tab_status.add_class('rxsr-only')
_tab_btns = []
def _tab_go(i):
    i = max(0, min(len(_TAB_PAGES) - 1, int(i)))
    _close_info()
    # Keep every pane mounted. Replacing children aborts browser-side
    # upload queues and can recreate a live WebGL output in Colab.
    for _j, (_label, _pane) in enumerate(_TAB_PAGES):
        _pane.layout.display = '' if _j == i else 'none'
    _tab_status.value = ('<div role="status" aria-live="polite" aria-atomic="true" '
                         'style="color:#475569;font-size:12px;margin:0 2px 5px">'
                         'Step %d of %d · <b>%s</b></div>' %
                         (i + 1, len(_TAB_PAGES), _TAB_PAGES[i][0].split(' ', 1)[1]))
    for _j, _b in enumerate(_tab_btns):
        _b.button_style = 'primary' if _j == i else ''
        _b.description = _TAB_PAGES[_j][0]
    utility_bar.layout.display = '' if i == 1 else 'none'
for _i, (_t, _pane) in enumerate(_TAB_PAGES):
    _btn = W.Button(description=_t, layout=W.Layout(min_width='118px'))
    _btn.on_click(lambda _c, _k=_i: _tab_go(_k))
    _tab_btns.append(_btn)
_tab_go(0)
_tab_strip = W.HBox(_tab_btns, layout=W.Layout(width='100%', flex_flow='row wrap'))
_tab_strip.add_class('rxtabs')
app = W.VBox([_tab_strip, _tab_status, _tab_body])
render_viewer()
refresh()
_t = 'pdb2reaction'
header = W.HTML(
    '<div class="rxheader">'
    '<span class="rxheader-product">%s</span>'
    '<span class="rxheader-meta"><span class="rxheader-chip">COLAB WORKSPACE</span>'
    '<span>backend <b style="color:#e2e8f0;">%s</b></span></span></div>'
    % (_t, BACKEND))
rootbox = W.VBox([header, app, W.HTML('<hr style="margin:5px 0">'), cmdline_box])
rootbox.add_class('rxapp')
display(rootbox)


# Hosted Colab's output callback is the reliable binary-transfer path for this
# ordinary ipywidgets upload zone. Browser batches are FIFO, carry an opaque
# identity and clear-generation, and all feed the same append-only validator.
try:
    from google.colab import output as _dnd_out
    import base64 as _dnd_b64
    def _rxgui_drop(files, batch, generation):
        claimed, reason = _claim_drop_batch(batch, generation)
        if not claimed:
            return {'ok': False, 'n': 0, 'batch': str(batch or ''),
                    'generation': generation, 'stale': reason == 'stale generation',
                    'error': reason}
        if not isinstance(files, list) or not files:
            return {'ok': False, 'n': 0, 'batch': batch, 'generation': generation,
                    'error': 'No files received.'}
        if len(files) > 64:
            input_msg.value = '<div role="alert" style="color:#991b1b">At most 64 files can be added at once.</div>'
            return {'ok': False, 'n': 0, 'batch': batch, 'generation': generation,
                    'error': 'file-count limit'}
        pairs = []; total = 0
        try:
            for item in files:
                if not isinstance(item, dict) or item.get('error'):
                    raise ValueError('The browser could not read one of the dropped files.')
                name = item.get('name')
                payload = item.get('b64')
                if not name or not isinstance(payload, str):
                    raise ValueError('A dropped file is missing its name or content.')
                content = _dnd_b64.b64decode(payload, validate=True)
                total += len(content)
                if len(content) > 256 * 1024 * 1024 or total > 512 * 1024 * 1024:
                    raise ValueError('Upload size limit exceeded.')
                pairs.append((name, content))
            accepted = _accept_upload_pairs(pairs, 'drag & drop')
            return {'ok': bool(accepted), 'n': len(pairs), 'batch': batch,
                    'generation': generation}
        except Exception as exc:
            input_msg.value = ('<div role="alert" style="color:#991b1b">Drop failed: %s; '
                               'existing files were kept.</div>' % html.escape(str(exc)))
            return {'ok': False, 'n': 0, 'batch': batch, 'generation': generation,
                    'error': str(exc)}
    _dnd_out.register_callback('pdb2reaction_gui.on_drop', _rxgui_drop)
    display(HTML(r"""<script>
(function(){
  var callback='pdb2reaction_gui.on_drop';
  function setPrompt(box,text){
    var label=box.querySelector('.rxdrop-prompt b'); if(label)label.textContent=text;
  }
  function epoch(box){
    var node=box.querySelector('.rxdrop-epoch-value');
    var value=Number(node&&node.dataset.generation);
    return Number.isInteger(value)?value:0;
  }
  function opaqueId(){
    return (globalThis.crypto&&globalThis.crypto.randomUUID)?
      globalThis.crypto.randomUUID():
      (Date.now().toString(36)+'-'+Math.random().toString(36).slice(2));
  }
  function jsonResult(result){
    return result&&result.data&&result.data['application/json'];
  }
  function readFile(file){
    return new Promise(function(resolve,reject){
      var reader=new FileReader();
      reader.onload=function(){
        resolve({name:file.name,b64:(String(reader.result).split(',')[1]||'')});
      };
      reader.onerror=function(){reject(new Error('Could not read '+file.name));};
      reader.readAsDataURL(file);
    });
  }
  function wire(box){
    if(box.dataset.rxDndCallback===callback)return true;
    box.dataset.rxDndCallback=callback;
    var busy=false,queue=[];
    var stop=function(event){event.preventDefault();event.stopPropagation();};
    function submit(files){
      var selected=Array.from(files||[]);
      if(!selected.length)return;
      var total=selected.reduce(function(sum,file){return sum+file.size;},0);
      if(selected.length>64||total>512*1024*1024){
        setPrompt(box,'Files not added — batch is too large');return;
      }
      if(selected.some(function(file){return file.size>256*1024*1024;})){
        setPrompt(box,'Files not added — one file is too large');return;
      }
      queue.push({id:opaqueId(),generation:epoch(box),files:selected});
      if(busy)setPrompt(box,'Queued '+queue.length+
        (queue.length===1?' upload batch':' upload batches'));
      pump();
    }
    async function pump(){
      if(busy||!queue.length)return;
      var item=queue.shift();busy=true;box.classList.add('rxbusy');
      setPrompt(box,'Adding '+item.files.length+
        (item.files.length===1?' file…':' files…'));
      try{
        var out=await Promise.all(item.files.map(readFile));
        var result=await google.colab.kernel.invokeFunction(
          callback,[out,item.id,item.generation],{});
        var payload=jsonResult(result);
        var matched=payload&&payload.batch===item.id&&
                    Number(payload.generation)===item.generation;
        var ok=matched&&payload.ok;
        setPrompt(box,ok?'Drop more files here':
          (payload&&payload.stale?'Cancelled by Clear':'Not added — try again'));
      }catch(_error){
        setPrompt(box,'Drop failed — try again');
      }finally{
        busy=false;box.classList.remove('rxbusy');pump();
      }
    }
    box.addEventListener('dragenter',function(event){
      stop(event);box.classList.add('rxdrag');
      if(event.dataTransfer)event.dataTransfer.dropEffect='copy';
    });
    box.addEventListener('dragover',function(event){
      stop(event);box.classList.add('rxdrag');
      if(event.dataTransfer)event.dataTransfer.dropEffect='copy';
    });
    box.addEventListener('dragleave',function(event){
      stop(event);
      if(!box.contains(event.relatedTarget))box.classList.remove('rxdrag');
    });
    box.addEventListener('drop',function(event){
      stop(event);box.classList.remove('rxdrag');
      submit(event.dataTransfer&&event.dataTransfer.files);
    });
    return true;
  }
  function wireAll(){
    var boxes=document.querySelectorAll('.rxapp .rxdrop');
    for(var i=0;i<boxes.length;i++)wire(boxes[i]);
    return boxes.length>0;
  }
  if(!wireAll()){
    var timer=setInterval(function(){if(wireAll())clearInterval(timer);},300);
    setTimeout(function(){clearInterval(timer);},30000);
  }
})();
</script>"""))
except Exception:
    _dnd_out = None

